# 🌍 Country Development Clustering - Cluster Analysis

## 📋 Notebook Purpose

This notebook performs **clustering analysis** to group countries by development patterns:
1. **Identify optimal number** of country clusters
2. **Analyze cluster profiles** and characteristics
3. **Allocate $1 billion development budget** based on needs
4. **Visualize cluster patterns** for policy recommendations
5. **Create interpretable cluster labels** for stakeholders

## 📊 Data Overview

| Aspect | Detail |
|:---|:---|
| **Input** | Transformed and scaled data from EDA |
| **Countries** | 167 countries |
| **Features** | 9 standardized development indicators |
| **Algorithm** | K-Means clustering |

### 🎯 Analysis Objectives

1. **Find natural country groupings** using K-Means
2. **Validate clusters** with multiple metrics (silhouette, elbow, etc.)
3. **Characterize each cluster** by feature profiles
4. **Create policy recommendations** for each cluster
5. **Allocate budget** proportionally to country needs

## 📁 Notebook Structure
```
02-clustering-analysis.ipynb
├── 1. Data Loading
│ └── Load transformed data from EDA
├── 2. Finding Optimal Clusters (k)
│ ├── Elbow method (inertia)
│ ├── Silhouette score
│ ├── Calinski-Harabasz index
│ └── Davies-Bouldin index
├── 3. Cluster Analysis (k=5)
│ ├── K-Means clustering
│ ├── Cluster sizes and distribution
│ ├── Cluster profiles (mean values)
│ └── Feature importance
├── 4. Cluster Visualization
│ ├── PCA visualization
│ ├── Cluster profiles heatmap
│ ├── Radar charts
│ └── Feature distribution boxplots
├── 5. Cluster Interpretation
│ ├── Naming clusters
│ ├── Describing characteristics
│ └── Policy implications
├── 6. Budget Allocation
│ ├── Priority assignment
│ ├── Budget distribution
│ └── Per-country allocation
└── 7. Conclusions & Recommendations
├── Summary of findings
└── Policy recommendations

```

## 🏆 Key Results

- **Optimal clusters**: k=5 (statistically robust and policy-relevant)
- **Cluster distribution**: Ranging from 24-44 countries
- **All clusters**: No fragile clusters (all 24+ countries)
- **Budget allocation**: Needs-based distribution ($1B total)

### 📊 Cluster Summary

| Cluster | Name | Countries | Priority | Budget |
|:---|:---|:---|:---|:---|
| 0 | Low-Development | 41 | Very High | 35% |
| 1 | Developing-Economy | 31 | High | 25% |
| 2 | High-Income Trade | 44 | Medium | 20% |
| 3 | Emerging-Economy | 27 | Low | 15% |
| 4 | Advanced-Economy | 24 | Very Low | 5% |

## 📤 Output

- Clustered data: `../data/processed/countries_with_clusters_k5.csv`
- Cluster profiles: `../data/processed/cluster_profiles_named.csv`
- Figures: `../figures/clustering/`
- Final report: `../reports/clustering_report.md`

## 📌 Next Steps

- **Interpretation**: Translate clusters to policy recommendations
- **Stakeholder presentation**: Communicate findings
- **Budget allocation**: Implement needs-based distribution

In [ ]:
# ============================================================================
# 1. SETUP AND IMPORTS
# ============================================================================

import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from matplotlib.patches import Rectangle, FancyBboxPatch
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# Machine learning
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from scipy.spatial import ConvexHull
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, Birch
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, silhouette_samples, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from scipy.spatial import ConvexHull


# Statistical tests
from scipy.stats import f_oneway, kruskal
from math import pi






# Get paths
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)


print("✅ All imports successful!")
print("=" * 50)

In [ ]:
df_processed=pd.read_csv('../data/processed/countries_transformed_scaled.csv')

In [ ]:
# ============================================================================
# 2. COMPARE K VALUES WITH INTERPRETABILITY FOCUS
# ============================================================================

def compare_k_values_interpretability(data, k_values=[3,4, 5, 6, 7], random_state=42):
    """
    Compare different k values with focus on interpretability.
    """
    results = {}
    
    print("🔍 COMPARING K VALUES WITH INTERPRETABILITY FOCUS")
    print("=" * 80)
    
    for k in k_values:
        print(f"\n📊 ANALYZING k={k}")
        print("-" * 60)
        
        # Perform clustering
        kmeans = KMeans(n_clusters=k, random_state=random_state, n_init=10)
        labels = kmeans.fit_predict(data)
        
        
        # Calculate metrics
        silhouette = silhouette_score(data, labels)
        calinski = calinski_harabasz_score(data, labels)
        davies = davies_bouldin_score(data, labels)
        
        # Get cluster sizes
        cluster_sizes = np.bincount(labels)
        
        print(f"📈 Quantitative Metrics:")
        print(f"   Silhouette Score: {silhouette:.4f}")
        print(f"   Calinski-Harabasz: {calinski:.2f}")
        print(f"   Davies-Bouldin: {davies:.4f}")
        
        print(f"\n📊 Cluster Distribution:")
        for i, size in enumerate(cluster_sizes):
            pct = size / len(data) * 100
            print(f"   Cluster {i}: {size} countries ({pct:.1f}%)")
        
        # Check for very small clusters
        min_size = cluster_sizes.min()
        max_size = cluster_sizes.max()
        size_ratio = max_size / min_size if min_size > 0 else float('inf')
        
        print(f"\n⚖️ Balance Assessment:")
        print(f"   Smallest cluster: {min_size} countries")
        print(f"   Largest cluster: {max_size} countries")
        print(f"   Size ratio (max/min): {size_ratio:.2f}")
        
        if min_size < 10:
            print("   ⚠️ WARNING: Very small cluster detected (<10 countries)")
        elif size_ratio > 5:
            print("   ⚠️ WARNING: Highly imbalanced clusters")
        else:
            print("   ✅ Reasonably balanced clusters")
        
        # Store results
        results[k] = {
            'labels': labels,
            'silhouette': silhouette,
            'calinski': calinski,
            'davies': davies,
            'cluster_sizes': cluster_sizes,
            'min_size': min_size,
            'max_size': max_size,
            'size_ratio': size_ratio,
            'kmeans_model': kmeans
        }
    
    return results

# Compare k values
k_comparison = compare_k_values_interpretability(df_processed, k_values=[3, 4, 5, 6, 7])

In [ ]:
# ============================================================================
# 3. Interpretability Analysis for Each k
# ============================================================================

def analyze_interpretability(k_results, k, df_original, df_processed):
    """
    Analyze interpretability of clusters for a given k.
    """
    labels = k_results['labels']
    
    # Add clusters to original data
    df_with_clusters = df_original.copy()
    df_with_clusters['Cluster'] = labels
    
    # Get numeric columns for profiling
    numeric_cols = df_with_clusters.select_dtypes(include=[np.number]).columns.tolist()
    if 'Cluster' in numeric_cols:
        numeric_cols.remove('Cluster')
    
    # Calculate cluster profiles
    profiles = df_with_clusters.groupby('Cluster')[numeric_cols].mean()
    
    print(f"\n🔍 INTERPRETABILITY ANALYSIS FOR k={k}")
    print("=" * 60)
    
    # Analyze each cluster
    for cluster in profiles.index:
        print(f"\n🏷️ Cluster {cluster} ({int(k_results['cluster_sizes'][cluster])} countries):")
        print("-" * 40)
        
        # Get key distinguishing features
        profile = profiles.loc[cluster]
        
        # Find features where this cluster is extreme
        all_means = profiles.mean()
        all_stds = profiles.std()
        
        # Calculate z-scores for this cluster
        z_scores = (profile - all_means) / all_stds
        
        # Top 3 highest features
        highest = z_scores.nlargest(3)
        print("  Distinctive strengths (high):")
        for feat, score in highest.items():
            print(f"    • {feat}: {profile[feat]:.2f} (z-score: {score:.2f})")
        
        # Top 3 lowest features
        lowest = z_scores.nsmallest(3)
        print("  Distinctive challenges (low):")
        for feat, score in lowest.items():
            print(f"    • {feat}: {profile[feat]:.2f} (z-score: {score:.2f})")
        
        # Policy implications
        print("  💡 Policy Implications:")
        if 'child_mort' in profile.index and profile['child_mort'] > 50:
            print("    • Urgent healthcare and child survival interventions")
        if 'income' in profile.index and profile['income'] < 5000:
            print("    • Focus on economic development and poverty reduction")
        if 'life_expec' in profile.index and profile['life_expec'] < 60:
            print("    • Healthcare system strengthening needed")
        if 'gdpp' in profile.index and profile['gdpp'] > 20000:
            print("    • Focus on sustainable development and innovation")
        if 'inflation' in profile.index and profile['inflation'] > 10:
            print("    • Economic stabilization and monetary policy focus")
        
        if not any([profile.get('child_mort', 0) > 50, 
                   profile.get('income', 0) < 5000,
                   profile.get('life_expec', 0) < 60]):
            print("    • Balanced development approach recommended")
    
    return profiles

# Analyze interpretability for each k
data_path = "../data/raw/Country-data.csv"
df_raw = pd.read_csv(data_path)

interpretability_results = {}
for k, results in k_comparison.items():
    interpretability_results[k] = analyze_interpretability(
        results, k, df_raw, df_processed
    )

In [ ]:
# ============================================================================
# 4. Summary Comparison Table
# ============================================================================

# Create comprehensive comparison table
comparison_summary = pd.DataFrame()

for k, results in k_comparison.items():
    comparison_summary[k] = [
        results['silhouette'],
        results['calinski'],
        results['davies'],
        results['min_size'],
        results['max_size'],
        results['size_ratio'],
        len(results['cluster_sizes'])
    ]

comparison_summary.index = [
    'Silhouette Score',
    'Calinski-Harabasz',
    'Davies-Bouldin',
    'Smallest Cluster',
    'Largest Cluster',
    'Size Ratio (max/min)',
    'Number of Clusters'
]

print("📊 COMPREHENSIVE COMPARISON: k=3, 4, 5, 6, 7")
print("=" * 80)
display(comparison_summary.round(4))

In [ ]:
# ============================================================================
# 4.1 Complete Summary Comparison Table (k=3 to 7)
# ============================================================================

# Create comprehensive comparison table including k=3
comparison_summary_complete = pd.DataFrame()

for k, results in k_comparison.items():
    comparison_summary_complete[k] = [
        results['silhouette'],
        results['calinski'],
        results['davies'],
        results['min_size'],
        results['max_size'],
        results['size_ratio'],
        len(results['cluster_sizes'])
    ]

comparison_summary_complete.index = [
    'Silhouette Score',
    'Calinski-Harabasz',
    'Davies-Bouldin',
    'Smallest Cluster',
    'Largest Cluster',
    'Size Ratio (max/min)',
    'Number of Clusters'
]

# Display the complete table
print("📊 COMPLETE COMPARISON: k=3, 4, 5, 6, 7")
print("=" * 80)
display(comparison_summary_complete.round(4))

# ============================================================================
# 5.3 Decision Matrix (FIXED)
# ============================================================================

print("\n📊 DECISION MATRIX: k=3 to 7")
print("=" * 80)

# Define criteria weights - FIXED to include all metrics
criteria = {
    'Silhouette Score': 'higher_better',
    'Calinski-Harabasz': 'higher_better',
    'Davies-Bouldin': 'lower_better',
    'Smallest Cluster': 'higher_better',  # Larger is better (no fragile clusters)
    'Largest Cluster': 'neutral',  # Not used for scoring (just information)
    'Size Ratio (max/min)': 'lower_better',  # Lower is better (balanced clusters)
    'Number of Clusters': 'neutral'  # Not used for scoring
}

# Create decision matrix
decision_matrix = pd.DataFrame(index=comparison_summary_complete.columns)

# Normalize scores (0 to 1) - only for metrics that matter
for metric in comparison_summary_complete.index:
    # Skip neutral metrics
    if criteria.get(metric) == 'neutral':
        continue
        
    values = comparison_summary_complete.loc[metric]
    
    if criteria[metric] == 'higher_better':
        normalized = (values - values.min()) / (values.max() - values.min())
    else:  # lower_better
        normalized = (values.max() - values) / (values.max() - values.min())
    
    decision_matrix[metric] = normalized

# Calculate overall score (average of all metrics)
decision_matrix['Overall_Score'] = decision_matrix.mean(axis=1)

# Sort by overall score
decision_matrix = decision_matrix.sort_values('Overall_Score', ascending=False)

print("\n📈 NORMALIZED SCORES (0-1, higher is better):")
print("=" * 60)
display(decision_matrix.round(3))

# ============================================================================
# 5.4 Final Recommendation
# ============================================================================

print("\n🎯 FINAL RECOMMENDATION")
print("=" * 60)

best_k = decision_matrix.index[0]
best_score = decision_matrix.loc[best_k, 'Overall_Score']

print(f"\n✅ Optimal k = {best_k} (Overall Score: {best_score:.3f})")

# Detailed breakdown
print("\n📊 Detailed Breakdown:")
print("-" * 60)
for k in decision_matrix.index:
    score = decision_matrix.loc[k, 'Overall_Score']
    if k == best_k:
        print(f"   k={k}: {score:.3f} ⭐ BEST")
    else:
        diff = (best_score - score) * 100
        print(f"   k={k}: {score:.3f} ({diff:.1f}% worse than best)")

# ============================================================================
# 5.5 Summary Table for Report
# ============================================================================

print("\n📋 SUMMARY TABLE FOR REPORT:")
print("=" * 80)

# Create a clean summary table
summary_for_report = comparison_summary_complete.copy()

# Add a "Recommendation" row
def get_recommendation(k, data):
    if data.loc['Smallest Cluster', k] < 10:
        return '❌ REJECT (Fragile)'
    elif k == 5:
        return '✅ RECOMMENDED'
    elif k == 4:
        return '✅ GOOD ALTERNATIVE'
    elif k == 3:
        return '✅ SIMPLE OPTION'
    elif data.loc['Size Ratio (max/min)', k] > 3:
        return '⚠️ UNBALANCED'
    else:
        return '⚠️ ACCEPTABLE'

recommendations = [get_recommendation(k, comparison_summary_complete) for k in comparison_summary_complete.columns]
summary_for_report.loc['Recommendation'] = recommendations

print(summary_for_report.round(4).to_string())

# ============================================================================
# 5.6 Save Results
# ============================================================================

# Save the comparison table
comparison_summary_complete.to_csv('../data/processed/clustering_comparison_k3_k7.csv')
print("\n✅ Comparison table saved to: ../data/processed/clustering_comparison_k3_k7.csv")

In [ ]:
# ============================================================================
# 4.2 ELBOW CURVE AND CLUSTERING METRICS VISUALIZATION
# ============================================================================

print("📊 GENERATING ELBOW CURVE AND CLUSTERING METRICS")
print("=" * 80)

# Evaluate k from 2 to 12
k_range = range(2, 13)
inertias = []
silhouette_scores = []
calinski_scores = []
davies_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(df_processed)
    
    # Get labels
    labels = kmeans.labels_
    
    # Calculate metrics
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(df_processed, labels))
    calinski_scores.append(calinski_harabasz_score(df_processed, labels))
    davies_scores.append(davies_bouldin_score(df_processed, labels))

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Clustering Metrics for k=2 to k=12', fontsize=16, fontweight='bold', y=1.02)

# 1. Elbow Curve (Inertia)
ax1 = axes[0, 0]
ax1.plot(range(2, 13), inertias, 'o-', color='#2E86AB', linewidth=2, markersize=8)
ax1.axvline(x=5, color='red', linestyle='--', linewidth=2, alpha=0.7, label='k=5 selected')
ax1.set_xlabel('Number of Clusters (k)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Inertia (Within-Cluster Sum of Squares)', fontsize=12, fontweight='bold')
ax1.set_title('Elbow Method', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()

# 2. Silhouette Score
ax2 = axes[0, 1]
ax2.plot(range(2, 13), silhouette_scores, 'o-', color='#A23B72', linewidth=2, markersize=8)
ax2.axvline(x=5, color='red', linestyle='--', linewidth=2, alpha=0.7, label='k=5 selected')
ax2.axhline(y=0.2, color='green', linestyle=':', linewidth=1.5, alpha=0.5, label='Threshold: 0.2')
ax2.set_xlabel('Number of Clusters (k)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Silhouette Score', fontsize=12, fontweight='bold')
ax2.set_title('Silhouette Score', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend()

# 3. Calinski-Harabasz Score
ax3 = axes[1, 0]
ax3.plot(range(2, 13), calinski_scores, 'o-', color='#F18F01', linewidth=2, markersize=8)
ax3.axvline(x=5, color='red', linestyle='--', linewidth=2, alpha=0.7, label='k=5 selected')
ax3.set_xlabel('Number of Clusters (k)', fontsize=12, fontweight='bold')
ax3.set_ylabel('Calinski-Harabasz Score', fontsize=12, fontweight='bold')
ax3.set_title('Calinski-Harabasz Index', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.legend()

# 4. Davies-Bouldin Score
ax4 = axes[1, 1]
ax4.plot(range(2, 13), davies_scores, 'o-', color='#6A994E', linewidth=2, markersize=8)
ax4.axvline(x=5, color='red', linestyle='--', linewidth=2, alpha=0.7, label='k=5 selected')
ax4.set_xlabel('Number of Clusters (k)', fontsize=12, fontweight='bold')
ax4.set_ylabel('Davies-Bouldin Score', fontsize=12, fontweight='bold')
ax4.set_title('Davies-Bouldin Index (Lower is Better)', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3)
ax4.legend()

# Highlight k=5 on all plots with a vertical line
for ax in [ax1, ax2, ax3, ax4]:
    ax.axvline(x=5, color='red', linestyle='--', linewidth=2, alpha=0.7)

plt.tight_layout()
plt.savefig('../figures/clustering/elbow_curve.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("✅ Elbow curve saved to: ../figures/clustering/elbow_curve.png")

# Print the metrics table for reference
print("\n📊 CLUSTERING METRICS SUMMARY")
print("=" * 80)
metrics_df = pd.DataFrame({
    'k': range(2, 13),
    'Inertia': inertias,
    'Silhouette': silhouette_scores,
    'Calinski-Harabasz': calinski_scores,
    'Davies-Bouldin': davies_scores
})
display(metrics_df.round(4))

print("""
📊 STATISTICAL QUALITY:
   • Silhouette Score: 0.2226
     → Comparable to the other granular solutions considered (k=4–7)
   • Davies-Bouldin Index: 1.4620
     → Provides cluster separation comparable to the other selected candidates
   • Cluster size: 24–44 countries
     → No very small or potentially fragile clusters
   • Size ratio: 1.83
     → Best cluster balance among k=4–7

🔍 COMPARED TO k=3:
   • k=3 achieves the strongest internal validation scores among the
     directly compared solutions.
   • However, three clusters provide a relatively coarse segmentation
     of the multidimensional development structure.
   • k=5 provides additional differentiation between countries with
     distinct health, demographic, economic, trade, and macroeconomic profiles.
   • Therefore, k=5 was preferred for the intended exploratory and
     policy-oriented analysis despite the lower internal validation scores.

🔍 COMPARED TO k=6:
   • k=6 has the lowest silhouette score among k=3–7 (0.1972).
   • Its cluster balance is weaker than k=5
     (size ratio: 2.00 vs. 1.83).
   • The additional cluster therefore provides limited benefit relative
     to the increased segmentation.

🔍 COMPARED TO k=7:
   • k=7 produces two small clusters containing only 8 and 9 countries.
   • Size ratio increases substantially to 4.75, indicating strong imbalance.
   • This makes the solution less suitable for a country-level
     policy-oriented analysis.

💡 POLICY VALUE OF k=5:
   • Severe Vulnerability (44 countries)
     → Highest priority for basic health, education, and poverty reduction
   • Macroeconomic Vulnerability (24 countries)
     → Focus on macroeconomic stabilization and social development
   • Trade-Integrated (27 countries)
     → Focus on trade resilience, diversification, and sustainable integration
   • Moderate Development (41 countries)
     → Focus on continued healthcare, education, infrastructure,
       and economic development
   • Advanced Development (31 countries)
     → Focus on sustainability, innovation, and long-term resilience

✅ CONCLUSION:
   k=5 was selected as the final analytical solution because it provides
   a useful balance between:

   • Statistical quality
   • Cluster size and balance
   • Development-profile granularity
   • Interpretability
   • Policy-oriented usefulness

   This selection represents an analytical judgment rather than an
   objectively optimal value of k. The k=3 solution performs better
   on several internal validation metrics, but k=5 provides a more
   granular segmentation that is better aligned with the objectives
   of the analysis.
""")



In [ ]:
# ============================================================================
# 4.3 K=5 SELECTION JUSTIFICATION FIGURE - CONSISTENT FONTS
# ============================================================================

print("📊 CREATING K=5 SELECTION JUSTIFICATION FIGURE")
print("=" * 80)



# ============================================================================
# 1. FIGURE SETUP
# ============================================================================

fig, ax = plt.subplots(figsize=(22, 18))  # Slightly larger figure
ax.set_xlim(0, 22)
ax.set_ylim(0, 18)
ax.axis("off")

# ============================================================================
# 2. COLOR CONFIGURATION
# ============================================================================

colors = {
    "primary": "#1B2A4A",
    "stat": "#2471A3",
    "stat_bg": "#EBF5FB",
    "stat_dark": "#1A5276",
    "comparison": "#C0392B",
    "comparison_bg": "#FDEDEC",
    "comparison_dark": "#922B21",
    "policy": "#27AE60",
    "policy_bg": "#EAFAF1",
    "policy_dark": "#1E8449",
    "conclusion": "#F39C12",
    "conclusion_bg": "#FEF5E7",
    "conclusion_dark": "#D68910",
    "white": "#FFFFFF",
    "text": "#1C2833",
    "subtext": "#5D6D7E",
    "border": "#BDC3C7",
}

# ============================================================================
# 3. FONT SIZE DEFINITIONS (CONSISTENT)
# ============================================================================

FONTS = {
    "header_title": 34,
    "header_subtitle": 22,
    "section_header": 23,
    "label_bold": 20,
    "value_bold": 21,
    "body_text": 19,
    "note_italic": 18,
    "small_text": 17,
    "footer": 18,
}

# ============================================================================
# 4. HEADER
# ============================================================================

header = FancyBboxPatch(
    (0.8, 15.4),
    20.4,
    1.8,
    boxstyle="round,pad=0.15",
    facecolor=colors["primary"],
    edgecolor="none",
)
ax.add_patch(header)

ax.text(
    11,
    16.5,
    "⭐ K=5 SELECTION JUSTIFICATION",
    fontsize=FONTS["header_title"],
    fontweight="bold",
    color="white",
    ha="center",
    va="center",
)

ax.text(
    11,
    15.7,
    "Analytical Trade-off Between Statistical Quality and Policy Relevance",
    fontsize=FONTS["header_subtitle"],
    color="#BDC3C7",
    ha="center",
    va="center",
)

# ============================================================================
# 5. SECTION 1 — STATISTICAL QUALITY (BLUE BOX)
# ============================================================================

stat_box = FancyBboxPatch(
    (0.8, 10.2),
    9.5,
    4.8,
    boxstyle="round,pad=0.1",
    facecolor=colors["stat_bg"],
    edgecolor=colors["stat"],
    linewidth=2.5,
)
ax.add_patch(stat_box)

# Header
stat_h = Rectangle(
    (0.8, 14.2),
    9.5,
    0.8,
    facecolor=colors["stat"],
    edgecolor="none",
)
ax.add_patch(stat_h)
ax.text(
    5.55,
    14.6,
    "📊 STATISTICAL QUALITY",
    fontsize=FONTS["section_header"],
    fontweight="bold",
    color="white",
    ha="center",
    va="center",
)

# Content
stats = [
    ("Silhouette Score:", "0.2226"),
    ("Davies-Bouldin:", "1.4620"),
    ("Cluster Size:", "24–44"),
    ("Size Ratio:", "1.83"),
]

notes = [
    "Reasonable for k=5",
    "Acceptable separation",
    "No very small clusters",
    "Best among k=4–7",
]

for i, (label, value) in enumerate(stats):
    y = 13.8 - i * 0.9
    ax.text(1.5, y, label, fontsize=FONTS["label_bold"], fontweight="bold", color=colors["text"])
    ax.text(7.0, y, value, fontsize=FONTS["value_bold"], fontweight="bold", color=colors["stat"])
    
    note_y = y - 0.38
    ax.text(1.5, note_y, notes[i], fontsize=FONTS["note_italic"], color=colors["subtext"], style="italic")

# ============================================================================
# 6. SECTION 2 — COMPARISON WITH OTHER K (RED BOX)
# ============================================================================

comp_box = FancyBboxPatch(
    (11.7, 10.2),
    9.5,
    4.8,
    boxstyle="round,pad=0.1",
    facecolor=colors["comparison_bg"],
    edgecolor=colors["comparison"],
    linewidth=2.5,
)
ax.add_patch(comp_box)

# Header
comp_h = Rectangle(
    (11.7, 14.2),
    9.5,
    0.8,
    facecolor=colors["comparison"],
    edgecolor="none",
)
ax.add_patch(comp_h)
ax.text(
    16.45,
    14.6,
    "🔍 COMPARED TO OTHER K",
    fontsize=FONTS["section_header"],
    fontweight="bold",
    color="white",
    ha="center",
    va="center",
)

# Content
comparisons = [
    ("⚠ k=3", "Best scores, too coarse"),
    ("• k=4", "Similar, less granular"),
    ("❌ k=6", "Lower silhouette"),
    ("❌ k=7", "Fragile & unbalanced"),
]

for i, (k, desc) in enumerate(comparisons):
    y = 13.7 - i * 0.9
    color = colors["comparison"] if "⚠" in k or "❌" in k else colors["subtext"]
    ax.text(12.5, y, k, fontsize=FONTS["label_bold"], fontweight="bold", color=color)
    ax.text(14.5, y, desc, fontsize=FONTS["body_text"], color=colors["text"])

# ============================================================================
# 7. SECTION 3 — POLICY VALUE (GREEN BOX)
# ============================================================================

policy_box = FancyBboxPatch(
    (0.8, 1.8),
    9.5,
    8.0,
    boxstyle="round,pad=0.1",
    facecolor=colors["policy_bg"],
    edgecolor=colors["policy"],
    linewidth=2.5,
)
ax.add_patch(policy_box)

# Header
policy_h = Rectangle(
    (0.8, 9.0),
    9.5,
    0.8,
    facecolor=colors["policy"],
    edgecolor="none",
)
ax.add_patch(policy_h)
ax.text(
    5.55,
    9.4,
    "💡 POLICY VALUE OF K=5",
    fontsize=FONTS["section_header"],
    fontweight="bold",
    color="white",
    ha="center",
    va="center",
)

# Content
policy_clusters = [
    ("🔴 Severe Vulnerability", "44", "Highest priority: health, education"),
    ("🟠 Macroeconomic Vulnerability", "24", "Stabilization + social development"),
    ("🔵 Trade-Integrated", "27", "Trade resilience, diversification"),
    ("🔴 Moderate Development", "41", "Healthcare, education, infrastructure"),
    ("🟢 Advanced Development", "31", "Sustainability, innovation, resilience"),
]

for i, (name, size, policy) in enumerate(policy_clusters):
    y = 8.5 - i * 1.15
    ax.text(1.5, y, name, fontsize=FONTS["label_bold"], fontweight="bold", color=colors["text"])
    ax.text(6.5, y, f"{size} countries", fontsize=FONTS["value_bold"], fontweight="bold", color=colors["subtext"])
    ax.text(1.5, y - 0.45, f"→ {policy}", fontsize=FONTS["body_text"], color=colors["subtext"], style="italic")

# ============================================================================
# 8. SECTION 4 — CONCLUSION (ORANGE BOX)
# ============================================================================

conclusion_box = FancyBboxPatch(
    (11.7, 1.8),
    9.5,
    8.0,
    boxstyle="round,pad=0.1",
    facecolor=colors["conclusion_bg"],
    edgecolor=colors["conclusion"],
    linewidth=2.5,
)
ax.add_patch(conclusion_box)

# Header
conclusion_h = Rectangle(
    (11.7, 9.0),
    9.5,
    0.8,
    facecolor=colors["conclusion"],
    edgecolor="none",
)
ax.add_patch(conclusion_h)
ax.text(
    16.45,
    9.4,
    "✅ CONCLUSION",
    fontsize=FONTS["section_header"],
    fontweight="bold",
    color="white",
    ha="center",
    va="center",
)

# Main text
ax.text(
    16.45,
    8.4,
    "K=5: Analytical Judgment",
    fontsize=FONTS["header_subtitle"],
    fontweight="bold",
    color=colors["conclusion"],
    ha="center",
)

ax.text(
    16.45,
    7.8,
    "Selected for the best balance of:",
    fontsize=FONTS["body_text"],
    color=colors["text"],
    ha="center",
)

# Reasons
reasons = [
    "Statistical quality",
    "Granularity",
    "Cluster balance",
    "Interpretability",
    "Policy relevance",
]

for i, reason in enumerate(reasons):
    y = 7.1 - i * 0.8
    ax.text(12.7, y, "✓", fontsize=FONTS["value_bold"], fontweight="bold", color=colors["policy"])
    ax.text(14.2, y, reason, fontsize=FONTS["body_text"], color=colors["text"])

# Note
ax.text(
    16.45,
    3.0,
    "k=3 has stronger validation scores,",
    fontsize=FONTS["small_text"],
    color=colors["subtext"],
    ha="center",
    style="italic",
)
ax.text(
    16.45,
    2.5,
    "but k=5 is more policy-relevant.",
    fontsize=FONTS["small_text"],
    color=colors["subtext"],
    ha="center",
    style="italic",
)

# ============================================================================
# 9. FOOTER
# ============================================================================

ax.plot(
    [0.8, 21.2],
    [0.8, 0.8],
    color="#BDC3C7",
    linewidth=1,
    linestyle="--",
)

ax.text(
    11,
    0.4,
    "© Country Clustering for Budget Allocation Analysis",
    fontsize=FONTS["footer"],
    color="#95A5A6",
    ha="center",
    style="italic",
)

# ============================================================================
# 10. SAVE FIGURE
# ============================================================================

plt.tight_layout()
plt.savefig(
    "../figures/clustering/k5_selection_justification.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
    edgecolor="none",
)
plt.show()

print("\n✅ Figure saved to: ../figures/clustering/k5_selection_justification.png")
print("\n📊 FONT SIZE SUMMARY:")
print(f"   • Header Title: {FONTS['header_title']}pt")
print(f"   • Header Subtitle: {FONTS['header_subtitle']}pt")
print(f"   • Section Headers: {FONTS['section_header']}pt")
print(f"   • Labels (Bold): {FONTS['label_bold']}pt")
print(f"   • Values (Bold): {FONTS['value_bold']}pt")
print(f"   • Body Text: {FONTS['body_text']}pt")
print(f"   • Notes (Italic): {FONTS['note_italic']}pt")
print(f"   • Small Text: {FONTS['small_text']}pt")
print(f"   • Footer: {FONTS['footer']}pt")
print("\n   ✅ All fonts are consistent across sections!")

In [ ]:
# ============================================================================
# 5. PERFORM CLUSTERING
# ============================================================================

print("🔬 PERFORMING K-MEANS CLUSTERING")
print("=" * 60)

# Use k=5 from comparison
kmeans_final = KMeans(n_clusters=5, random_state=42, n_init=10)
final_labels = kmeans_final.fit_predict(df_processed)

# Add to data
df_with_clusters_final = df_raw.copy()
df_with_clusters_final['cluster'] = final_labels

print(f"✅ Clustering complete with 5 clusters")
print("\n📊 Final Cluster Distribution:")
cluster_counts = np.bincount(final_labels)
for i, count in enumerate(cluster_counts):
    pct = count / len(final_labels) * 100
    print(f"   Cluster {i}: {count} countries ({pct:.1f}%)")

In [ ]:
# ============================================================================
# 6. PROFESSIONAL CLUSTER NAMING AND POLICY PROFILE CONFIGURATION
# ============================================================================

# ============================================================================


# ----------------------------------------------------------------------------
# 1. FULL CLUSTER NAMES
# ----------------------------------------------------------------------------

CLUSTER_NAMES = {
    0: "Intermediate Development Profile",
    1: "High Development Profile",
    2: "High Development Vulnerability",
    3: "Trade-Intensive Development Profile",
    4: "Higher-Inflation Development Profile",
}


# ----------------------------------------------------------------------------
# 2. SHORT NAMES
# ----------------------------------------------------------------------------

CLUSTER_SHORT_NAMES = {
    0: "Intermediate Development",
    1: "High Development",
    2: "High Development Vulnerability",
    3: "Trade-Intensive",
    4: "Higher Inflation",
}


# ----------------------------------------------------------------------------
# 3. ULTRA-SHORT LABELS
# ----------------------------------------------------------------------------

CLUSTER_LABELS = {
    0: "Intermediate Dev.",
    1: "High Development",
    2: "High Vulnerability",
    3: "Trade-Intensive",
    4: "Higher Inflation",
}


# ----------------------------------------------------------------------------
# 4. POLICY PRIORITY
# ----------------------------------------------------------------------------

CLUSTER_PRIORITY = {
    0: "High",
    1: "Very Low",
    2: "Very High",
    3: "Medium",
    4: "High",
}


# ----------------------------------------------------------------------------
# 5. HYPOTHETICAL BUDGET ALLOCATION
# ----------------------------------------------------------------------------
#
# These percentages are a separate policy-design layer.
# They are NOT directly produced by K-Means.
# ----------------------------------------------------------------------------

CLUSTER_BUDGET = {
    0: 25,
    1: 2,
    2: 45,
    3: 8,
    4: 20,
}


# ----------------------------------------------------------------------------
# 6. COLORS
# ----------------------------------------------------------------------------

CLUSTER_COLORS = {
    0: "#E74C3C",
    1: "#2ECC71",
    2: "#C0392B",
    3: "#3498DB",
    4: "#F39C12",
}


# ----------------------------------------------------------------------------
# 7. CLUSTER DESCRIPTIONS
# ----------------------------------------------------------------------------

CLUSTER_DESCRIPTIONS = {

    # ------------------------------------------------------------------------
    # CLUSTER 0
    # ------------------------------------------------------------------------

    0: (
        "Countries with an intermediate development profile across the "
        "indicators included in the analysis. Compared with the high "
        "development vulnerability cluster, these countries exhibit "
        "substantially better health and economic outcomes, while remaining "
        "below the strongest-performing cluster on several development "
        "indicators. The cluster has average child mortality of 32.3 per "
        "1,000 live births, life expectancy of 69.2 years, income per capita "
        "of approximately $7,958, and GDP per capita of $3,917. The profile "
        "suggests continued needs in healthcare, education, infrastructure, "
        "and broader economic development."
    ),


    # ------------------------------------------------------------------------
    # CLUSTER 1
    # ------------------------------------------------------------------------

    1: (
        "Countries with the strongest overall development profile among the "
        "five clusters. The cluster is characterized by very low child "
        "mortality (5.7 per 1,000 live births), high life expectancy "
        "(79.7 years), high income per capita ($32,959), high GDP per capita "
        "($35,055), low fertility (1.75 births per woman), and relatively "
        "low inflation (2.1%). Within the scope of the nine indicators "
        "analyzed, these countries exhibit comparatively favorable health "
        "and economic outcomes. Policy emphasis may therefore shift toward "
        "sustainability, innovation, social protection, and long-term "
        "resilience rather than basic development needs."
    ),


    # ------------------------------------------------------------------------
    # CLUSTER 2
    # ------------------------------------------------------------------------

    2: (
        "Countries with the highest level of development vulnerability in "
        "the dataset. This cluster is characterized by very high child "
        "mortality (94.1 per 1,000 live births), low life expectancy "
        "(59.9 years), very low income per capita ($2,170), very low GDP "
        "per capita ($952), high fertility (4.97 births per woman), and "
        "elevated inflation (11.6%). These characteristics indicate "
        "substantial health, demographic, and economic development "
        "challenges and support the highest policy priority for basic "
        "healthcare, education, infrastructure, and poverty-reduction "
        "interventions."
    ),


    # ------------------------------------------------------------------------
    # CLUSTER 3
    # ------------------------------------------------------------------------

    3: (
        "Countries characterized by exceptionally high trade intensity, "
        "with exports averaging 77.5% and imports 76.9% of GDP. These "
        "economies also demonstrate relatively strong health outcomes, "
        "high income, low fertility, and low inflation. The defining "
        "feature of this cluster within the selected indicators is the "
        "combination of high trade intensity and relatively favorable "
        "development outcomes. Policy considerations may therefore include "
        "trade resilience, economic diversification, and sustainable "
        "participation in international markets."
    ),


    # ------------------------------------------------------------------------
    # CLUSTER 4
    # ------------------------------------------------------------------------

    4: (
        "Countries with relatively stronger income and human-development "
        "outcomes that are distinguished from the other clusters primarily "
        "by comparatively elevated inflation. The cluster has average "
        "income per capita of $27,186, life expectancy of 73.8 years, "
        "child mortality of 22.1 per 1,000 live births, and inflation of "
        "14.0%, the highest among the five clusters. Health expenditure "
        "is also relatively low at $4.62 per capita. The profile therefore "
        "suggests a combination of relatively stronger development outcomes "
        "with greater inflationary pressure. Policy considerations may "
        "include macroeconomic stabilization alongside continued investment "
        "in health and social development. Cluster membership should not be "
        "interpreted as a general classification of economic weakness or "
        "overall macroeconomic vulnerability."
    ),
}


# ----------------------------------------------------------------------------
# 8. CLUSTER ORDER
# ----------------------------------------------------------------------------

CLUSTER_ORDER = [0, 1, 2, 3, 4]


# ============================================================================
# 9. VALIDATION
# ============================================================================

EXPECTED_CLUSTERS = set(CLUSTER_ORDER)

assert set(CLUSTER_NAMES.keys()) == EXPECTED_CLUSTERS
assert set(CLUSTER_SHORT_NAMES.keys()) == EXPECTED_CLUSTERS
assert set(CLUSTER_LABELS.keys()) == EXPECTED_CLUSTERS
assert set(CLUSTER_PRIORITY.keys()) == EXPECTED_CLUSTERS
assert set(CLUSTER_BUDGET.keys()) == EXPECTED_CLUSTERS
assert set(CLUSTER_COLORS.keys()) == EXPECTED_CLUSTERS
assert set(CLUSTER_DESCRIPTIONS.keys()) == EXPECTED_CLUSTERS

assert sum(CLUSTER_BUDGET.values()) == 100, (
    "Budget allocations must sum to 100%."
)


# ============================================================================
# 10. DISPLAY CONFIGURATION
# ============================================================================

print("=" * 80)
print("PROFESSIONAL CLUSTER NAMES AND POLICY PROFILES")
print("=" * 80)

for cluster_id in CLUSTER_ORDER:

    print(f"\nCluster {cluster_id}:")
    print(f"  Full Name  : {CLUSTER_NAMES[cluster_id]}")
    print(f"  Short Name : {CLUSTER_SHORT_NAMES[cluster_id]}")
    print(f"  Label      : {CLUSTER_LABELS[cluster_id]}")
    print(f"  Priority   : {CLUSTER_PRIORITY[cluster_id]}")
    print(f"  Budget     : {CLUSTER_BUDGET[cluster_id]}%")
    print(f"  Color      : {CLUSTER_COLORS[cluster_id]}")


# ============================================================================
# 11. BUDGET VALIDATION
# ============================================================================

total_budget = sum(CLUSTER_BUDGET.values())

print("\n" + "=" * 80)
print("BUDGET VALIDATION")
print("=" * 80)

print(f"Total allocation: {total_budget}%")

if total_budget == 100:
    print("✓ Budget allocation correctly sums to 100%.")
else:
    print("⚠ WARNING: Budget allocation does not sum to 100%.")


# ============================================================================
# 12. LOOKUP FUNCTIONS
# ============================================================================

def get_cluster_name(cluster_id):
    """Return the full professional name for a cluster."""
    return CLUSTER_NAMES.get(
        cluster_id,
        f"Unknown Cluster {cluster_id}"
    )


def get_cluster_short_name(cluster_id):
    """Return the short name for a cluster."""
    return CLUSTER_SHORT_NAMES.get(
        cluster_id,
        f"Unknown Cluster {cluster_id}"
    )


def get_cluster_label(cluster_id):
    """Return the compact visualization label for a cluster."""
    return CLUSTER_LABELS.get(
        cluster_id,
        f"Cluster {cluster_id}"
    )


def get_cluster_priority(cluster_id):
    """Return the policy priority assigned to a cluster."""
    return CLUSTER_PRIORITY.get(
        cluster_id,
        "Unknown"
    )


def get_cluster_budget(cluster_id):
    """Return the hypothetical budget allocation percentage."""
    return CLUSTER_BUDGET.get(
        cluster_id,
        0
    )


def get_cluster_description(cluster_id):
    """Return the evidence-based description for a cluster."""
    return CLUSTER_DESCRIPTIONS.get(
        cluster_id,
        "No description available."
    )


# ============================================================================
# END
# ============================================================================

In [ ]:
# ============================================================================
# 7. PCA VISUALIZATION OF CLUSTERS 
# ============================================================================

print("\n📊 PCA VISUALIZATION OF CLUSTERS")
print("=" * 80)


# ============================================================================
# STEP 1: Prepare Data
# ============================================================================

X_pca = df_processed.values

# Apply PCA
pca = PCA(n_components=2)
X_pca_2d = pca.fit_transform(X_pca)

# Create DataFrame
pca_df = pd.DataFrame(X_pca_2d, columns=['PC1', 'PC2'])
pca_df['Cluster'] = df_with_clusters_final['cluster'].values

# Calculate explained variance
explained_variance = pca.explained_variance_ratio_

# ============================================================================
# STEP 2: Create PCA Scatter Plot 
# ============================================================================

fig, ax = plt.subplots(figsize=(18, 13), dpi=100)  
fig.patch.set_facecolor('white')

# Plot each cluster
for cluster_num in CLUSTER_ORDER:
    mask = pca_df['Cluster'] == cluster_num
    cluster_data = pca_df[mask]
    
    short_name = CLUSTER_SHORT_NAMES.get(cluster_num, f'Cluster {cluster_num}')
    label = CLUSTER_LABELS.get(cluster_num, f'Cluster {cluster_num}')
    color = CLUSTER_COLORS.get(cluster_num, '#808080')
    
    scatter = ax.scatter(
        cluster_data['PC1'], 
        cluster_data['PC2'],
        label=short_name,
        alpha=0.75,
        s=160,  
        c=color,
        edgecolors='white',
        linewidth=2,  
        zorder=3
    )
    
    # ====== CONVEX HULL ======
    if len(cluster_data) > 2:
        points = cluster_data[['PC1', 'PC2']].values
        hull = ConvexHull(points)
        for simplex in hull.simplices:
            ax.plot(
                points[simplex, 0], points[simplex, 1], 
                color=color, alpha=0.4, linewidth=2, linestyle='--'  
            )
        hull_points = points[hull.vertices]
        ax.fill(
            hull_points[:, 0], hull_points[:, 1], 
            color=color, alpha=0.10, zorder=1  
        )
    
    centroid_x = cluster_data['PC1'].mean()
    centroid_y = cluster_data['PC2'].mean()
    ax.scatter(
        centroid_x, centroid_y, 
        s=400, c=color, marker='*', 
        edgecolors='black', linewidth=2, zorder=5
    )
    ax.annotate(
        f' {label}', (centroid_x, centroid_y),
        fontsize=20, fontweight='bold', color='black', 
        ha='left', va='center', zorder=6
    )

ax.set_xlabel(
    f'Principal Component 1 ({explained_variance[0]*100:.1f}% variance explained)', 
    fontsize=20, fontweight='bold', labelpad=12  
)
ax.set_ylabel(
    f'Principal Component 2 ({explained_variance[1]*100:.1f}% variance explained)', 
    fontsize=20, fontweight='bold', labelpad=12  
)
ax.set_title(
    'PCA Visualization of Country Development Clusters (k=5)', 
    fontsize=26, fontweight='bold', pad=25, color='#1a1a1a'  
)

legend = ax.legend(
    loc='best', 
    fontsize=16,  
    frameon=True,
    fancybox=True,
    shadow=True,
    edgecolor='#cccccc',
    title='Development Clusters',
    title_fontsize=18,  
    markerscale=1.5  
)
legend.get_frame().set_facecolor('white')
legend.get_frame().set_alpha(0.9)

# ====== GRID AND BACKGROUND ======
ax.grid(True, alpha=0.15, linestyle='-', linewidth=0.8)
ax.set_axisbelow(True)
ax.set_facecolor('#F8F9FA')
ax.tick_params(axis='both', labelsize=16)  

# ====== SPINES ======
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#333333')
ax.spines['bottom'].set_color('#333333')
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)

# ====== VARIANCE EXPLAINED BOX ======
ax.text(
    0.02, 0.98, 
    f'Total Variance Explained: {sum(explained_variance)*100:.1f}%',
    transform=ax.transAxes,
    fontsize=14,  
    fontweight='bold',
    verticalalignment='top',
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.85, edgecolor='#cccccc')
)

# ====== ANNOTATION  ======
annotation_text = (
    "Note: PCA projection shows 2D representation of 9D feature space.\n"
    "Cluster separation in full-dimensional space is more distinct."
)
ax.text(
    0.02, 0.02, annotation_text,
    transform=ax.transAxes,
    fontsize=13,  
    style='italic',
    verticalalignment='bottom',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.85, edgecolor='#cccccc')
)

plt.tight_layout()

# ====== SAVE ======
plt.savefig(
    '../figures/clustering/pca_clusters_professional.png', 
    dpi=300, 
    bbox_inches='tight',
    facecolor='white',
    edgecolor='none'
)

plt.show()

print("✅ Enhanced PCA visualization saved to: ../figures/clustering/pca_clusters_professional.png")
print(f"   Explained variance: PC1={explained_variance[0]*100:.1f}%, PC2={explained_variance[1]*100:.1f}%")
print(f"   Total variance explained: {sum(explained_variance)*100:.1f}%")

# ============================================================================
# PCA LOADINGS VISUALIZATION
# ============================================================================

print("\n📊 PCA LOADINGS VISUALIZATION")
print("=" * 80)

# Get feature loadings
loadings = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2'],
    index=df_processed.columns
)

fig, axes = plt.subplots(1, 2, figsize=(20, 9))  

# ====== PLOT 1: PC1 Loadings ======
ax1 = axes[0]
colors_pc1 = ['#2ECC71' if x > 0 else '#E74C3C' for x in loadings['PC1']]
bars1 = ax1.barh(
    loadings.index, loadings['PC1'], 
    color=colors_pc1, alpha=0.8, edgecolor='white', linewidth=2
)
ax1.axvline(x=0, color='black', linewidth=1.5, linestyle='--', alpha=0.5)

ax1.set_xlabel('PC1 Loading', fontsize=18, fontweight='bold')
ax1.set_title(
    f'PC1 Feature Contributions ({explained_variance[0]*100:.1f}% variance)', 
    fontsize=22, fontweight='bold'
)
ax1.tick_params(axis='both', labelsize=16)
ax1.grid(True, alpha=0.2)

# Add value labels
for bar, val in zip(bars1, loadings['PC1']):
    ax1.text(
        val + 0.02, bar.get_y() + bar.get_height()/2, 
        f'{val:.2f}', va='center', fontsize=14, fontweight='bold'
    )

# ====== PLOT 2: PC2 Loadings ======
ax2 = axes[1]
colors_pc2 = ['#2ECC71' if x > 0 else '#E74C3C' for x in loadings['PC2']]
bars2 = ax2.barh(
    loadings.index, loadings['PC2'], 
    color=colors_pc2, alpha=0.8, edgecolor='white', linewidth=2
)
ax2.axvline(x=0, color='black', linewidth=1.5, linestyle='--', alpha=0.5)

ax2.set_xlabel('PC2 Loading', fontsize=18, fontweight='bold')
ax2.set_title(
    f'PC2 Feature Contributions ({explained_variance[1]*100:.1f}% variance)', 
    fontsize=22, fontweight='bold'
)
ax2.tick_params(axis='both', labelsize=16)
ax2.grid(True, alpha=0.2)

# Add value labels
for bar, val in zip(bars2, loadings['PC2']):
    ax2.text(
        val + 0.02, bar.get_y() + bar.get_height()/2, 
        f'{val:.2f}', va='center', fontsize=14, fontweight='bold'
    )

plt.suptitle(
    'PCA Feature Loadings - PC1 vs PC2', 
    fontsize=26, fontweight='bold', y=1.02
)

plt.tight_layout()
plt.savefig(
    '../figures/clustering/pca_loadings_professional.png', 
    dpi=300, bbox_inches='tight', facecolor='white'
)
plt.show()

print("✅ PCA loadings saved to: ../figures/clustering/pca_loadings_professional.png")

# ============================================================================
# PRINT PCA INTERPRETATION
# ============================================================================

print("\n💡 PCA INTERPRETATION:")
print("-" * 60)

# Features with positive PC1
positive_pc1 = loadings[loadings['PC1'] > 0.2].index.tolist()
print("\nFeatures with positive PC1 (Lower Development):")
for feature in positive_pc1:
    print(f"  • {feature}: +{loadings.loc[feature, 'PC1']:.2f}")

# Features with negative PC1
negative_pc1 = loadings[loadings['PC1'] < -0.2].index.tolist()
print("\nFeatures with negative PC1 (Higher Development):")
for feature in negative_pc1:
    print(f"  • {feature}: {loadings.loc[feature, 'PC1']:.2f}")

# Features with positive PC2
positive_pc2 = loadings[loadings['PC2'] > 0.3].index.tolist()
print("\nFeatures with positive PC2 (Trade Orientation):")
for feature in positive_pc2:
    print(f"  • {feature}: +{loadings.loc[feature, 'PC2']:.2f}")

# ============================================================================
# CLUSTER SUMMARY
# ============================================================================

print("\n" + "=" * 80)
print("📊 CLUSTER SUMMARY WITH POLICY PRIORITIES")
print("=" * 80)

for cluster_id in CLUSTER_ORDER:
    print(f"\nCluster {cluster_id}: {CLUSTER_NAMES.get(cluster_id, f'Cluster {cluster_id}')}")
    print(f"  Short Name: {CLUSTER_SHORT_NAMES.get(cluster_id, f'Cluster {cluster_id}')}")
    print(f"  Label: {CLUSTER_LABELS.get(cluster_id, f'Cluster {cluster_id}')}")
    print(f"  Priority: {CLUSTER_PRIORITY.get(cluster_id, 'Not specified')}")
    print(f"  Budget: {CLUSTER_BUDGET.get(cluster_id, 0)}%")
    print(f"  Color: {CLUSTER_COLORS.get(cluster_id, '#808080')}")

In [ ]:
# ============================================================================
# 8. CLUSTER PROFILE VISUALIZATIONS (k=5) 
# ============================================================================

print("📊 CLUSTER PROFILE VISUALIZATIONS")
print("=" * 80)



# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Ensure we have the correct features
numeric_cols = df_with_clusters_final.select_dtypes(include=[np.number]).columns.tolist()
exclude_cols = ['cluster', 'Cluster_Name', 'Cluster_Short', 'Cluster_Label']
numeric_cols = [col for col in numeric_cols if col not in exclude_cols]

print(f"Features for analysis: {numeric_cols}")

# Calculate mean values per cluster
cluster_means = df_with_clusters_final.groupby('cluster')[numeric_cols].mean()

# ============================================================================
# FIGURE 1: HEATMAP OF CLUSTER PROFILES - WITH COMMA FORMATTING
# ============================================================================

print("\n📊 FIGURE 1: Heatmap of Cluster Profiles")

# Standardize for better visualization (z-scores)
cluster_means_std = (cluster_means - cluster_means.mean()) / cluster_means.std()

# ============================================================================
# CREATE ANNOTATION TABLE WITH COMMA FORMATTING
# ============================================================================

# Get the data in the SAME shape as the heatmap will show
# Heatmap shows: rows = Features, columns = Clusters
heatmap_data = cluster_means_std.T  # Transpose: rows = features, columns = clusters
annot_data = cluster_means.T.copy()  # Same shape: rows = features, columns = clusters

# Format numbers with commas for thousands
def format_with_commas(value):
    """Format numbers with commas for thousands separators"""
    if pd.isna(value):
        return ''
    if isinstance(value, (int, float)):
        if abs(value) >= 10000:
            return f'{value:,.0f}'
        elif abs(value) >= 1000:
            return f'{value:,.1f}'
        elif abs(value) < 1 and value != 0:
            return f'{value:.2f}'
        else:
            return f'{value:.1f}'
    return str(value)

# Apply formatting to all values
annot_formatted = annot_data.map(format_with_commas)

# ============================================================================
# DEBUG: Print shapes to verify
# ============================================================================

print(f"\n📐 Data shapes:")
print(f"   heatmap_data shape: {heatmap_data.shape}")
print(f"   annot_formatted shape: {annot_formatted.shape}")
print(f"   Number of features (rows): {len(heatmap_data.index)}")
print(f"   Number of clusters (columns): {len(heatmap_data.columns)}")

# ============================================================================
# CREATE HEATMAP
# ============================================================================

fig, ax = plt.subplots(figsize=(16, 8))

# Create heatmap WITH annotations using annot parameter (Simpler approach)
sns.heatmap(
    heatmap_data,
    annot=annot_formatted.values,  # Use formatted values as strings
    fmt='',  # Empty fmt because we're providing strings
    cmap='viridis',
    center=0,
    linewidths=2,
    linecolor='white',
    cbar_kws={'label': 'Standardized Value', 'shrink': 0.8},
    ax=ax,
    annot_kws={'size': 12, 'fontweight': 'bold', 'color': 'black'}
)

ax.set_title(
    'Cluster Profiles: Mean Values by Feature', 
    fontsize=18, fontweight='bold', pad=20
)
ax.set_xlabel('Cluster', fontsize=14, fontweight='bold')
ax.set_ylabel('Feature', fontsize=14, fontweight='bold')

# Rename x-tick labels using CLUSTER_SHORT_NAMES
cluster_labels = [CLUSTER_SHORT_NAMES.get(i, f'Cluster {i}') for i in CLUSTER_ORDER]
ax.set_xticklabels(cluster_labels, fontsize=12, fontweight='bold', rotation=45, ha='right')

# Rotate y-tick labels for readability
plt.yticks(rotation=0, fontsize=12, fontweight='bold')

# Customize colorbar
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=11)
cbar.set_label('Standardized Value', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/clustering/cluster_profiles_heatmap.png', 
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("✅ Heatmap saved to: ../figures/clustering/cluster_profiles_heatmap.png")
# ============================================================================
# FIGURE 2: RADAR CHART FOR EACH CLUSTER
# ============================================================================

print("\n📊 FIGURE 2: Radar Charts for Each Cluster")

# Select features for radar chart
radar_features = numeric_cols

# Normalize data for radar chart (0-1 scale)
def normalize_radar(data):
    return (data - data.min()) / (data.max() - data.min())

# Calculate normalized means
cluster_means_norm = cluster_means[radar_features].apply(normalize_radar, axis=0)

# Create radar chart
n_clusters = len(CLUSTER_ORDER)
n_cols = 3
n_rows = (n_clusters + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6 * n_rows), 
                         subplot_kw=dict(projection='polar'))
axes = axes.flatten()

# Get colors for each cluster from CLUSTER_COLORS
radar_colors = [CLUSTER_COLORS[i] for i in CLUSTER_ORDER]

# Number of variables
N = len(radar_features)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # Close the loop

for idx, cluster_num in enumerate(CLUSTER_ORDER):
    ax = axes[idx]
    
    # Get values for this cluster
    values = cluster_means_norm.loc[cluster_num].values.flatten().tolist()
    values += values[:1]  # Close the loop
    
    # Get cluster label
    label = CLUSTER_LABELS.get(cluster_num, f'Cluster {cluster_num}')
    color = radar_colors[idx]
    
    # Plot
    ax.plot(angles, values, 'o-', linewidth=3, color=color, 
            label=label)
    ax.fill(angles, values, alpha=0.25, color=color)
    
    # Add feature labels
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([f.replace('_', ' ').title() for f in radar_features], fontsize=9)
    
    # Set y-axis limits
    ax.set_ylim(0, 1.1)
    
    # Title with cluster label
    ax.set_title(f'{label}', fontsize=14, fontweight='bold', pad=20)
    ax.grid(True)

# Hide any unused subplots
for i in range(n_clusters, len(axes)):
    axes[i].axis('off')

# Add legend
plt.suptitle('Cluster Profiles: Radar Chart Comparison', 
            fontsize=20, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('../figures/clustering/cluster_profiles_radar.png', 
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("✅ Radar chart saved to: ../figures/clustering/cluster_profiles_radar.png")

# ============================================================================
# FIGURE 3: GROUPED BAR CHART (CORRECTED)
# ============================================================================

print("\n📊 FIGURE 3: Grouped Bar Chart - All Features Comparison")

# Calculate grid dimensions
n_features = len(numeric_cols)
n_cols = 3
n_rows = (n_features + n_cols - 1) // n_cols

# Create figure with appropriate size
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
if n_rows * n_cols > 1:
    axes = axes.flatten()
else:
    axes = [axes]

# Colors for clusters from CLUSTER_COLORS
bar_colors = [CLUSTER_COLORS[i] for i in CLUSTER_ORDER]

for i, feature in enumerate(numeric_cols):
    ax = axes[i]
    
    # Get values for this feature across clusters
    values = [cluster_means.loc[cluster_num, feature] for cluster_num in CLUSTER_ORDER]
    
    # Create bar chart
    bars = ax.bar(range(len(CLUSTER_ORDER)), values, color=bar_colors, alpha=0.8, 
                  edgecolor='white', linewidth=2)
    
    # Add value labels on top of bars
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05 * max(values),
                f'{val:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    # Customize with cluster labels
    display_name = feature.replace('_', ' ').title()
    ax.set_title(display_name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Cluster', fontsize=10)
    ax.set_ylabel('Mean Value', fontsize=10)
    ax.set_xticks(range(len(CLUSTER_ORDER)))
    ax.set_xticklabels([CLUSTER_LABELS.get(i, f'C{i}') for i in CLUSTER_ORDER], 
                       rotation=45, ha='right', fontsize=9)
    ax.grid(True, alpha=0.2, axis='y')
    ax.set_axisbelow(True)

# Hide any unused subplots
for i in range(n_features, len(axes)):
    axes[i].axis('off')

plt.suptitle('Feature Comparison Across Clusters (k=5)', 
            fontsize=20, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/clustering/cluster_profiles_barchart.png', 
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("✅ Bar chart saved to: ../figures/clustering/cluster_profiles_barchart.png")

# ============================================================================
# FIGURE 4: BOXPLOTS FOR EACH FEATURE BY CLUSTER
# ============================================================================

print("\n📊 FIGURE 4: Boxplots - Feature Distribution by Cluster")

# Select key features for boxplots (avoid too many plots)
key_features = ['child_mort', 'income', 'gdpp', 'inflation', 'life_expec', 'exports']
key_features = [f for f in key_features if f in numeric_cols]

n_key_features = len(key_features)
n_cols = 3
n_rows = (n_key_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6 * n_rows))
if n_rows * n_cols > 1:
    axes = axes.flatten()
else:
    axes = [axes]

# Create boxplot colors for each cluster
boxplot_colors = [CLUSTER_COLORS[i] for i in CLUSTER_ORDER]

for i, feature in enumerate(key_features):
    ax = axes[i]
    
    # Create boxplot with cluster colors
    bp = df_with_clusters_final.boxplot(column=feature, by='cluster', ax=ax,
                                        patch_artist=True,
                                        boxprops=dict(facecolor='lightblue', alpha=0.7),
                                        medianprops=dict(color='darkblue', linewidth=2))
    
    # Customize with cluster labels
    display_name = feature.replace('_', ' ').title()
    ax.set_title(display_name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Cluster', fontsize=11)
    ax.set_ylabel('Value', fontsize=11)
    
    # Update x-tick labels
    ax.set_xticklabels([CLUSTER_LABELS.get(int(label.get_text()), label.get_text()) 
                        for label in ax.get_xticklabels()], 
                       rotation=45, ha='right', fontsize=10)
    ax.grid(True, alpha=0.2)
    ax.set_axisbelow(True)

# Hide any unused subplots
for i in range(n_key_features, len(axes)):
    axes[i].axis('off')

plt.suptitle('Distribution of Key Features by Cluster', 
            fontsize=20, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/clustering/cluster_profiles_boxplots.png', 
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("✅ Boxplots saved to: ../figures/clustering/cluster_profiles_boxplots.png")

# ============================================================================
# FIGURE 5: CLUSTER SIZE AND COMPOSITION - FIXED
# ============================================================================

print("\n📊 FIGURE 5: Cluster Size and Composition")

# Get cluster summary
cluster_summary = df_with_clusters_final.groupby('cluster').size().reset_index(name='Count')
cluster_summary['Percentage'] = (cluster_summary['Count'] / len(df_with_clusters_final) * 100).round(1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))  

# ----------------------------------------------------------------------------
# Plot 1: Cluster sizes (bar chart) - FIXED
# ----------------------------------------------------------------------------

cluster_sizes = [cluster_summary[cluster_summary['cluster'] == i]['Count'].values[0] 
                 for i in CLUSTER_ORDER]
cluster_labels = [CLUSTER_SHORT_NAMES.get(i, f'Cluster {i}') for i in CLUSTER_ORDER]
colors = [CLUSTER_COLORS[i] for i in CLUSTER_ORDER]

# Create bars
bars = ax1.bar(cluster_labels, cluster_sizes, color=colors, 
               edgecolor='white', linewidth=2, alpha=0.8)

# Add value labels on top of bars with proper positioning
max_height = max(cluster_sizes)
for bar, size in zip(bars, cluster_sizes):
    ax1.text(
        bar.get_x() + bar.get_width()/2,
        size + 0.5,  # Position slightly above bar
        f'{size} countries',
        ha='center',
        va='bottom',
        fontsize=12,
        fontweight='bold'
    )

# Customize - FIXED y-axis limit
ax1.set_title('Number of Countries per Cluster', fontsize=15, fontweight='bold', pad=15)
ax1.set_ylabel('Number of Countries', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.2, axis='y')
ax1.set_axisbelow(True)

# Set y-axis limit with proper padding (20% above max value)
ax1.set_ylim(0, max_height * 1.25)

# Rotate x-axis labels
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=30, ha='right', fontsize=11, fontweight='bold')



# ----------------------------------------------------------------------------
# Plot 2: Cluster percentages (pie chart)
# ----------------------------------------------------------------------------

percentages = [cluster_summary[cluster_summary['cluster'] == i]['Percentage'].values[0] 
               for i in CLUSTER_ORDER]

wedges, texts, autotexts = ax2.pie(
    percentages,
    labels=cluster_labels,
    colors=colors,
    autopct='%1.1f%%',
    startangle=90,
    explode=[0.05] * len(CLUSTER_ORDER),
    shadow=True,
    textprops={'fontsize': 11, 'fontweight': 'bold'}
)

# Style the percentage text
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontsize(14)
    autotext.set_fontweight('bold')

ax2.set_title('Cluster Size Distribution', fontsize=15, fontweight='bold', pad=15)

# ----------------------------------------------------------------------------
# Main Title and Save
# ----------------------------------------------------------------------------

plt.suptitle('Cluster Size and Composition (k=5)', 
            fontsize=18, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('../figures/clustering/cluster_sizes.png', 
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("✅ Cluster size chart saved to: ../figures/clustering/cluster_sizes.png")


# ============================================================================
# FIGURE 6: CLUSTER SUMMARY TABLE
# ============================================================================

print("\n📊 FIGURE 6: Cluster Summary Table")

# Create a summary table
summary_data = []
for cluster_id in CLUSTER_ORDER:
    row = {
        'Cluster ID': cluster_id,
        'Name': CLUSTER_NAMES.get(cluster_id, f'Cluster {cluster_id}'),
        'Short Name': CLUSTER_SHORT_NAMES.get(cluster_id, f'Cluster {cluster_id}'),
        'Priority': CLUSTER_PRIORITY.get(cluster_id, 'Not specified'),
        'Budget': f"{CLUSTER_BUDGET.get(cluster_id, 0)}%",
        'Size': cluster_summary[cluster_summary['cluster'] == cluster_id]['Count'].values[0],
        'Percentage': f"{cluster_summary[cluster_summary['cluster'] == cluster_id]['Percentage'].values[0]:.1f}%"
    }
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)

# Create a nice table visualization
fig, ax = plt.subplots(figsize=(14, 6))
ax.axis('tight')
ax.axis('off')

# Create table
table = ax.table(cellText=summary_df.values,
                 colLabels=summary_df.columns,
                 cellLoc='left',
                 loc='center',
                 colColours=['#4472C4'] * len(summary_df.columns))

# Style the table
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)

# Color rows by cluster
for i, cluster_id in enumerate(CLUSTER_ORDER):
    color = CLUSTER_COLORS.get(cluster_id, '#FFFFFF')
    for j in range(len(summary_df.columns)):
        table[(i+1, j)].set_facecolor(color)
        table[(i+1, j)].set_text_props(color='white' if cluster_id in [0, 2] else 'black')

# Style header
for j in range(len(summary_df.columns)):
    table[(0, j)].set_facecolor('#4472C4')
    table[(0, j)].set_text_props(color='white', fontweight='bold')

ax.set_title('Cluster Summary Table', fontsize=18, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('../figures/clustering/cluster_summary_table.png', 
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("✅ Cluster summary table saved to: ../figures/clustering/cluster_summary_table.png")

# ============================================================================
# PRINT SUMMARY STATISTICS
# ============================================================================

print("\n" + "=" * 80)
print("📊 CLUSTER SUMMARY STATISTICS")
print("=" * 80)

for cluster_id in CLUSTER_ORDER:
    print(f"\n{CLUSTER_NAMES.get(cluster_id, f'Cluster {cluster_id}')}:")
    print(f"  Short Name: {CLUSTER_SHORT_NAMES.get(cluster_id, f'Cluster {cluster_id}')}")
    print(f"  Priority: {CLUSTER_PRIORITY.get(cluster_id, 'Not specified')}")
    print(f"  Budget: {CLUSTER_BUDGET.get(cluster_id, 0)}%")
    size = cluster_summary[cluster_summary['cluster'] == cluster_id]['Count'].values[0]
    pct = cluster_summary[cluster_summary['cluster'] == cluster_id]['Percentage'].values[0]
    print(f"  Countries: {size} ({pct:.1f}%)")
    print(f"  Key Features:")
    for feature in ['child_mort', 'income', 'gdpp', 'inflation', 'life_expec']:
        if feature in numeric_cols:
            val = cluster_means.loc[cluster_id, feature]
            print(f"    • {feature.replace('_', ' ').title()}: {val:.2f}")

print("\n✅ All visualizations complete!")
print("   📁 Figures saved to: ../figures/clustering/")

In [ ]:
# ============================================================================
# 9. COMPLETE GEOGRAPHIC CLUSTER VISUALIZATION - USING CONFIG
# ============================================================================

print("🗺️ GENERATING WORLD MAP BY CLUSTER")
print("=" * 80)


# ============================================================================
# STEP 1: PREPARE DATA WITH CLUSTER NAMES AND COLORS
# ============================================================================

print("\n📊 Preparing cluster data...")

# Copy data
map_data = df_with_clusters_final.copy()

# Add cluster names using the configuration
map_data['Cluster_Name'] = map_data['cluster'].map(CLUSTER_NAMES)
map_data['Cluster_Short'] = map_data['cluster'].map(CLUSTER_SHORT_NAMES)
map_data['Cluster_Label'] = map_data['cluster'].map(CLUSTER_LABELS)

# Define colors directly for each cluster ID
CLUSTER_ID_COLORS = {
    0: '#E74C3C',  # Red - Moderate Development
    1: '#2ECC71',  # Green - Advanced Development
    2: '#C0392B',  # Dark Red - Severe Vulnerability
    3: '#3498DB',  # Blue - Trade-Integrated
    4: '#F39C12'   # Orange - Macroeconomic Vulnerability
}

# Add colors
map_data['Cluster_Color'] = map_data['cluster'].map(CLUSTER_ID_COLORS)

# Add priority and budget
map_data['Priority'] = map_data['cluster'].map(CLUSTER_PRIORITY)
map_data['Budget_%'] = map_data['cluster'].map(CLUSTER_BUDGET)

print(f"✅ Data prepared: {len(map_data)} countries")
print(f"   Clusters: {sorted(map_data['cluster'].unique())}")

# ============================================================================
# STEP 2: CREATE PLOTLY MAP (INTERACTIVE)
# ============================================================================

print("\n🗺️ Creating Plotly interactive map...")

try:
    import plotly.graph_objects as go
    
    # Create a figure
    fig = go.Figure()
    
    # Add a trace for each cluster
    for cluster_id in CLUSTER_ORDER:
        # Get data for this cluster
        cluster_data = map_data[map_data['cluster'] == cluster_id]
        cluster_name = CLUSTER_NAMES[cluster_id]
        cluster_short = CLUSTER_SHORT_NAMES[cluster_id]
        color = CLUSTER_ID_COLORS[cluster_id]
        
        # Skip if no data
        if len(cluster_data) == 0:
            continue
        
        # Add choropleth trace for this cluster
        fig.add_trace(
            go.Choropleth(
                locations=cluster_data['country'],
                locationmode='country names',
                z=[1] * len(cluster_data),  # Dummy values for coloring
                colorscale=[[0, color], [1, color]],
                showscale=False,
                name=f"{cluster_id}: {cluster_short}",
                hovertemplate=(
                    '<b>%{location}</b><br>' +
                    f'Cluster: {cluster_name}<br>' +
                    'Child Mortality: %{customdata[0]:.1f} per 1000<br>' +
                    'Income: $%{customdata[1]:,.0f}<br>' +
                    'Life Expectancy: %{customdata[2]:.1f} years<br>' +
                    'Priority: %{customdata[3]}<br>' +
                    'Budget: %{customdata[4]}%<br>' +
                    '<extra></extra>'
                ),
                customdata=cluster_data[['child_mort', 'income', 'life_expec', 'Priority', 'Budget_%']].values
            )
        )
    
    # Update layout
    fig.update_layout(
        title=dict(
            text='Country Clusters by Development Profile (k=5)',
            font=dict(size=24, family='Arial Black', color='#1a1a1a'),
            x=0.5,
            xanchor='center'
        ),
        legend=dict(
            title=dict(
                text='Development Profiles',
                font=dict(size=14, family='Arial Bold')
            ),
            font=dict(size=11, family='Arial'),
            bgcolor='rgba(255,255,255,0.9)',
            bordercolor='#cccccc',
            borderwidth=1,
            x=0.02,
            y=0.98,
            xanchor='left',
            yanchor='top'
        ),
        width=1200,
        height=700,
        margin=dict(l=20, r=20, t=60, b=20),
        paper_bgcolor='white',
        plot_bgcolor='white',
        geo=dict(
            showframe=False,
            showcoastlines=True,
            coastlinecolor='#333333',
            coastlinewidth=0.8,
            projection_type='equirectangular',
            showcountries=True,
            countrycolor='white',
            countrywidth=0.5,
            showland=True,
            landcolor='#F5F5F5',
            showocean=True,
            oceancolor='#E8F4F8',
            showlakes=True,
            lakecolor='#E8F4F8'
        )
    )
    
    # Save as HTML
    fig.write_html('../figures/clustering/cluster_map_interactive.html')
    print("✅ Interactive map saved to: ../figures/clustering/cluster_map_interactive.html")
    
    # Try to save as PNG
    try:
        fig.write_image('../figures/clustering/cluster_map_interactive.png', scale=2)
        print("✅ Map saved as PNG: ../figures/clustering/cluster_map_interactive.png")
    except:
        print("   ⚠️ Could not save PNG (install kaleido: pip install kaleido)")
    
    # Show
    fig.show()
    
except Exception as e:
    print(f"⚠️ Plotly map failed: {e}")
    print("   Continuing with matplotlib...")

# ============================================================================
# STEP 3: CREATE MATPLOTLIB MAP - DISTRIBUTION CHARTS
# ============================================================================

print("\n📊 Creating distribution charts...")

# Create figure with 2x3 layout
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# ============================================================================
# Chart 1: Cluster Sizes (Bar Chart)
# ============================================================================

ax1 = axes[0, 0]
cluster_counts = map_data['cluster'].value_counts().sort_index()
cluster_names_short = [CLUSTER_SHORT_NAMES[i] for i in cluster_counts.index]
colors = [CLUSTER_ID_COLORS[i] for i in cluster_counts.index]

bars = ax1.bar(cluster_names_short, cluster_counts.values, color=colors, 
               edgecolor='white', linewidth=2, alpha=0.8)

# Add value labels
for bar, count in zip(bars, cluster_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{count}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax1.set_title('Number of Countries per Cluster', fontsize=14, fontweight='bold')
ax1.set_ylabel('Number of Countries', fontsize=12)
ax1.grid(True, alpha=0.2, axis='y')
ax1.set_axisbelow(True)
plt.setp(ax1.get_xticklabels(), rotation=15, ha='right')

# ============================================================================
# Chart 2: Cluster Percentages (Pie Chart)
# ============================================================================

ax2 = axes[0, 1]
percentages = (cluster_counts / len(map_data) * 100).values

wedges, texts, autotexts = ax2.pie(
    percentages, 
    labels=cluster_names_short,
    colors=colors,
    autopct='%1.1f%%',
    startangle=90,
    explode=[0.05] * len(cluster_counts),
    shadow=True,
    textprops={'fontsize': 11, 'fontweight': 'bold'}
)

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontsize(12)
    autotext.set_fontweight('bold')

ax2.set_title('Cluster Size Distribution', fontsize=14, fontweight='bold')

# ============================================================================
# Chart 3: Priority Distribution
# ============================================================================

ax3 = axes[0, 2]
priority_counts = map_data['Priority'].value_counts()
priority_colors = {
    'Very High': '#C0392B',
    'High': '#E74C3C',
    'Medium': '#F39C12',
    'Low': '#3498DB',
    'Very Low': '#2ECC71'
}
priority_colors_list = [priority_colors.get(p, '#888888') for p in priority_counts.index]

bars = ax3.bar(priority_counts.index, priority_counts.values, color=priority_colors_list,
               edgecolor='white', linewidth=2, alpha=0.8)

for bar, count in zip(bars, priority_counts.values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{count}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax3.set_title('Priority Distribution', fontsize=14, fontweight='bold')
ax3.set_ylabel('Number of Countries', fontsize=12)
ax3.grid(True, alpha=0.2, axis='y')
ax3.set_axisbelow(True)

# ============================================================================
# Chart 4: Cluster Profiles - Key Indicators (Heatmap-style)
# ============================================================================

ax4 = axes[1, 0]

# Select indicators
indicators = ['child_mort', 'life_expec', 'income', 'gdpp', 'inflation']
indicator_labels = ['Child Mort.', 'Life Exp.', 'Income', 'GDP', 'Inflation']

# Calculate means
cluster_means = map_data.groupby('cluster')[indicators].mean()

# Normalize
cluster_means_norm = (cluster_means - cluster_means.min()) / (cluster_means.max() - cluster_means.min())

# Create heatmap
im = ax4.imshow(cluster_means_norm.values, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=1)

# Add labels
ax4.set_xticks(range(len(indicator_labels)))
ax4.set_xticklabels(indicator_labels, fontsize=10, rotation=45, ha='right')
ax4.set_yticks(range(len(CLUSTER_ORDER)))
ax4.set_yticklabels([CLUSTER_SHORT_NAMES[i] for i in CLUSTER_ORDER], fontsize=10)

# Add value annotations
for i in range(len(CLUSTER_ORDER)):
    for j in range(len(indicators)):
        value = cluster_means.iloc[i, j]
        if indicators[j] in ['income', 'gdpp']:
            text = f'${value:,.0f}'
        elif indicators[j] == 'inflation':
            text = f'{value:.1f}%'
        elif indicators[j] == 'child_mort':
            text = f'{value:.1f}'
        else:
            text = f'{value:.1f}'
        ax4.text(j, i, text, ha='center', va='center', fontsize=8, fontweight='bold')

ax4.set_title('Cluster Profiles - Key Indicators', fontsize=14, fontweight='bold')

# Add colorbar
plt.colorbar(im, ax=ax4, label='Normalized Value', shrink=0.8)

# ============================================================================
# Chart 5: Budget Allocation
# ============================================================================

ax5 = axes[1, 1]

budget_data = []
for cluster_id in CLUSTER_ORDER:
    count = len(map_data[map_data['cluster'] == cluster_id])
    budget = CLUSTER_BUDGET[cluster_id]
    budget_data.append({
        'Cluster': CLUSTER_SHORT_NAMES[cluster_id],
        'Budget': budget,
        'Countries': count,
        'Color': CLUSTER_ID_COLORS[cluster_id]
    })

budget_df = pd.DataFrame(budget_data)
budget_df = budget_df.sort_values('Budget', ascending=True)

bars = ax5.barh(budget_df['Cluster'], budget_df['Budget'], 
                color=budget_df['Color'], edgecolor='white', linewidth=2, alpha=0.8)

# Add labels
for bar, row in zip(bars, budget_df.itertuples()):
    ax5.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f"{row.Budget}% ({row.Countries} countries)", 
             va='center', fontsize=11, fontweight='bold')

ax5.set_xlabel('Budget Allocation (%)', fontsize=12)
ax5.set_title('Budget Allocation by Cluster', fontsize=14, fontweight='bold')
ax5.grid(True, alpha=0.2, axis='x')
ax5.set_axisbelow(True)

# ============================================================================
# Chart 6: Legend / Summary Table
# ============================================================================

ax6 = axes[1, 2]
ax6.axis('off')

# Create summary table
table_data = [['Cluster', 'Name', 'Countries', 'Priority', 'Budget']]
for cluster_id in CLUSTER_ORDER:
    count = len(map_data[map_data['cluster'] == cluster_id])
    table_data.append([
        f"{cluster_id}",
        CLUSTER_SHORT_NAMES[cluster_id],
        f"{count}",
        CLUSTER_PRIORITY[cluster_id],
        f"{CLUSTER_BUDGET[cluster_id]}%"
    ])

# Create table
table = ax6.table(cellText=table_data, loc='center', cellLoc='center',
                  colWidths=[0.1, 0.35, 0.15, 0.2, 0.15])

table.auto_set_font_size(False)
table.set_fontsize(11)

# Style header
for j in range(5):
    cell = table[(0, j)]
    cell.set_facecolor('#2C3E50')
    cell.set_text_props(color='white', fontweight='bold')

# Color rows by cluster
for i, cluster_id in enumerate(CLUSTER_ORDER, 1):
    color = CLUSTER_ID_COLORS[cluster_id]
    for j in range(5):
        cell = table[(i, j)]
        cell.set_facecolor(color)
        cell.set_text_props(color='white', fontweight='bold', alpha=0.9)

ax6.set_title('Cluster Summary', fontsize=14, fontweight='bold', pad=20)

# ============================================================================
# MAIN TITLE AND SAVE
# ============================================================================

plt.suptitle('Country Clusters by Development Profile (k=5)', 
            fontsize=22, fontweight='bold', y=1.02, color='#1a1a1a')

plt.tight_layout()
plt.savefig('../figures/clustering/cluster_analysis_dashboard.png', 
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("✅ Dashboard saved to: ../figures/clustering/cluster_analysis_dashboard.png")

# ============================================================================
# STEP 4: PRINT SUMMARY
# ============================================================================

print("\n" + "=" * 80)
print("📊 CLUSTER SUMMARY WITH COLORS")
print("=" * 80)

for cluster_id in CLUSTER_ORDER:
    name = CLUSTER_NAMES[cluster_id]
    short = CLUSTER_SHORT_NAMES[cluster_id]
    color = CLUSTER_ID_COLORS[cluster_id]
    count = len(map_data[map_data['cluster'] == cluster_id])
    pct = (count / len(map_data)) * 100
    priority = CLUSTER_PRIORITY[cluster_id]
    budget = CLUSTER_BUDGET[cluster_id]
    
    # Color block
    color_block = f"\033[48;5;{int(color[1:3], 16)//36}m  \033[0m"
    
    print(f"\n  Cluster {cluster_id}:")
    print(f"    Full Name  : {name}")
    print(f"    Short Name : {short}")
    print(f"    Color      : {color}")
    print(f"    Countries  : {count} ({pct:.1f}%)")
    print(f"    Priority   : {priority}")
    print(f"    Budget     : {budget}%")

# ============================================================================
# STEP 5: SAVE DATA
# ============================================================================

# Export for external tools
geo_export = map_data[['country', 'cluster', 'Cluster_Name', 'Cluster_Short', 
                       'Cluster_Color', 'Priority', 'Budget_%']].copy()
geo_export.to_csv('../data/processed/cluster_geographic_data.csv', index=False)
print("\n✅ Geographic data saved to: ../data/processed/cluster_geographic_data.csv")

print("\n✅ All geographic visualizations complete!")
print("\n📁 Files saved:")
print("   • ../figures/clustering/cluster_analysis_dashboard.png (6-in-1 dashboard)")
print("   • ../figures/clustering/cluster_map_interactive.html (interactive map)")
print("   • ../data/processed/cluster_geographic_data.csv (data export)")

In [ ]:
# ============================================================================
# 9.1 WORLD MAP VISUALIZATION - COUNTRIES COLORED BY CLUSTER
# ============================================================================

print("🗺️ GENERATING WORLD MAP BY CLUSTER")
print("=" * 80)

# ============================================================================
# STEP 1: Define Colors with Human-Readable Names
# ============================================================================

# Direct color mapping with human-readable color names
COLOR_NAMES = {
    '#E74C3C': 'Red',
    '#2ECC71': 'Green', 
    '#C0392B': 'Dark Red',
    '#3498DB': 'Blue',
    '#F39C12': 'Orange',
    '#888888': 'Gray'
}

CLUSTER_ID_COLORS = {
    0: '#E74C3C',  # Red - Moderate Development
    1: '#2ECC71',  # Green - Advanced Development
    2: '#C0392B',  # Dark Red - Severe Vulnerability
    3: '#3498DB',  # Blue - Trade-Integrated
    4: '#F39C12'   # Orange - Macroeconomic Vulnerability
}

# Map cluster ID to color name
CLUSTER_COLOR_NAMES = {
    cluster_id: COLOR_NAMES[color] for cluster_id, color in CLUSTER_ID_COLORS.items()
}

# ============================================================================
# STEP 2: Prepare Data
# ============================================================================

try:
    import plotly.express as px
    import plotly.graph_objects as go
    
    # Prepare data for map
    map_data = df_with_clusters_final.copy()
    
    # Add cluster names
    map_data['Cluster_Name'] = map_data['cluster'].map(CLUSTER_NAMES)
    map_data['Cluster_Short'] = map_data['cluster'].map(CLUSTER_SHORT_NAMES)
    map_data['Cluster_Color'] = map_data['cluster'].map(CLUSTER_ID_COLORS)
    map_data['Color_Name'] = map_data['cluster'].map(CLUSTER_COLOR_NAMES)
    
    # Create color mapping for Plotly
    cluster_colors = {}
    for cluster_id in CLUSTER_ORDER:
        cluster_name = CLUSTER_NAMES[cluster_id]
        cluster_colors[cluster_name] = CLUSTER_ID_COLORS[cluster_id]
    
    print("\n🎨 Cluster Color Mapping (with Color Names):")
    print("-" * 70)
    for cluster_id in CLUSTER_ORDER:
        name = CLUSTER_NAMES[cluster_id]
        short = CLUSTER_SHORT_NAMES[cluster_id]
        color = CLUSTER_ID_COLORS[cluster_id]
        color_name = CLUSTER_COLOR_NAMES[cluster_id]
        count = len(map_data[map_data['cluster'] == cluster_id])
        print(f"   Cluster {cluster_id}: {short:<25} → {color_name:<10} ({color})  {count} countries")
    print("-" * 70)
    
    # ============================================================================
    # STEP 3: Create Map
    # ============================================================================
    
    fig = px.choropleth(
        map_data,
        locations='country',
        locationmode='country names',
        color='Cluster_Name',
        color_discrete_map=cluster_colors,
        title='Country Clusters by Development Profile (k=5)',
        hover_data={
            'country': True,
            'child_mort': ':.1f',
            'income': '$,.0f',
            'life_expec': ':.1f',
            'total_fer': ':.2f',
            'Cluster_Name': True,
            'Cluster_Short': True,
            'Color_Name': True
        },
        labels={
            'child_mort': 'Child Mortality (per 1000)',
            'income': 'Income per capita ($)',
            'life_expec': 'Life Expectancy (years)',
            'total_fer': 'Total Fertility',
            'Cluster_Name': 'Development Profile',
            'Cluster_Short': 'Cluster',
            'Color_Name': 'Color'
        }
    )
    
    # Update layout
    fig.update_layout(
        title=dict(
            text='Country Clusters by Development Profile (k=5)',
            font=dict(size=24, family='Arial Black', color='#1a1a1a'),
            x=0.5,
            xanchor='center'
        ),
        legend=dict(
            title=dict(
                text='Development Profiles',
                font=dict(size=14, family='Arial Bold')
            ),
            font=dict(size=11, family='Arial'),
            bgcolor='rgba(255,255,255,0.9)',
            bordercolor='#cccccc',
            borderwidth=1,
            x=0.02,
            y=0.98,
            xanchor='left',
            yanchor='top'
        ),
        geo=dict(
            showframe=False,
            showcoastlines=True,
            coastlinecolor='#333333',
            coastlinewidth=0.8,
            projection_type='natural earth',
            showcountries=True,
            countrycolor='white',
            countrywidth=0.5,
            showland=True,
            landcolor='#F5F5F5',
            showocean=True,
            oceancolor='#E8F4F8',
            showlakes=True,
            lakecolor='#E8F4F8'
        ),
        width=1200,
        height=700,
        margin=dict(l=20, r=20, t=60, b=20),
        paper_bgcolor='white',
        plot_bgcolor='white'
    )
    
    # Save
    fig.write_html('../figures/clustering/world_map_clusters.html')
    print("\n✅ Interactive world map saved to: ../figures/clustering/world_map_clusters.html")
    
    try:
        fig.write_image('../figures/clustering/world_map_clusters.png', scale=2)
        print("✅ Static world map saved to: ../figures/clustering/world_map_clusters.png")
    except:
        print("⚠️ Could not save as PNG (kaleido not installed)")
    
    fig.show()
    
except Exception as e:
    print(f"⚠️ World map generation failed: {e}")
    print("   Install plotly with: pip install plotly")

# ============================================================================
# STEP 4: Print Summary with Color Names
# ============================================================================

print("\n" + "=" * 80)
print("📊 CLUSTER SUMMARY WITH COLOR NAMES")
print("=" * 80)
print(f"{'ID':<4} {'Cluster Name':<45} {'Count':<8} {'Color':<12} {'Hex Code':<10}")
print("-" * 80)

for cluster_id in CLUSTER_ORDER:
    name = CLUSTER_NAMES[cluster_id]
    short = CLUSTER_SHORT_NAMES[cluster_id]
    color_hex = CLUSTER_ID_COLORS[cluster_id]
    color_name = CLUSTER_COLOR_NAMES[cluster_id]
    count = len(df_with_clusters_final[df_with_clusters_final['cluster'] == cluster_id])
    pct = (count / len(df_with_clusters_final)) * 100
    print(f"{cluster_id:<4} {short:<45} {count:<8} {color_name:<12} {color_hex:<10}")
print("-" * 80)
print(f"{'':<4} {'TOTAL':<45} {len(df_with_clusters_final):<8}")
print("=" * 80)

# ============================================================================
# STEP 5: Budget and Priority with Color Names
# ============================================================================

print("\n📊 BUDGET & PRIORITY SUMMARY WITH COLORS")
print("=" * 80)
print(f"{'Cluster':<30} {'Countries':<10} {'Priority':<12} {'Budget':<8} {'Color':<12}")
print("-" * 80)

for cluster_id in CLUSTER_ORDER:
    short = CLUSTER_SHORT_NAMES[cluster_id]
    priority = CLUSTER_PRIORITY[cluster_id]
    budget = CLUSTER_BUDGET[cluster_id]
    color_name = CLUSTER_COLOR_NAMES[cluster_id]
    count = len(df_with_clusters_final[df_with_clusters_final['cluster'] == cluster_id])
    print(f"{short:<30} {count:<10} {priority:<12} {budget:>2}%        {color_name:<12}")
print("-" * 80)
print(f"{'TOTAL':<30} {len(df_with_clusters_final):<10}")
print("=" * 80)



# ============================================================================
# STEP 6: Create a Color Legend Figure (ULTRA-COMPACT)
# ============================================================================

print("\n🎨 CREATING COLOR LEGEND FIGURE")
print("=" * 80)

fig, ax = plt.subplots(figsize=(9, 3.5))  # Even smaller
ax.axis('off')

# Create color legend table
legend_data = [['Cluster', 'Name', 'Color', 'Hex', 'Count']]  # Shorter headers
for cluster_id in CLUSTER_ORDER:
    short = CLUSTER_SHORT_NAMES[cluster_id]
    color_hex = CLUSTER_ID_COLORS[cluster_id]
    color_name = CLUSTER_COLOR_NAMES[cluster_id]
    count = len(df_with_clusters_final[df_with_clusters_final['cluster'] == cluster_id])
    color_cell = f"■ {color_name}"
    legend_data.append([
        str(cluster_id),
        short[:20] + '...' if len(short) > 20 else short,  # Truncate long names
        color_cell,
        color_hex,
        str(count)
    ])

# Create table
table = ax.table(cellText=legend_data, 
                 loc='center', 
                 cellLoc='center',
                 colWidths=[0.06, 0.30, 0.22, 0.20, 0.10])

table.auto_set_font_size(False)
table.set_fontsize(10)

# Reduce cell padding
for (i, j), cell in table.get_celld().items():
    cell.set_height(0.07)  # Tighter cells

# Style header
for j in range(5):
    cell = table[(0, j)]
    cell.set_facecolor('#2C3E50')
    cell.set_text_props(color='white', fontweight='bold')

# Color rows
for i, cluster_id in enumerate(CLUSTER_ORDER, 1):
    color_hex = CLUSTER_ID_COLORS[cluster_id]
    for j in range(5):
        cell = table[(i, j)]
        if j == 2:  # Color column
            cell.set_facecolor(color_hex)
            cell.set_text_props(color='white', fontweight='bold')
        else:
            cell.set_facecolor('#F8F9FA' if i % 2 == 0 else '#FFFFFF')

# No title for maximum compactness
plt.tight_layout(pad=0.3)

plt.savefig('../figures/clustering/cluster_color_legend.png', 
            dpi=300, 
            bbox_inches='tight', 
            facecolor='white',
            pad_inches=0.05)

plt.show()

print("✅ Color legend saved to: ../figures/clustering/cluster_color_legend.png")



print("\n✅ All visualizations complete!")
print("\n📁 Files saved:")
print("   • ../figures/clustering/world_map_clusters.html")
print("   • ../figures/clustering/world_map_clusters.png")
print("   • ../figures/clustering/cluster_color_legend.png")

In [ ]:
# ============================================================================
# 9.2 RADAR CHARTS FOR CLUSTER PROFILES
# ============================================================================

print("📊 GENERATING RADAR CHARTS")
print("=" * 80)



# ============================================================================
# STEP 1: Prepare Data for Radar Charts
# ============================================================================

# Define features for radar chart
radar_features = ['child_mort', 'income', 'gdpp', 'life_expec', 'health', 'inflation']

# Get cluster profiles - using your cluster IDs
# Create a DataFrame with cluster means
cluster_means = df_with_clusters_final.groupby('cluster')[radar_features].mean()

# Add cluster names using your CLUSTER_SHORT_NAMES
cluster_means.index = [CLUSTER_SHORT_NAMES[i] for i in cluster_means.index]

# Rename features for better display
feature_labels = {
    'child_mort': 'Child Mortality',
    'income': 'Income',
    'gdpp': 'GDP per capita',
    'life_expec': 'Life Expectancy',
    'health': 'Health Expenditure',
    'inflation': 'Inflation'
}

# ============================================================================
# STEP 2: Normalize Data for Radar (0-1 scale)
# ============================================================================

radar_data = cluster_means.copy()

# For child_mort and inflation, invert (lower is better)
radar_data['child_mort'] = 1 - (radar_data['child_mort'] / radar_data['child_mort'].max())
radar_data['inflation'] = 1 - (radar_data['inflation'] / radar_data['inflation'].max())

# Normalize all features to 0-1
for col in radar_data.columns:
    radar_data[col] = (radar_data[col] - radar_data[col].min()) / (radar_data[col].max() - radar_data[col].min())

# ============================================================================
# STEP 3: Define Colors Using the CLUSTER_ID_COLORS
# ============================================================================

# Use your CLUSTER_ID_COLORS
CLUSTER_ID_COLORS = {
    0: '#E74C3C',  # Red
    1: '#2ECC71',  # Green
    2: '#C0392B',  # Dark Red
    3: '#3498DB',  # Blue
    4: '#F39C12'   # Orange
}

# Map colors to cluster names in order
cluster_names_order = [CLUSTER_SHORT_NAMES[i] for i in CLUSTER_ORDER]
colors = [CLUSTER_ID_COLORS[i] for i in CLUSTER_ORDER]

# ============================================================================
# STEP 4: Create Radar Charts
# ============================================================================

n_clusters = len(radar_data)
n_cols = 3
n_rows = (n_clusters + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6*n_rows), 
                         subplot_kw={'projection': 'polar'})
axes = axes.flatten()

# Number of features
N = len(radar_features)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]  # Close the loop

for i, cluster_name in enumerate(radar_data.index):
    ax = axes[i]
    
    # Get values for this cluster
    values = radar_data.loc[cluster_name].values.flatten().tolist()
    values += values[:1]  # Close the loop
    
    # Plot
    ax.plot(angles, values, 'o-', linewidth=2.5, color=colors[i], 
            label=cluster_name, markersize=6)
    ax.fill(angles, values, alpha=0.25, color=colors[i])
    
    # Customize
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([feature_labels[f] for f in radar_features], size=10, fontweight='bold')
    ax.set_ylim(0, 1)
    
    # Add value labels
    for j, (angle, val) in enumerate(zip(angles[:-1], values[:-1])):
        ax.text(angle, val + 0.05, f'{val:.2f}', 
                ha='center', va='bottom', fontsize=8, fontweight='bold')
    
    # Title with cluster info
    count = len(df_with_clusters_final[df_with_clusters_final['cluster'] == i])
    ax.set_title(f'{cluster_name}\n({count} countries)', 
                size=14, fontweight='bold', pad=20, color=colors[i])
    ax.grid(True, alpha=0.3)

# Remove empty subplots
for i in range(n_clusters, len(axes)):
    fig.delaxes(axes[i])

# ============================================================================
# STEP 5: Add Main Title and Legend
# ============================================================================

plt.suptitle('Cluster Profiles - Radar Chart Comparison', 
            fontsize=20, fontweight='bold', y=1.02)

# Add a single legend
legend_elements = []
for i, cluster_name in enumerate(radar_data.index):
    legend_elements.append(
        plt.Line2D([0], [0], color=colors[i], lw=3, label=cluster_name)
    )
fig.legend(handles=legend_elements, loc='lower center', 
           bbox_to_anchor=(0.5, 0.02), ncol=5, fontsize=11)

plt.tight_layout()
plt.savefig('../figures/clustering/radar_charts.png', 
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("✅ Radar charts saved to: ../figures/clustering/radar_charts.png")

# ============================================================================
# STEP 6: Create Combined Radar Chart (All Clusters Together)
# ============================================================================

print("\n📊 GENERATING COMBINED RADAR CHART")

fig, ax = plt.subplots(figsize=(12, 10), subplot_kw={'projection': 'polar'})

# Plot all clusters on one radar
for i, cluster_name in enumerate(radar_data.index):
    values = radar_data.loc[cluster_name].values.flatten().tolist()
    values += values[:1]
    
    ax.plot(angles, values, 'o-', linewidth=2.5, color=colors[i], 
            label=cluster_name, markersize=6)
    ax.fill(angles, values, alpha=0.15, color=colors[i])

# Customize
ax.set_xticks(angles[:-1])
ax.set_xticklabels([feature_labels[f] for f in radar_features], 
                   size=12, fontweight='bold')
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

# Add radial grid labels
ax.set_rlabel_position(30)
ax.tick_params(labelsize=10)

# Title
ax.set_title('Cluster Profiles Comparison', 
            size=18, fontweight='bold', pad=30, color='#1a1a1a')

# Legend
ax.legend(loc='upper right', bbox_to_anchor=(1.2, 1.0), fontsize=11)

plt.tight_layout()
plt.savefig('../figures/clustering/radar_charts_combined.png', 
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("✅ Combined radar chart saved to: ../figures/clustering/radar_charts_combined.png")

# ============================================================================
# STEP 7: Print Summary Data
# ============================================================================

print("\n" + "=" * 80)
print("📊 RADAR CHART DATA SUMMARY")
print("=" * 80)

print("\nNormalized Values (0-1, higher is better except for inflation):")
print("-" * 80)
print(f"{'Cluster':<25}", end="")
for feat in radar_features:
    print(f"{feat:<15}", end="")
print()
print("-" * 80)

for cluster_name in radar_data.index:
    print(f"{cluster_name:<25}", end="")
    for feat in radar_features:
        val = radar_data.loc[cluster_name, feat]
        print(f"{val:.3f}".ljust(15), end="")
    print()

print("-" * 80)
print("\nLegend:")
print("  • Higher values (closer to 1) = Better outcome")
print("  • For inflation: Higher values = Lower inflation (better)")

print("\n✅ All radar chart visualizations complete!")
print("\n📁 Files saved:")
print("   • ../figures/clustering/radar_charts.png (Individual charts)")
print("   • ../figures/clustering/radar_charts_combined.png (Combined chart)")

In [ ]:
# ============================================================================
# 9.3 PARALLEL COORDINATES PLOT 
# ============================================================================

print("📊 GENERATING ENHANCED PARALLEL COORDINATES PLOT")
print("=" * 80)


# ============================================================================
# STEP 1: Prepare Data
# ============================================================================

# Prepare data for parallel coordinates
parallel_data = df_with_clusters_final.copy()

# Add cluster names using the CLUSTER_NAMES
parallel_data['Cluster_Name'] = parallel_data['cluster'].map(CLUSTER_NAMES)
parallel_data['Cluster_Short'] = parallel_data['cluster'].map(CLUSTER_SHORT_NAMES)

# Select features for visualization with descriptive names
feature_mapping = {
    'child_mort': 'Child Mortality',
    'income': 'Income',
    'gdpp': 'GDP per Capita',
    'life_expec': 'Life Expectancy',
    'inflation': 'Inflation',
    'total_fer': 'Total Fertility',
    'exports': 'Exports',
    'imports': 'Imports',
    'health': 'Health Expenditure'
}

# Choose features for parallel coordinates
parallel_features = ['child_mort', 'income', 'gdpp', 'life_expec', 'inflation', 'total_fer']

# ============================================================================
# STEP 2: Normalize Features
# ============================================================================

# Normalize features for parallel coordinates
scaler = MinMaxScaler()
parallel_data_scaled = parallel_data.copy()
parallel_data_scaled[parallel_features] = scaler.fit_transform(parallel_data_scaled[parallel_features])

# Rename columns for better display
parallel_data_scaled_renamed = parallel_data_scaled.rename(columns=feature_mapping)
parallel_features_renamed = [feature_mapping[f] for f in parallel_features]

# ============================================================================
# STEP 3: Define Colors Using the CLUSTER_ID_COLORS
# ============================================================================

# Define colors using CLUSTER_ID_COLORS
CLUSTER_ID_COLORS = {
    0: '#E74C3C',  # Red - Moderate Development
    1: '#2ECC71',  # Green - Advanced Development
    2: '#C0392B',  # Dark Red - Severe Vulnerability
    3: '#3498DB',  # Blue - Trade-Integrated
    4: '#F39C12'   # Orange - Macroeconomic Vulnerability
}

# Map colors to cluster names
cluster_colors = {}
for cluster_id in CLUSTER_ORDER:
    cluster_name = CLUSTER_NAMES[cluster_id]
    cluster_colors[cluster_name] = CLUSTER_ID_COLORS[cluster_id]

print("\n🎨 Cluster Color Mapping for Parallel Coordinates:")
print("-" * 60)
for cluster_name, color in cluster_colors.items():
    short = CLUSTER_SHORT_NAMES[cluster_id]
    count = len(parallel_data[parallel_data['cluster'] == cluster_id])
    print(f"   {short:<25} → {color} ({count} countries)")
print("-" * 60)

# ============================================================================
# STEP 4: Create Parallel Coordinates Plot
# ============================================================================

# Create figure with enhanced styling
fig, ax = plt.subplots(figsize=(16, 9), dpi=100)
fig.patch.set_facecolor('white')

# Plot parallel coordinates with better transparency and line widths
for cluster_name, color in cluster_colors.items():
    # Filter data for this cluster
    cluster_data = parallel_data_scaled_renamed[
        parallel_data_scaled_renamed['Cluster_Name'] == cluster_name
    ]
    
    # Skip if no data
    if len(cluster_data) == 0:
        continue
    
    # Plot each line with slight transparency for better visualization
    for idx, row in cluster_data.iterrows():
        ax.plot(
            parallel_features_renamed,
            [row[feat] for feat in parallel_features_renamed],
            color=color,
            alpha=0.15,  
            linewidth=1.0,
            solid_capstyle='round'
        )
    
    # Add a thicker mean line for each cluster
    mean_values = cluster_data[parallel_features_renamed].mean()
    ax.plot(
        parallel_features_renamed,
        mean_values,
        color=color,
        alpha=0.9,
        linewidth=3.5,
        label=CLUSTER_SHORT_NAMES.get(CLUSTER_ORDER[list(cluster_colors.keys()).index(cluster_name)], cluster_name),
        solid_capstyle='round'
    )

# Add vertical lines at each axis with better styling
for pos, feature in enumerate(parallel_features_renamed):
    ax.axvline(x=pos, color='#CCCCCC', linestyle='-', linewidth=0.8, alpha=0.5)
    
    # Add subtle background shading for alternate axes
    if pos % 2 == 0:
        ax.axvspan(pos - 0.5, pos + 0.5, alpha=0.03, color='#333333')

# ============================================================================
# STEP 5: Styling
# ============================================================================

ax.set_title('Cluster Profiles Across Development Indicators (k=5)',
             fontsize=18,
             fontweight='bold',
             pad=25,
             color='#1a1a1a')

ax.set_xlabel('Development Indicators', fontsize=13, fontweight='bold', labelpad=10)
ax.set_ylabel('Normalized Value (0-1 Scale)', fontsize=13, fontweight='bold', labelpad=10)

# Customize y-axis
ax.set_ylim(-0.05, 1.05)
ax.set_yticks(np.arange(0, 1.1, 0.2))
ax.set_yticklabels(['0.0', '0.2', '0.4', '0.6', '0.8', '1.0'])
ax.grid(True, alpha=0.15, linestyle='-', linewidth=0.5, axis='y')
ax.set_axisbelow(True)

# Remove top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#333333')
ax.spines['bottom'].set_color('#333333')
ax.spines['left'].set_linewidth(1.2)
ax.spines['bottom'].set_linewidth(1.2)

# Customize x-axis tick labels
ax.set_xticks(range(len(parallel_features_renamed)))
ax.set_xticklabels(parallel_features_renamed, rotation=30, ha='right', fontsize=11, weight='medium')

# ============================================================================
# STEP 6: Enhanced Legend
# ============================================================================

# Create custom legend handles with cluster colors
legend_handles = []
for cluster_id in CLUSTER_ORDER:
    short_name = CLUSTER_SHORT_NAMES[cluster_id]
    color = CLUSTER_ID_COLORS[cluster_id]
    count = len(parallel_data[parallel_data['cluster'] == cluster_id])
    legend_handles.append(
        Line2D([0], [0], color=color, lw=3, label=f"{short_name} ({count} countries)")
    )

legend = ax.legend(
    handles=legend_handles,
    loc='upper right',
    fontsize=10,
    frameon=True,
    fancybox=True,
    shadow=True,
    edgecolor='#CCCCCC',
    title='Development Profiles',
    title_fontsize=12,
    markerfirst=False
)
legend.get_frame().set_facecolor('white')
legend.get_frame().set_alpha(0.95)

# Add a background highlight for the legend
legend_box = legend.get_frame()
legend_box.set_linewidth(1.5)

# ============================================================================
# STEP 7: Add Annotations
# ============================================================================

# Add annotation with interpretation tips
ax.text(
    0.02, -0.12,
    'Note: Thick lines represent cluster averages. Countries with similar development\n'
    'patterns show similar profiles across indicators. Higher values indicate better outcomes,\n'
    'except for Child Mortality, Inflation, and Total Fertility where lower is better.',
    transform=ax.transAxes,
    fontsize=9,
    style='italic',
    verticalalignment='bottom',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.9, edgecolor='#CCCCCC')
)

# Add value grid lines for better readability
for y in [0.25, 0.50, 0.75]:
    ax.axhline(y=y, color='#E0E0E0', linestyle=':', linewidth=0.8, alpha=0.5, zorder=0)

# ============================================================================
# STEP 8: Save and Display
# ============================================================================

plt.tight_layout()
plt.subplots_adjust(bottom=0.15)

# Save high quality
plt.savefig('../figures/clustering/parallel_coordinates.png',
            dpi=300,
            bbox_inches='tight',
            facecolor='white',
            edgecolor='none')

plt.show()

print("\n✅ Enhanced parallel coordinates plot saved to: ../figures/clustering/parallel_coordinates.png")

# ============================================================================
# STEP 9: Print Summary
# ============================================================================

print("\n" + "=" * 80)
print("📊 PARALLEL COORDINATES SUMMARY")
print("=" * 80)

print("\nCluster Averages (Normalized 0-1):")
print("-" * 80)
print(f"{'Cluster':<25}", end="")
for feat in parallel_features:
    print(f"{feature_mapping[feat][:12]:<15}", end="")
print()
print("-" * 80)

for cluster_id in CLUSTER_ORDER:
    cluster_name = CLUSTER_SHORT_NAMES[cluster_id]
    cluster_data = parallel_data_scaled[parallel_data_scaled['cluster'] == cluster_id]
    if len(cluster_data) > 0:
        means = cluster_data[parallel_features].mean()
        print(f"{cluster_name:<25}", end="")
        for feat in parallel_features:
            print(f"{means[feat]:.3f}".ljust(15), end="")
        print()
print("-" * 80)

print("\n💡 Interpretation:")
print("   • Child Mortality: Lower is better")
print("   • Inflation: Lower is better")
print("   • Total Fertility: Lower is better")
print("   • Income, GDP, Life Expectancy, Health: Higher is better")

print("\n📁 Files saved:")
print("   • ../figures/clustering/parallel_coordinates.png")

In [ ]:
# ============================================================================
# 10 ANOVA + KRUSKAL-WALLIS ANALYSIS
# ============================================================================


print("📊 ANOVA AND KRUSKAL-WALLIS ANALYSIS")
print("=" * 80)

# ============================================================================
# 1. FEATURES AND FINAL CLUSTER ASSIGNMENTS
# ============================================================================

feature_names = [
    'child_mort',
    'exports',
    'health',
    'imports',
    'income',
    'inflation',
    'life_expec',
    'total_fer',
    'gdpp'
]

# IMPORTANT:
# final_labels contains the 167 country-level cluster assignments.
labels_array = np.asarray(final_labels).ravel()

# Check dimensions
print(f"Number of countries: {len(df_processed)}")
print(f"Number of cluster assignments: {len(labels_array)}")
print(f"Clusters: {np.sort(np.unique(labels_array))}")

if len(df_processed) != len(labels_array):
    raise ValueError(
        "df_processed and final_labels must contain the same number of observations."
    )

# ============================================================================
# 2. CALCULATE ANOVA AND KRUSKAL-WALLIS
# ============================================================================

anova_results = []
kruskal_results = []

unique_clusters = np.sort(np.unique(labels_array))

for feature in feature_names:

    # Extract feature values
    values = df_processed[feature].to_numpy()

    # Divide observations according to final cluster assignment
    groups = [
        values[labels_array == cluster]
        for cluster in unique_clusters
    ]

    # ------------------------------------------------------------
    # One-way ANOVA
    # ------------------------------------------------------------

    f_stat, p_value = f_oneway(*groups)

    # ------------------------------------------------------------
    # Eta squared
    # ------------------------------------------------------------

    grand_mean = np.mean(values)

    ss_between = sum(
        len(group) * (np.mean(group) - grand_mean) ** 2
        for group in groups
    )

    ss_total = np.sum(
        (values - grand_mean) ** 2
    )

    eta_squared = ss_between / ss_total

    # ------------------------------------------------------------
    # Kruskal-Wallis
    # ------------------------------------------------------------

    h_stat, kw_p_value = kruskal(*groups)

    anova_results.append({
        'Feature': feature,
        'F': f_stat,
        'p': p_value,
        'Eta²': eta_squared
    })

    kruskal_results.append({
        'Feature': feature,
        'H': h_stat,
        'p': kw_p_value
    })

# ============================================================================
# 3. CREATE RESULTS DATAFRAMES
# ============================================================================

anova_report = pd.DataFrame(anova_results)
kruskal_report = pd.DataFrame(kruskal_results)

# ============================================================================
# 4. READABLE FEATURE NAMES
# ============================================================================

feature_labels_readable = {
    'child_mort': 'Child Mortality',
    'exports': 'Exports',
    'health': 'Health Expenditure',
    'imports': 'Imports',
    'income': 'Income',
    'inflation': 'Inflation',
    'life_expec': 'Life Expectancy',
    'total_fer': 'Total Fertility',
    'gdpp': 'GDP per Capita'
}

anova_report['Feature'] = anova_report['Feature'].map(
    feature_labels_readable
)

kruskal_report['Feature'] = kruskal_report['Feature'].map(
    feature_labels_readable
)

# ============================================================================
# 5. SORT ANOVA RESULTS BY EFFECT SIZE
# ============================================================================

anova_report = anova_report.sort_values(
    'Eta²',
    ascending=False
).reset_index(drop=True)

# ============================================================================
# 6. CREATE CLEAN ANOVA TABLE FOR REPORT
# ============================================================================

anova_table = anova_report[
    ['Feature', 'F', 'p', 'Eta²']
].copy()

anova_table['F'] = anova_table['F'].map(
    lambda x: f'{x:.2f}'
)

anova_table['p'] = anova_table['p'].map(
    lambda x: '<0.001' if x < 0.001 else f'{x:.3f}'
)

anova_table['Eta²'] = anova_table['Eta²'].map(
    lambda x: f'{x:.3f}'
)

print("\n📋 ANOVA RESULTS")
print("=" * 80)

display(anova_table)

# ============================================================================
# 7. KRUSKAL-WALLIS ROBUSTNESS CHECK
# ============================================================================

kruskal_report = kruskal_report.set_index('Feature')

significant_kw = kruskal_report['p'] < 0.05

print("\n📋 KRUSKAL-WALLIS ROBUSTNESS CHECK")
print("=" * 80)

print(
    f"Significant features: "
    f"{significant_kw.sum()}/{len(feature_names)}"
)

if significant_kw.all():
    print("All nine indicators show significant differences across clusters (p < 0.05).")

# ============================================================================
# 8. STRONGEST BETWEEN-CLUSTER DIFFERENCES
# ============================================================================

print("\n📊 STRONGEST BETWEEN-CLUSTER DIFFERENCES")
print("=" * 80)

for _, row in anova_report.head(3).iterrows():
    print(
        f"• {row['Feature']}: η² = {row['Eta²']:.3f}"
    )

# ============================================================================
# 9. SAVE RESULTS
# ============================================================================

anova_report.to_csv(
    '../data/processed/anova_results.csv',
    index=False
)

kruskal_report.to_csv(
    '../data/processed/kruskal_wallis_results.csv'
)

print("\n✅ ANOVA results saved.")
print("✅ Kruskal-Wallis results saved.")

In [ ]:
# ============================================================================
# 11. BUDGET ALLOCATION STRATEGY
# ============================================================================

print("💰 ILLUSTRATIVE NEED-BASED BUDGET ALLOCATION SCENARIO")
print("=" * 80)



# ============================================================================
# STEP 1: TOTAL BUDGET
# ============================================================================

TOTAL_BUDGET = 1_000_000_000  # $1 Billion


# ============================================================================
# STEP 2: POLICY-BASED CLUSTER ALLOCATION
# ============================================================================
#
# IMPORTANT:
# This is an illustrative need-based policy scenario, NOT an optimization
# result or an empirically estimated funding formula.
#
# The clustering analysis identifies groups of countries with similar
# multidimensional socioeconomic profiles. The budget allocation then
# translates these profiles into a simple policy-priority framework.
#
# GENERAL ALLOCATION PRINCIPLE:
#
#   Greater development vulnerability
#                 ↓
#        Higher budget priority
#
# The allocation reflects the relative development and policy needs
# identified from the cluster profiles:
#
#   Cluster 2 - Severe Development Vulnerability
#       Very High Priority
#       Countries in this cluster exhibit the most severe health,
#       demographic, and economic vulnerabilities, including very high
#       child mortality, low life expectancy, very low income/GDP, and
#       high fertility. They therefore receive the largest allocation
#       within this illustrative framework.
#
#   Cluster 0 - Developing Economies with Moderate Human Development
#       High Priority
#       Countries in this cluster have substantial development gaps in
#       health, income, and socioeconomic conditions, although their
#       outcomes are considerably stronger than those of Cluster 2.
#
#   Cluster 4 - Economies with Macroeconomic Vulnerability
#       High Priority
#       Countries in this cluster have relatively strong income and
#       moderate human-development outcomes, but exhibit distinctive
#       macroeconomic vulnerability, particularly high inflation and
#       relatively low health expenditure. Funding priorities therefore
#       include macroeconomic stabilization and continued investment in
#       health and social development.
#
#   Cluster 3 - Trade-Integrated Economies
#       Medium Priority
#       Countries in this cluster have relatively strong development
#       outcomes and exceptionally high trade integration. Policy
#       priorities can therefore emphasize trade resilience,
#       diversification, and sustainable economic integration rather
#       than basic development assistance.
#
#   Cluster 1 - Advanced Human Development Economies
#       Very Low Priority
#       Countries in this cluster have the strongest overall
#       human-development and socioeconomic profile in the dataset.
#       They therefore receive the smallest allocation within this
#       illustrative development-assistance framework.
#
# IMPORTANT:
# The percentages assigned to each cluster are policy assumptions used
# to demonstrate how clustering can inform resource allocation.
# They are NOT statistically optimized funding levels and should not be
# interpreted as a causal or empirically validated budget formula.
#
# The allocation is therefore intended as a transparent policy scenario
# that links observed cluster characteristics to differentiated funding
# priorities.
#
# Total allocation = 100%
# ============================================================================

CLUSTER_BUDGET = {
    0: 0.25,   # Developing Economies with Moderate Human Development
    1: 0.02,   # Advanced Human Development Economies
    2: 0.45,   # Severe Development Vulnerability
    3: 0.08,   # Trade-Integrated Economies
    4: 0.20    # Economies with Macroeconomic Vulnerability
}


# ============================================================================
# STEP 3: POLICY PRIORITY
# ============================================================================

CLUSTER_PRIORITY = {
    0: "High",
    1: "Very Low",
    2: "Very High",
    3: "Medium",
    4: "High"
}


# ============================================================================
# STEP 4: ALLOCATION RATIONALE
# ============================================================================

CLUSTER_ALLOCATION_RATIONALE = {

    0: (
        "Substantial development gaps remain in health, income, "
        "and socioeconomic conditions."
    ),

    1: (
        "Strongest overall human-development and socioeconomic profile; "
        "lowest development-assistance priority."
    ),

    2: (
        "Most severe health, demographic, and economic vulnerabilities; "
        "highest priority for basic development and poverty-reduction "
        "interventions."
    ),

    3: (
        "Relatively strong development outcomes with very high trade "
        "integration; emphasis can be placed on resilience, diversification, "
        "and sustainable economic integration."
    ),

    4: (
        "Relatively strong development indicators but substantial "
        "macroeconomic vulnerability, particularly high inflation and "
        "lower health expenditure."
    )
}


# ============================================================================
# STEP 5: VERIFY ALLOCATION
# ============================================================================

total_allocation = sum(CLUSTER_BUDGET.values())

assert np.isclose(total_allocation, 1.0), (
    f"Budget allocation must equal 100%, "
    f"but currently equals {total_allocation * 100:.1f}%."
)

print(f"\nTotal Budget: ${TOTAL_BUDGET:,.0f}")
print(f"Total Allocation: {total_allocation * 100:.0f}%")


# ============================================================================
# STEP 6: CREATE COUNTRY-LEVEL ALLOCATION
# ============================================================================

def assign_budget(df, cluster_column, total_budget, cluster_budget):
    """
    Assign cluster-level and per-country budget allocations.

    The total budget is first allocated across clusters according to the
    policy scenario. Within each cluster, the allocated amount is divided
    equally among countries in that cluster.

    Parameters
    ----------
    df : pandas.DataFrame
        Country-level dataset containing cluster assignments.

    cluster_column : str
        Name of the cluster assignment column.

    total_budget : float
        Total available budget.

    cluster_budget : dict
        Dictionary mapping cluster IDs to budget shares.

    Returns
    -------
    pandas.DataFrame
        Country-level budget allocation dataset.
    """

    budget_df = df.copy()

    # ------------------------------------------------------------------------
    # Number of countries in each cluster
    # ------------------------------------------------------------------------

    cluster_counts = (
        budget_df[cluster_column]
        .value_counts()
        .to_dict()
    )

    # ------------------------------------------------------------------------
    # Cluster budget share (%)
    # ------------------------------------------------------------------------

    budget_df["Cluster_Budget_%"] = (
        budget_df[cluster_column]
        .map(cluster_budget)
        * 100
    )

    # ------------------------------------------------------------------------
    # Total budget assigned to the country's cluster
    # ------------------------------------------------------------------------

    budget_df["Cluster_Budget_$"] = (
        budget_df[cluster_column]
        .map(cluster_budget)
        * total_budget
    )

    # ------------------------------------------------------------------------
    # Calculate equal per-country allocation within each cluster
    # ------------------------------------------------------------------------

    per_country_budget = {}

    for cluster_id, share in cluster_budget.items():

        n_countries = cluster_counts.get(cluster_id, 0)

        if n_countries > 0:

            per_country_budget[cluster_id] = (
                total_budget * share / n_countries
            )

        else:

            per_country_budget[cluster_id] = 0

    # ------------------------------------------------------------------------
    # Assign per-country budget
    # ------------------------------------------------------------------------

    budget_df["Per_Country_Budget_$"] = (
        budget_df[cluster_column]
        .map(per_country_budget)
    )

    return budget_df


# Apply allocation
budget_df = assign_budget(
    df=df_with_clusters_final,
    cluster_column="cluster",
    total_budget=TOTAL_BUDGET,
    cluster_budget=CLUSTER_BUDGET
)


# ============================================================================
# STEP 7: ADD CLUSTER INFORMATION
# ============================================================================

budget_df["Cluster_Name"] = (
    budget_df["cluster"].map(CLUSTER_NAMES)
)

budget_df["Cluster_Short"] = (
    budget_df["cluster"].map(CLUSTER_SHORT_NAMES)
)

budget_df["Priority"] = (
    budget_df["cluster"].map(CLUSTER_PRIORITY)
)

budget_df["Allocation_Rationale"] = (
    budget_df["cluster"].map(CLUSTER_ALLOCATION_RATIONALE)
)


# ============================================================================
# STEP 8: CREATE CLUSTER-LEVEL SUMMARY
# ============================================================================

budget_summary = (
    budget_df
    .groupby("cluster")
    .agg(
        Countries=("country", "count"),
        Total_Budget=("Cluster_Budget_$", "first"),
        Per_Country=("Per_Country_Budget_$", "first"),
        Budget_Percent=("Cluster_Budget_%", "first"),
        Cluster_Name=("Cluster_Name", "first"),
        Cluster_Short=("Cluster_Short", "first"),
        Priority=("Priority", "first"),
        Allocation_Rationale=("Allocation_Rationale", "first")
    )
    .sort_index()
)


# ============================================================================
# STEP 9: DISPLAY BUDGET ALLOCATION
# ============================================================================

print("\n" + "=" * 80)
print("📊 BUDGET ALLOCATION BY CLUSTER")
print("=" * 80)

for cluster_id in CLUSTER_ORDER:

    row = budget_summary.loc[cluster_id]

    print(f"\nCluster {cluster_id}: {row['Cluster_Name']}")
    print("-" * 70)

    print(f"Countries:        {int(row['Countries'])}")
    print(f"Priority:         {row['Priority']}")
    print(f"Budget Share:     {row['Budget_Percent']:.0f}%")
    print(f"Total Allocation: ${row['Total_Budget']:,.0f}")
    print(f"Per Country:      ${row['Per_Country']:,.2f}")

    print(f"Rationale:        {row['Allocation_Rationale']}")


# ============================================================================
# STEP 10: VERIFY TOTAL BUDGET
# ============================================================================

allocated_total = budget_summary["Total_Budget"].sum()

print("\n" + "=" * 80)
print("💰 BUDGET VERIFICATION")
print("=" * 80)

print(f"Required Budget:  ${TOTAL_BUDGET:,.0f}")
print(f"Allocated Budget: ${allocated_total:,.0f}")
print(f"Difference:       ${allocated_total - TOTAL_BUDGET:,.2f}")

assert np.isclose(
    allocated_total,
    TOTAL_BUDGET,
    rtol=0,
    atol=1
), "Allocated budget does not equal total budget."

print("✅ Budget allocation verified.")


# ============================================================================
# STEP 11: CREATE SIMPLE VISUALIZATION
# ============================================================================

cluster_names = [
    CLUSTER_SHORT_NAMES[i]
    for i in CLUSTER_ORDER
]

budget_values = [
    CLUSTER_BUDGET[i] * 100
    for i in CLUSTER_ORDER
]


fig, ax = plt.subplots(figsize=(11, 6))

bars = ax.bar(
    cluster_names,
    budget_values
)


# Add percentages above bars
for bar, value in zip(bars, budget_values):

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.8,
        f"{value:.0f}%",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold"
    )


ax.set_title(
    "Illustrative Need-Based Budget Allocation by Development Profile",
    fontsize=15,
    fontweight="bold"
)

ax.set_ylabel("Budget Allocation (%)")

ax.set_ylim(
    0,
    max(budget_values) + 8
)

ax.grid(
    axis="y",
    alpha=0.2
)

ax.set_axisbelow(True)

plt.xticks(
    rotation=15,
    ha="right"
)

plt.tight_layout()


plt.savefig(
    "../figures/clustering/budget_allocation.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()


print(
    "\n✅ Budget visualization saved to: "
    "../figures/clustering/budget_allocation.png"
)


# ============================================================================
# STEP 12: SAVE COUNTRY-LEVEL RESULTS
# ============================================================================

budget_df.to_csv(
    "../data/processed/budget_allocation_final.csv",
    index=False
)

print("\n✅ Country-level allocation saved:")
print("   ../data/processed/budget_allocation_final.csv")


# ============================================================================
# STEP 13: SAVE CLUSTER-LEVEL RESULTS
# ============================================================================

budget_summary.to_csv(
    "../data/processed/budget_allocation_summary.csv"
)

print("\n✅ Cluster-level summary saved:")
print("   ../data/processed/budget_allocation_summary.csv")


# ============================================================================
# STEP 14: FINAL SUMMARY TABLE
# ============================================================================

print("\n" + "=" * 80)
print("📋 FINAL BUDGET ALLOCATION")
print("=" * 80)


final_table = budget_summary[
    [
        "Cluster_Name",
        "Countries",
        "Priority",
        "Budget_Percent",
        "Total_Budget",
        "Per_Country"
    ]
].copy()


# Format values for display

final_table["Total_Budget"] = (
    final_table["Total_Budget"]
    .map("${:,.0f}".format)
)

final_table["Per_Country"] = (
    final_table["Per_Country"]
    .map("${:,.2f}".format)
)

final_table["Budget_Percent"] = (
    final_table["Budget_Percent"]
    .map("{:.0f}%".format)
)


print(final_table.to_string())


# ============================================================================
# STEP 15: PRINT POLICY INTERPRETATION
# ============================================================================

print("\n" + "=" * 80)
print("📝 POLICY INTERPRETATION")
print("=" * 80)

print(
    """
This allocation represents an illustrative need-based policy scenario.

The clustering analysis identifies countries with similar multidimensional
socioeconomic characteristics. The budget allocation applies a policy
priority framework to these clusters rather than treating the cluster
structure itself as an optimization of funding levels.

The largest share is assigned to Severe Development Vulnerability because
this group exhibits the most severe health, demographic, and economic
challenges.

Developing Economies with Moderate Human Development and Economies with
Macroeconomic Vulnerability receive substantial allocations because they
continue to face important development or stabilization needs.

Trade-Integrated Economies receive a smaller allocation because their
overall development outcomes are relatively strong, while their policy
needs are more focused on trade resilience and sustainable integration.

Advanced Human Development Economies receive the smallest share because
they have the strongest overall socioeconomic and human-development profile.

The allocation should therefore be interpreted as an illustrative policy
scenario rather than an empirically optimized budget formula.
"""
)


# ============================================================================
# STEP 16: FINAL STATUS
# ============================================================================

print("\n" + "=" * 80)
print("✅ BUDGET ALLOCATION ANALYSIS COMPLETE")
print("=" * 80)

print(f"\nTotal Budget: ${TOTAL_BUDGET:,.0f}")
print("Allocation: 100%")
print("Method: Illustrative need-based policy scenario")
print("Country allocation: Equal within each cluster")

In [ ]:
# ============================================================================
# 12. BUDGET VISUALIZATION
# ============================================================================

print("📊 BUDGET VISUALIZATION")
print("=" * 80)



# ============================================================================
# STEP 1: Get Cluster Data in Order
# ============================================================================

# Get clusters in order
cluster_ids = CLUSTER_ORDER
cluster_names_list = [CLUSTER_SHORT_NAMES[i] for i in cluster_ids]
cluster_full_names = [CLUSTER_NAMES[i] for i in cluster_ids]

# Get budgets in order
budget_values = [
    CLUSTER_BUDGET[cluster_id] * TOTAL_BUDGET / 1_000_000
    for cluster_id in cluster_ids
]

budget_pct = [
    CLUSTER_BUDGET[cluster_id] * 100
    for cluster_id in cluster_ids
]

# Get colors in order
CLUSTER_ID_COLORS = {
    0: '#E74C3C',  # Red - Intermediate Development
    1: '#2ECC71',  # Green - High Development
    2: '#C0392B',  # Dark Red - High Development Vulnerability
    3: '#3498DB',  # Blue - Trade-Intensive
    4: '#F39C12'   # Orange - Higher Inflation
}

colors = [CLUSTER_ID_COLORS[i] for i in cluster_ids]

# ============================================================================
# STEP 2: Create Figure
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# ============================================================================
# PLOT 1: BAR CHART
# ============================================================================

ax1 = axes[0]

bars = ax1.bar(
    cluster_names_list,
    budget_values,
    color=colors,
    edgecolor='white',
    linewidth=2,
    alpha=0.85
)

# Add budget values above bars
for bar, value, pct in zip(bars, budget_values, budget_pct):
    height = bar.get_height()
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        height + 2,
        f'${value:.0f}M\n({pct}%)',
        ha='center',
        va='bottom',
        fontsize=15,  
        fontweight='bold',
        color='#2C3E50'
    )

# Customize
ax1.set_xlabel('Development Cluster', fontsize=16, fontweight='bold', labelpad=10)  
ax1.set_ylabel('Budget Allocation (Millions USD)', fontsize=16, fontweight='bold', labelpad=10)  
ax1.set_title('Budget Allocation by Development Cluster', fontsize=17, fontweight='bold', pad=15)  

# Add grid
ax1.grid(True, alpha=0.2, axis='y')
ax1.set_axisbelow(True)

# Set y-axis limit with some padding
max_budget = max(budget_values)
ax1.set_ylim(0, max_budget * 1.25)

# Rotate x-axis labels
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=20, ha='right', fontsize=13, fontweight='bold') 

# Add a horizontal line for total average
avg_budget = sum(budget_values) / len(budget_values)
ax1.axhline(y=avg_budget, color='#7F8C8D', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Average: ${avg_budget:.0f}M')
ax1.legend(loc='upper right', fontsize=12)  # +2 (was 10)


# ============================================================================
# PLOT 2: PIE CHART
# ============================================================================

ax2 = axes[1]

# Labels without percentage
labels = cluster_names_list

# Explode the most important clusters
explode = [0.08 if pct >= 30 else 0.04 if pct >= 20 else 0.02 for pct in budget_pct]

wedges, texts, autotexts = ax2.pie(
    budget_pct,
    labels=labels,
    autopct='%1.1f%%',
    colors=colors,
    explode=explode,
    shadow=True,
    startangle=90,
    textprops={'fontsize': 14, 'fontweight': 'bold'}  
)

# Style the percentage text
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontsize(16)  
    autotext.set_fontweight('bold')

ax2.set_title('Budget Distribution by Cluster', fontsize=17, fontweight='bold', pad=15)  


# ============================================================================
# MAIN TITLE
# ============================================================================

plt.suptitle(
    f'Development Budget Allocation (${TOTAL_BUDGET:,.0f} Total)',
    fontsize=20,  # +2 (was 18)
    fontweight='bold',
    y=1.05,
    color='#1a1a1a'
)

# ============================================================================
# SAVE
# ============================================================================

plt.tight_layout()
plt.savefig(
    '../figures/clustering/budget_allocation_final.png',
    dpi=300,
    bbox_inches='tight',
    facecolor='white'
)

plt.show()

print("\n✅ Budget visualization saved to: ../figures/clustering/budget_allocation_final.png")

# ============================================================================
# STEP 3: Print Summary
# ============================================================================

print("\n" + "=" * 80)
print("💰 BUDGET ALLOCATION SUMMARY")
print("=" * 80)

print(f"\nTotal Budget: ${TOTAL_BUDGET:,.0f}")

print("\nAllocation by Cluster:")
print("-" * 70)

print(
    f"{'Cluster':<8}"
    f"{'Name':<30}"
    f"{'Budget %':<10}"
    f"{'Amount ($M)':<15}"
    f"{'Per Country ($)':<18}"
)

print("-" * 70)

for cluster_id in cluster_ids:

    name = CLUSTER_SHORT_NAMES[cluster_id]
    budget_share = CLUSTER_BUDGET[cluster_id]
    budget_percentage = budget_share * 100
    amount_m = budget_share * TOTAL_BUDGET / 1_000_000
    count = len(df_with_clusters_final[df_with_clusters_final["cluster"] == cluster_id])
    per_country = budget_share * TOTAL_BUDGET / count if count > 0 else 0

    print(
        f"{cluster_id:<8}"
        f"{name:<30}"
        f"{budget_percentage:>6.1f}%   "
        f"${amount_m:>11,.1f}M   "
        f"${per_country:>14,.2f}"
    )

print("-" * 70)

total_allocated = sum(CLUSTER_BUDGET[i] * TOTAL_BUDGET for i in cluster_ids)

print(
    f"{'':<8}"
    f"{'TOTAL':<30}"
    f"{sum(CLUSTER_BUDGET[i] for i in cluster_ids) * 100:>6.1f}%   "
    f"${total_allocated / 1_000_000:>11,.1f}M"
)

print("=" * 80)

# ============================================================================
# STEP 4: Create Additional Budget Visualization - FIXED
# ============================================================================

print("\n📊 CREATING PER COUNTRY BUDGET CHART")

fig, ax = plt.subplots(figsize=(16, 9))

# Calculate per country budget
per_country_budget = []
cluster_counts = []

for cluster_id in cluster_ids:
    count = len(df_with_clusters_final[df_with_clusters_final['cluster'] == cluster_id])
    cluster_counts.append(count)
    pct = CLUSTER_BUDGET[cluster_id]
    per_country = (pct * TOTAL_BUDGET) / count if count > 0 else 0
    per_country_budget.append(per_country)

# Calculate max value for y-axis limit with proper padding
max_per_country = max(per_country_budget)
y_limit = max_per_country * 1.25

bars = ax.bar(
    cluster_names_list,
    per_country_budget,
    color=colors,
    edgecolor='white',
    linewidth=2,
    alpha=0.85
)

# Add value labels on top of bars
for bar, value in zip(bars, per_country_budget):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + (max_per_country * 0.04),
        f'${value:,.0f}',
        ha='center',
        va='bottom',
        fontsize=16,
        fontweight='bold',
        color='#2C3E50'
    )

# Add count annotations below the bars 
for i, (cluster_id, bar, count) in enumerate(zip(cluster_ids, bars, cluster_counts)):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        0.03 * max_per_country,
        f'n={count}',
        ha='center',
        va='bottom',
        fontsize=15,
        fontweight='bold',
        color='white',
        bbox=dict(boxstyle="round,pad=0.3", facecolor='black', alpha=0.6, edgecolor='none')  
    )

# Customize
ax.set_xlabel('Development Cluster', fontsize=16, fontweight='bold', labelpad=10)
ax.set_ylabel('Budget per Country (USD)', fontsize=16, fontweight='bold', labelpad=10)
ax.set_title('Per Country Budget Allocation by Cluster', fontsize=18, fontweight='bold', pad=20)

# Set y-axis limit with padding
ax.set_ylim(0, y_limit)

# Add grid
ax.grid(True, alpha=0.2, axis='y')
ax.set_axisbelow(True)

# Rotate x-axis labels
plt.setp(ax.xaxis.get_majorticklabels(), rotation=20, ha='right', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(
    '../figures/clustering/per_country_budget.png',
    dpi=300,
    bbox_inches='tight',
    facecolor='white'
)
plt.show()

print("✅ Per country budget chart saved to: ../figures/clustering/per_country_budget.png")



print("\n✅ All budget visualizations complete!")
print("\n📁 Files saved:")
print("   • ../figures/clustering/budget_allocation_final.png")
print("   • ../figures/clustering/per_country_budget.png")

In [ ]:
# ============================================================================
# 12.1 SANKEY DIAGRAM - BUDGET FLOW 
# ============================================================================

print("📊 GENERATING SANKEY DIAGRAM")
print("=" * 80)

try:
    import plotly.graph_objects as go
    import os

    # =========================================================================
    # STEP 0: SETUP
    # =========================================================================

    os.makedirs("../figures/clustering", exist_ok=True)

    # =========================================================================
    # STEP 1: VALIDATE BUDGET CONFIGURATION
    # =========================================================================

    total_budget_share = sum(
        CLUSTER_BUDGET[cluster_id]
        for cluster_id in CLUSTER_ORDER
    )

    assert abs(total_budget_share - 1.0) < 1e-9, (
        f"Cluster budget shares must sum to 1.0, "
        f"but sum to {total_budget_share:.6f}."
    )

    # Calculate actual cluster budgets in millions of USD
    TOTAL_BUDGET_M = TOTAL_BUDGET / 1_000_000
    cluster_budget_m = {
        cluster_id: (CLUSTER_BUDGET[cluster_id] * TOTAL_BUDGET_M)
        for cluster_id in CLUSTER_ORDER
    }

    # =========================================================================
    # STEP 2: DEFINE POLICY-AREA ALLOCATION
    # =========================================================================

    policy_allocations = {

        # Cluster 0: Intermediate Development = $250M
        0: {
            "Healthcare": 70,
            "Education": 60,
            "Infrastructure": 60,
            "Economic Development": 60
        },

        # Cluster 1: High Development = $20M
        1: {
            "Innovation": 8,
            "Sustainability": 6,
            "Governance": 6
        },

        # Cluster 2: High Development Vulnerability = $450M
        2: {
            "Healthcare": 160,
            "Education": 110,
            "Infrastructure": 100,
            "Economic Development": 80
        },

        # Cluster 3: Trade-Intensive Development = $80M
        3: {
            "Innovation": 25,
            "Governance": 25,
            "Sustainability": 30
        },

        # Cluster 4: Higher-Inflation Development = $200M
        4: {
            "Healthcare": 60,
            "Stabilization": 60,
            "Economic Development": 40,
            "Education": 40
        }
    }

    # =========================================================================
    # STEP 3: VALIDATE POLICY ALLOCATIONS
    # =========================================================================

    print("\n🔍 VALIDATING SANKEY FLOWS")
    print("-" * 80)

    for cluster_id in CLUSTER_ORDER:
        expected = cluster_budget_m[cluster_id]
        allocated = sum(policy_allocations[cluster_id].values())
        assert abs(expected - allocated) < 1e-9, (
            f"Cluster {cluster_id} does not balance. "
            f"Expected ${expected:.1f}M, "
            f"but policy allocations total ${allocated:.1f}M."
        )
        print(f"Cluster {cluster_id}: ${expected:.1f}M → ${allocated:.1f}M ✅")

    total_policy_allocation = sum(
        sum(policy_allocations[cluster_id].values())
        for cluster_id in CLUSTER_ORDER
    )

    assert abs(total_policy_allocation - TOTAL_BUDGET_M) < 1e-9, (
        f"Total policy allocation does not equal total budget. "
        f"Expected ${TOTAL_BUDGET_M:.1f}M, "
        f"got ${total_policy_allocation:.1f}M."
    )

    print("-" * 80)
    print(f"Total policy allocation: ${total_policy_allocation:.1f}M = Total budget ${TOTAL_BUDGET_M:.1f}M ✅")

    # =========================================================================
    # STEP 4: DEFINE LABELS
    # =========================================================================

    labels = ["$1 Billion Budget"]

    cluster_label_indices = {}
    for cluster_id in CLUSTER_ORDER:
        cluster_label_indices[cluster_id] = len(labels)
        labels.append(CLUSTER_SHORT_NAMES[cluster_id])

    policy_areas = [
        "Healthcare",
        "Education",
        "Infrastructure",
        "Economic Development",
        "Stabilization",
        "Innovation",
        "Sustainability",
        "Governance"
    ]

    policy_label_indices = {}
    for policy in policy_areas:
        policy_label_indices[policy] = len(labels)
        labels.append(policy)

    # =========================================================================
    # STEP 5: BUILD SANKEY FLOWS
    # =========================================================================

    source = []
    target = []
    values = []

    # Budget → Clusters
    budget_node = 0
    for cluster_id in CLUSTER_ORDER:
        source.append(budget_node)
        target.append(cluster_label_indices[cluster_id])
        values.append(cluster_budget_m[cluster_id])

    # Clusters → Policy Areas
    for cluster_id in CLUSTER_ORDER:
        cluster_node = cluster_label_indices[cluster_id]
        for policy, amount in policy_allocations[cluster_id].items():
            source.append(cluster_node)
            target.append(policy_label_indices[policy])
            values.append(amount)

    # =========================================================================
    # STEP 6: DEFINE COLORS
    # =========================================================================

    cluster_colors = [CLUSTER_ID_COLORS[cluster_id] for cluster_id in CLUSTER_ORDER]

    policy_colors = [
        "#FF6B6B",  # Healthcare
        "#FFA94D",  # Education
        "#FFD93D",  # Infrastructure
        "#6BCB77",  # Economic Development
        "#4D96FF",  # Stabilization
        "#A66CFF",  # Innovation
        "#FF6B9D",  # Sustainability
        "#66D9A8"   # Governance
    ]

    node_colors = (
        ["#808080"]          # Budget
        + cluster_colors
        + policy_colors
    )

    # =========================================================================
    # STEP 7: CREATE NODE POSITIONS
    # =========================================================================

    n_clusters = len(CLUSTER_ORDER)
    n_policies = len(policy_areas)

    # Budget
    node_x = [0.01]
    node_y = [0.50]

    # Clusters
    cluster_y_positions = [0.08, 0.28, 0.48, 0.68, 0.88]
    for y in cluster_y_positions[:n_clusters]:
        node_x.append(0.30)
        node_y.append(y)

    # Policy areas
    policy_y_positions = [0.05, 0.18, 0.31, 0.44, 0.57, 0.70, 0.83, 0.96]
    for y in policy_y_positions[:n_policies]:
        node_x.append(0.70)
        node_y.append(y)

    # =========================================================================
    # STEP 8: CREATE SANKEY DIAGRAM 
    # =========================================================================

    fig = go.Figure(
        data=[
            go.Sankey(
                arrangement="fixed",
                node=dict(
                    pad=25,                      
                    thickness=32,               
                    line=dict(color="white", width=2), 
                    label=labels,
                    color=node_colors,
                    x=node_x,
                    y=node_y,
                    hovertemplate=(
                        "%{label}<br>"
                        "Flow: %{value:.1f}M USD"
                        "<extra></extra>"
                    )
                ),
                link=dict(
                    source=source,
                    target=target,
                    value=values,
                    color=["rgba(128,128,128,0.25)" for _ in values],
                    hovertemplate=(
                        "%{source.label} → %{target.label}<br>"
                        "Allocation: %{value:.1f}M USD"
                        "<extra></extra>"
                    )
                )
            )
        ]
    )

    # =========================================================================
    # STEP 9: UPDATE LAYOUT 
    # =========================================================================

    fig.update_layout(
        title=dict(
            text="Budget Allocation Flow: $1 Billion from Budget to Development Priorities",
            font=dict(
                size=28,             
                family="Arial Black",
                color="#1a1a1a"
            ),
            x=0.5,
            xanchor="center",
            y=0.98
        ),
        font=dict(
            size=16,                 
            family="Arial"
        ),
        width=1600,                  
        height=950,                   
        margin=dict(
            l=80,                    
            r=80,
            t=130,                   
            b=80
        ),
        paper_bgcolor="white",
        plot_bgcolor="white"
    )

    # =========================================================================
    # STEP 10: SAVE PNG
    # =========================================================================

    png_path = "../figures/clustering/sankey_budget.png"

    try:
        fig.write_image(png_path, scale=3, width=1600, height=950)  
        print(f"\n✅ Sankey diagram saved as PNG: {png_path}")
    except Exception as e:
        print(f"\n⚠️ Could not save Sankey as PNG: {e}")
        print("   Install/update kaleido with: pip install --upgrade kaleido")

    # =========================================================================
    # STEP 11: SAVE HTML
    # =========================================================================

    html_path = "../figures/clustering/sankey_budget.html"
    fig.write_html(html_path)
    print(f"✅ Sankey diagram saved as HTML: {html_path}")

    # =========================================================================
    # STEP 12: DISPLAY
    # =========================================================================

    fig.show()

    # =========================================================================
    # STEP 13: PRINT BUDGET FLOW SUMMARY
    # =========================================================================

    print("\n" + "=" * 80)
    print("📊 SANKEY DIAGRAM SUMMARY")
    print("=" * 80)

    print("\nBudget Flow:")
    print("-" * 80)
    print(f"{'Cluster':<8} {'Name':<35} {'Budget %':>12} {'Amount ($M)':>15}")
    print("-" * 80)

    for cluster_id in CLUSTER_ORDER:
        name = CLUSTER_SHORT_NAMES[cluster_id]
        budget_pct = CLUSTER_BUDGET[cluster_id] * 100
        amount = cluster_budget_m[cluster_id]
        print(f"{cluster_id:<8} {name:<35} {budget_pct:>11.1f}% {amount:>14.1f}")

    print("-" * 80)
    print(f"{'TOTAL':<43} {100.0:>11.1f}% {TOTAL_BUDGET_M:>14.1f}")

    # =========================================================================
    # STEP 14: PRINT POLICY AREA ALLOCATIONS
    # =========================================================================

    policy_totals = {policy: 0 for policy in policy_areas}
    for cluster_id in CLUSTER_ORDER:
        for policy, amount in policy_allocations[cluster_id].items():
            policy_totals[policy] += amount

    print("\nPolicy Area Allocations:")
    print("-" * 80)
    print(f"{'Policy Area':<30} {'Amount ($M)':>15} {'Share':>12}")
    print("-" * 80)

    for policy in policy_areas:
        amount = policy_totals[policy]
        percentage = amount / TOTAL_BUDGET_M * 100
        print(f"{policy:<30} {amount:>14.1f} {percentage:>11.1f}%")

    print("-" * 80)
    print(f"{'TOTAL':<30} {total_policy_check:>14.1f} {100.0:>11.1f}%")

    print("=" * 80)
    print("\n" + "=" * 80)
    print("✅ SANKEY DIAGRAM GENERATION COMPLETE")
    print("=" * 80)

except Exception as e:
    print(f"⚠️ Sankey diagram failed: {e}")
    print("   Install required packages with: pip install plotly kaleido")

In [ ]:
# ============================================================================
# 13. GENERATE GEO PLOTS FOR EACH CLUSTER
# ============================================================================

print("🗺️ GENERATING CLUSTER MAPS")
print("=" * 80)


# ============================================================================
# STEP 0: Configuration and Validation
# ============================================================================

OUTPUT_DIR = "../figures/clustering"
PROCESSED_DIR = "../data/processed"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

TOTAL_BUDGET = 1_000_000_000

# ---------------------------------------------------------------------------
# Validate required configuration objects
# ---------------------------------------------------------------------------

required_objects = [
    "CLUSTER_ORDER",
    "CLUSTER_NAMES",
    "CLUSTER_SHORT_NAMES",
    "CLUSTER_ID_COLORS",
    "CLUSTER_PRIORITY",
    "CLUSTER_BUDGET",
    "df_with_clusters_final"
]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals()
]

if missing_objects:
    raise NameError(
        "The following required objects are missing: "
        + ", ".join(missing_objects)
    )

# ---------------------------------------------------------------------------
# Validate cluster configuration
# ---------------------------------------------------------------------------

expected_cluster_ids = list(CLUSTER_ORDER)

assert set(expected_cluster_ids) == set(CLUSTER_NAMES.keys()), \
    "CLUSTER_NAMES does not contain exactly the CLUSTER_ORDER IDs."

assert set(expected_cluster_ids) == set(CLUSTER_SHORT_NAMES.keys()), \
    "CLUSTER_SHORT_NAMES does not contain exactly the CLUSTER_ORDER IDs."

assert set(expected_cluster_ids) == set(CLUSTER_ID_COLORS.keys()), \
    "CLUSTER_ID_COLORS does not contain exactly the CLUSTER_ORDER IDs."

assert set(expected_cluster_ids) == set(CLUSTER_PRIORITY.keys()), \
    "CLUSTER_PRIORITY does not contain exactly the CLUSTER_ORDER IDs."

assert set(expected_cluster_ids) == set(CLUSTER_BUDGET.keys()), \
    "CLUSTER_BUDGET does not contain exactly the CLUSTER_ORDER IDs."


# ============================================================================
# STEP 1: Validate Cluster Assignments
# ============================================================================

print("\n🔍 VALIDATING CLUSTER ASSIGNMENTS")
print("-" * 70)

map_data = df_with_clusters_final.copy()

# Required columns
required_columns = [
    "country",
    "cluster",
    "child_mort",
    "income",
    "life_expec",
    "total_fer"
]

missing_columns = [
    col for col in required_columns
    if col not in map_data.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns in df_with_clusters_final: "
        f"{missing_columns}"
    )

# Check country count
total_countries = len(map_data)

if total_countries != 167:
    print(
        f"⚠️ WARNING: Expected 167 countries, "
        f"but found {total_countries}."
    )
else:
    print("✅ Total countries: 167")

# Check duplicate countries
duplicate_countries = map_data["country"].duplicated().sum()

if duplicate_countries > 0:
    raise ValueError(
        f"Found {duplicate_countries} duplicate country records."
    )

print("✅ Country names are unique")

# Check cluster IDs
actual_cluster_ids = sorted(map_data["cluster"].dropna().unique().tolist())
expected_sorted = sorted(expected_cluster_ids)

if actual_cluster_ids != expected_sorted:
    raise ValueError(
        f"Cluster IDs do not match configuration.\n"
        f"Expected: {expected_sorted}\n"
        f"Found:    {actual_cluster_ids}"
    )

print(f"✅ Cluster IDs validated: {expected_sorted}")

# Validate counts
expected_counts = {}

print("\nCluster counts:")
for cluster_id in CLUSTER_ORDER:

    count = int(
        (map_data["cluster"] == cluster_id).sum()
    )

    expected_counts[cluster_id] = count

    print(
        f"  Cluster {cluster_id}: "
        f"{CLUSTER_SHORT_NAMES[cluster_id]:<30} "
        f"{count:>3} countries"
    )

count_sum = sum(expected_counts.values())

if count_sum != total_countries:
    raise ValueError(
        f"Cluster counts ({count_sum}) do not equal "
        f"total countries ({total_countries})."
    )

print("-" * 70)
print(f"Total assigned: {count_sum}")
print("✅ All countries assigned to exactly one cluster")


# ============================================================================
# STEP 2: Validate Budget Configuration
# ============================================================================

print("\n💰 VALIDATING BUDGET CONFIGURATION")
print("-" * 70)

budget_sum = sum(
    CLUSTER_BUDGET[cluster_id]
    for cluster_id in CLUSTER_ORDER
)

print(f"Raw budget shares: {budget_sum:.4f}")

if abs(budget_sum - 1.0) > 1e-9:
    raise ValueError(
        f"CLUSTER_BUDGET must sum to 1.0, "
        f"but sums to {budget_sum:.4f}"
    )

print("✅ Budget shares sum to 100.0%")

for cluster_id in CLUSTER_ORDER:

    budget_fraction = CLUSTER_BUDGET[cluster_id]

    # IMPORTANT:
    # CLUSTER_BUDGET stores fractions:
    # 0.25 = 25%
    budget_pct = budget_fraction * 100

    budget_amount = budget_fraction * TOTAL_BUDGET
    budget_m = budget_amount / 1_000_000

    print(
        f"  Cluster {cluster_id}: "
        f"{budget_pct:>5.1f}% "
        f"→ ${budget_m:,.1f}M"
    )

print("-" * 70)
print(
    f"Total budget: "
    f"${TOTAL_BUDGET / 1_000_000:,.1f}M"
)
print("✅ Budget configuration validated")


# ============================================================================
# STEP 3: Create Map Metadata
# ============================================================================

print("\n🎨 PREPARING MAP DATA")
print("-" * 70)

# ---------------------------------------------------------------------------
# Cluster metadata
# ---------------------------------------------------------------------------

map_data["Cluster_Name"] = map_data["cluster"].map(CLUSTER_NAMES)
map_data["Cluster_Short"] = map_data["cluster"].map(CLUSTER_SHORT_NAMES)
map_data["Cluster_Color"] = map_data["cluster"].map(CLUSTER_ID_COLORS)
map_data["Priority"] = map_data["cluster"].map(CLUSTER_PRIORITY)

# ---------------------------------------------------------------------------
# IMPORTANT BUDGET CORRECTION
#
# CLUSTER_BUDGET is stored as:
#     0.25 = 25%
#     0.02 = 2%
#     0.45 = 45%
#
# Therefore:
#     Budget_% = CLUSTER_BUDGET * 100
# ---------------------------------------------------------------------------

map_data["Budget_%"] = (
    map_data["cluster"]
    .map(CLUSTER_BUDGET)
    * 100
)

map_data["Budget_Amount_USD"] = (
    map_data["cluster"]
    .map(CLUSTER_BUDGET)
    * TOTAL_BUDGET
)

map_data["Budget_Amount_M_USD"] = (
    map_data["Budget_Amount_USD"] / 1_000_000
)

# ---------------------------------------------------------------------------
# Country share
# ---------------------------------------------------------------------------

map_data["Cluster_Country_Count"] = (
    map_data["cluster"]
    .map(expected_counts)
)

map_data["Cluster_Country_Share_%"] = (
    map_data["Cluster_Country_Count"]
    / total_countries
    * 100
)

print("✅ Map metadata prepared")

print("\nBudget metadata check:")
for cluster_id in CLUSTER_ORDER:

    row = map_data[
        map_data["cluster"] == cluster_id
    ].iloc[0]

    print(
        f"  Cluster {cluster_id}: "
        f"{row['Budget_%']:.1f}% "
        f"(${row['Budget_Amount_M_USD']:.1f}M)"
    )


# ============================================================================
# STEP 4: Validate Country → Cluster → Budget Mapping
# ============================================================================

print("\n🔍 VALIDATING COUNTRY → CLUSTER → BUDGET MAPPING")
print("-" * 70)

for cluster_id in CLUSTER_ORDER:

    cluster_rows = map_data[
        map_data["cluster"] == cluster_id
    ]

    count = len(cluster_rows)

    budget_pct = (
        CLUSTER_BUDGET[cluster_id] * 100
    )

    budget_amount = (
        CLUSTER_BUDGET[cluster_id]
        * TOTAL_BUDGET
    )

    # Verify every country has the correct metadata
    assert (
        cluster_rows["Budget_%"]
        == budget_pct
    ).all()

    assert (
        cluster_rows["Budget_Amount_USD"]
        == budget_amount
    ).all()

    print(
        f"Cluster {cluster_id}: "
        f"{count:>3} countries → "
        f"{budget_pct:>5.1f}% → "
        f"${budget_amount / 1_000_000:>6.1f}M ✅"
    )

print("-" * 70)
print("✅ Country-cluster-budget mapping validated")


# ============================================================================
# STEP 5: Save Map Data
# ============================================================================

map_export_columns = [
    "country",
    "cluster",
    "Cluster_Name",
    "Cluster_Short",
    "Cluster_Color",
    "Priority",
    "Budget_%",
    "Budget_Amount_USD",
    "Budget_Amount_M_USD",
    "Cluster_Country_Count",
    "Cluster_Country_Share_%",
    "child_mort",
    "income",
    "life_expec",
    "total_fer"
]

map_export = map_data[
    map_export_columns
].sort_values(
    ["cluster", "country"]
)

map_csv_path = (
    f"{PROCESSED_DIR}/country_cluster_map_data.csv"
)

map_export.to_csv(
    map_csv_path,
    index=False
)

print(
    f"\n✅ Map data saved to: "
    f"{map_csv_path}"
)


# ============================================================================
# STEP 6: Define Color Mapping
# ============================================================================

cluster_colors = {
    CLUSTER_SHORT_NAMES[cluster_id]:
    CLUSTER_ID_COLORS[cluster_id]
    for cluster_id in CLUSTER_ORDER
}

print("\n🎨 COLOR MAPPING")
print("-" * 70)

for cluster_id in CLUSTER_ORDER:

    print(
        f"  Cluster {cluster_id}: "
        f"{CLUSTER_SHORT_NAMES[cluster_id]:<30} "
        f"→ {CLUSTER_ID_COLORS[cluster_id]}"
    )

print("-" * 70)


# ============================================================================
# STEP 7: Generate Individual Cluster Maps
# ============================================================================

print("\n📍 GENERATING INDIVIDUAL CLUSTER MAPS")
print("=" * 70)

png_saved_count = 0
html_saved_count = 0

for cluster_id in CLUSTER_ORDER:

    cluster_name = CLUSTER_NAMES[cluster_id]
    cluster_short = CLUSTER_SHORT_NAMES[cluster_id]
    cluster_color = CLUSTER_ID_COLORS[cluster_id]

    mask = map_data["cluster"] == cluster_id

    n_countries = int(mask.sum())

    budget_pct = CLUSTER_BUDGET[cluster_id] * 100
    budget_amount_m = (
        CLUSTER_BUDGET[cluster_id]
        * TOTAL_BUDGET
        / 1_000_000
    )

    priority = CLUSTER_PRIORITY[cluster_id]

    print(
        f"\n  Cluster {cluster_id}: "
        f"{cluster_short}"
    )

    print(
        f"    • Countries: {n_countries}"
    )

    print(
        f"    • Country Share: "
        f"{n_countries / total_countries * 100:.1f}%"
    )

    print(
        f"    • Priority: {priority}"
    )

    print(
        f"    • Budget: "
        f"{budget_pct:.1f}% "
        f"(${budget_amount_m:.1f}M)"
    )

    # ------------------------------------------------------------------------
    # Highlight selected cluster
    # ------------------------------------------------------------------------

    cluster_map_data = map_data.copy()

    cluster_map_data["Highlight"] = "Other Countries"

    cluster_map_data.loc[
        mask,
        "Highlight"
    ] = cluster_short

    color_map = {
        "Other Countries": "#E8E8E8",
        cluster_short: cluster_color
    }

    # ------------------------------------------------------------------------
    # Create map
    # ------------------------------------------------------------------------

    fig_cluster = px.choropleth(
        cluster_map_data,
        locations="country",
        locationmode="country names",
        color="Highlight",
        color_discrete_map=color_map,

        title=(
            f"{cluster_short} "
            f"({n_countries} countries)"
        ),

        hover_data={
            "country": True,
            "child_mort": ":.1f",
            "income": "$,.0f",
            "life_expec": ":.1f",
            "total_fer": ":.2f",
            "Cluster_Short": True,
            "Priority": True,
            "Budget_%": ":.1f",
            "Budget_Amount_M_USD": ":.1f",
            "Cluster_Country_Share_%": ":.1f",
        },

        labels={
            "country": "Country",
            "child_mort": "Child Mortality (per 1000)",
            "income": "Income per Capita ($)",
            "life_expec": "Life Expectancy (years)",
            "total_fer": "Total Fertility",
            "Cluster_Short": "Cluster",
            "Priority": "Priority",
            "Budget_%": "Budget Share (%)",
            "Budget_Amount_M_USD": "Cluster Budget ($M)",
            "Cluster_Country_Share_%": "Countries in Cluster (%)",
            "Highlight": "Map Highlight"
        },

        projection="natural earth"
    )

    # ------------------------------------------------------------------------
    # Layout
    # ------------------------------------------------------------------------

    fig_cluster.update_layout(

        title=dict(
            text=(
                f"{cluster_short} "
                f"({n_countries} countries)"
            ),
            font=dict(
                size=18,
                family="Arial Black",
                color="#1a1a1a"
            ),
            x=0.5,
            xanchor="center"
        ),

        geo=dict(
            showframe=False,
            showcoastlines=True,
            coastlinecolor="#333333",
            coastlinewidth=0.8,
            projection_type="natural earth",

            showland=True,
            landcolor="#F5F5F5",

            showocean=True,
            oceancolor="#E8F4F8",

            showcountries=True,
            countrycolor="white",
            countrywidth=0.5
        ),

        height=550,
        width=750,

        margin=dict(
            l=20,
            r=20,
            t=70,
            b=20
        ),

        paper_bgcolor="white",
        plot_bgcolor="white",

        showlegend=True,

        legend=dict(
            title=dict(
                text="Map Highlight",
                font=dict(size=11)
            ),
            font=dict(size=9),
            bgcolor="rgba(255,255,255,0.9)",
            bordercolor="#cccccc",
            borderwidth=1,
            x=0.02,
            y=0.98,
            xanchor="left",
            yanchor="top"
        )
    )

    # ------------------------------------------------------------------------
    # File name
    # ------------------------------------------------------------------------

    filename = (
        cluster_short
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("'", "")
        .replace("/", "_")
    )

    # ------------------------------------------------------------------------
    # Save HTML
    # ------------------------------------------------------------------------

    html_path = (
        f"{OUTPUT_DIR}/map_{filename}.html"
    )

    fig_cluster.write_html(
        html_path,
        include_plotlyjs="cdn"
    )

    print(
        f"    ✅ HTML saved: "
        f"map_{filename}.html"
    )

    html_saved_count += 1

    # ------------------------------------------------------------------------
    # Save PNG
    # ------------------------------------------------------------------------

    png_path = (
        f"{OUTPUT_DIR}/map_{filename}.png"
    )

    try:

        fig_cluster.write_image(
            png_path,
            scale=2
        )

        print(
            f"    ✅ PNG saved: "
            f"map_{filename}.png"
        )

        png_saved_count += 1

    except Exception as e:

        print(
            f"    ⚠️ PNG export failed: {e}"
        )

    # ------------------------------------------------------------------------
    # Display
    # ------------------------------------------------------------------------

    fig_cluster.show()


# ============================================================================
# STEP 8: Generate Combined Map
# ============================================================================

print("\n" + "=" * 70)
print("📍 GENERATING COMBINED CLUSTER MAP")
print("=" * 70)

fig_combined = px.choropleth(

    map_data,

    locations="country",

    locationmode="country names",

    color="Cluster_Short",

    color_discrete_map=cluster_colors,

    title="Country Clusters by Development Profile (k=5)",

    hover_data={
        "country": True,
        "child_mort": ":.1f",
        "income": "$,.0f",
        "life_expec": ":.1f",
        "total_fer": ":.2f",
        "Cluster_Short": True,
        "Priority": True,
        "Budget_%": ":.1f",
        "Budget_Amount_M_USD": ":.1f",
        "Cluster_Country_Share_%": ":.1f",
    },

    labels={
        "country": "Country",
        "child_mort": "Child Mortality (per 1000)",
        "income": "Income per Capita ($)",
        "life_expec": "Life Expectancy (years)",
        "total_fer": "Total Fertility",
        "Cluster_Short": "Development Profile",
        "Priority": "Priority",
        "Budget_%": "Budget Share (%)",
        "Budget_Amount_M_USD": "Cluster Budget ($M)",
        "Cluster_Country_Share_%": "Countries in Cluster (%)"
    },

    projection="natural earth"
)


# ============================================================================
# STEP 9: Combined Map Layout
# ============================================================================

fig_combined.update_layout(

    title=dict(
        text=(
            "Country Clusters by Development Profile "
            "(k=5)"
        ),

        font=dict(
            size=22,
            family="Arial Black",
            color="#1a1a1a"
        ),

        x=0.5,
        xanchor="center"
    ),

    legend=dict(
        title=dict(
            text="Development Profiles",
            font=dict(
                size=14,
                family="Arial Bold"
            )
        ),

        font=dict(
            size=11,
            family="Arial"
        ),

        bgcolor="rgba(255,255,255,0.9)",
        bordercolor="#cccccc",
        borderwidth=1,

        x=0.02,
        y=0.98,

        xanchor="left",
        yanchor="top"
    ),

    geo=dict(
        showframe=False,

        showcoastlines=True,
        coastlinecolor="#333333",
        coastlinewidth=0.8,

        projection_type="natural earth",

        showland=True,
        landcolor="#F5F5F5",

        showocean=True,
        oceancolor="#E8F4F8",

        showcountries=True,
        countrycolor="white",
        countrywidth=0.5
    ),

    width=1200,
    height=700,

    margin=dict(
        l=20,
        r=20,
        t=70,
        b=20
    ),

    paper_bgcolor="white",
    plot_bgcolor="white"
)


# ============================================================================
# STEP 10: Save Combined Map
# ============================================================================

combined_png_path = (
    f"{OUTPUT_DIR}/map_all_clusters.png"
)

combined_html_path = (
    f"{OUTPUT_DIR}/map_all_clusters.html"
)

try:

    fig_combined.write_image(
        combined_png_path,
        scale=2
    )

    print(
        f"✅ Combined map saved as PNG: "
        f"{combined_png_path}"
    )

except Exception as e:

    print(
        f"⚠️ Could not save combined map as PNG: {e}"
    )

fig_combined.write_html(
    combined_html_path,
    include_plotlyjs="cdn"
)

print(
    f"✅ Combined map saved as HTML: "
    f"{combined_html_path}"
)

fig_combined.show()


# ============================================================================
# STEP 11: Final Map Summary
# ============================================================================

print("\n" + "=" * 80)
print("📊 CLUSTER MAPS SUMMARY")
print("=" * 80)

print(
    f"\nTotal Countries: {total_countries}"
)

print(
    f"Number of Clusters: {len(CLUSTER_ORDER)}"
)

print("\nCluster Details:")
print("-" * 90)

print(
    f"{'ID':<5}"
    f"{'Cluster':<30}"
    f"{'Countries':>10}"
    f"{'Country %':>12}"
    f"{'Budget %':>12}"
    f"{'Budget ($M)':>15}"
    f"{'Priority':>15}"
)

print("-" * 90)

for cluster_id in CLUSTER_ORDER:

    short = CLUSTER_SHORT_NAMES[cluster_id]

    count = expected_counts[cluster_id]

    country_pct = (
        count
        / total_countries
        * 100
    )

    budget_pct = (
        CLUSTER_BUDGET[cluster_id]
        * 100
    )

    budget_m = (
        CLUSTER_BUDGET[cluster_id]
        * TOTAL_BUDGET
        / 1_000_000
    )

    priority = CLUSTER_PRIORITY[cluster_id]

    print(
        f"{cluster_id:<5}"
        f"{short:<30}"
        f"{count:>10}"
        f"{country_pct:>11.1f}%"
        f"{budget_pct:>11.1f}%"
        f"{budget_m:>14.1f}"
        f"{priority:>15}"
    )

print("-" * 90)

print(
    f"{'':<5}"
    f"{'TOTAL':<30}"
    f"{total_countries:>10}"
    f"{100.0:>11.1f}%"
    f"{100.0:>11.1f}%"
    f"{TOTAL_BUDGET / 1_000_000:>14.1f}"
    f"{'':>15}"
)

print("=" * 90)


# ============================================================================
# STEP 12: Final Validation
# ============================================================================

print("\n🔍 FINAL MAP VALIDATION")
print("-" * 70)

# Country count
assert len(map_data) == 167, \
    "Final map does not contain 167 countries."

# Unique countries
assert map_data["country"].is_unique, \
    "Duplicate countries detected."

# Cluster count
assert (
    map_data["cluster"].nunique()
    == len(CLUSTER_ORDER)
), "Incorrect number of clusters."

# Budget total
assert abs(
    map_data.groupby("cluster")["Budget_Amount_USD"]
    .first()
    .sum()
    - TOTAL_BUDGET
) < 1, \
    "Budget allocation does not equal total budget."

# Country allocation
assert (
    map_data.groupby("cluster")
    .size()
    .sum()
    == 167
), \
    "Country allocation does not equal 167."

print("✅ 167 unique countries")
print("✅ 5 clusters")
print("✅ All countries assigned")
print("✅ Budget shares = 100%")
print("✅ Budget amount = $1,000M")
print("✅ Map metadata validated")

print("\n📁 FILES GENERATED")
print("-" * 70)

print(
    f"   • Individual cluster maps: "
    f"{html_saved_count} HTML, "
    f"{png_saved_count} PNG"
)

print(
    "   • Combined map: HTML + PNG"
)

print(
    "   • Map data CSV: "
    "country_cluster_map_data.csv"
)

print("=" * 80)

if png_saved_count < len(CLUSTER_ORDER):

    print(
        "\n📌 Some PNG files could not be generated."
    )

    print(
        "   Install/update Kaleido with:"
    )

    print(
        "   pip install --upgrade kaleido"
    )

else:

    print(
        "\n✅ ALL CLUSTER MAPS GENERATED SUCCESSFULLY"
    )

print("=" * 80)

In [ ]:
# ============================================================================
# 14. COUNTRY LIST PER CLUSTER 
# ============================================================================

print("📋 COUNTRY LIST PER CLUSTER")
print("=" * 80)

import os
import pandas as pd

# ============================================================================
# STEP 0: SETUP
# ============================================================================

os.makedirs("../data/processed", exist_ok=True)

# ============================================================================
# STEP 1: VALIDATE CONFIGURATION
# ============================================================================

required_clusters = set(CLUSTER_ORDER)

assert required_clusters == set(CLUSTER_NAMES.keys()), (
    "CLUSTER_NAMES does not contain all cluster IDs."
)

assert required_clusters == set(CLUSTER_SHORT_NAMES.keys()), (
    "CLUSTER_SHORT_NAMES does not contain all cluster IDs."
)

assert required_clusters == set(CLUSTER_PRIORITY.keys()), (
    "CLUSTER_PRIORITY does not contain all cluster IDs."
)

assert required_clusters == set(CLUSTER_BUDGET.keys()), (
    "CLUSTER_BUDGET does not contain all cluster IDs."
)

assert "country" in df_with_clusters_final.columns, (
    "Dataframe does not contain a 'country' column."
)

assert "cluster" in df_with_clusters_final.columns, (
    "Dataframe does not contain a 'cluster' column."
)

# Validate budget shares
total_budget_share = sum(CLUSTER_BUDGET.values())

assert abs(total_budget_share - 1.0) < 1e-9, (
    f"Cluster budget shares must sum to 1.0, "
    f"but currently sum to {total_budget_share:.6f}."
)

print("✅ Cluster configuration validated.")
print(f"   Total budget allocation: {total_budget_share * 100:.1f}%")


# ============================================================================
# STEP 2: CREATE COUNTRY LIST BY CLUSTER
# ============================================================================

country_list = {}
total_countries = 0

for cluster_id in CLUSTER_ORDER:

    cluster_name = CLUSTER_NAMES[cluster_id]
    cluster_short = CLUSTER_SHORT_NAMES[cluster_id]

    # ------------------------------------------------------------
    # Get countries belonging to this cluster
    # ------------------------------------------------------------

    countries = (
        df_with_clusters_final.loc[
            df_with_clusters_final["cluster"] == cluster_id,
            "country"
        ]
        .dropna()
        .astype(str)
        .sort_values()
        .tolist()
    )

    # Store by CLUSTER_ID, not cluster name
    country_list[cluster_id] = countries

    total_countries += len(countries)

    # ------------------------------------------------------------
    # Metadata
    # ------------------------------------------------------------

    priority = CLUSTER_PRIORITY[cluster_id]

    # IMPORTANT:
    # CLUSTER_BUDGET is stored as a proportion.
    # Example: 0.25 = 25%.
    budget_share_pct = CLUSTER_BUDGET[cluster_id] * 100

    # ------------------------------------------------------------
    # Print cluster information
    # ------------------------------------------------------------

    print("\n" + "=" * 70)
    print(f"📌 Cluster {cluster_id}: {cluster_name}")
    print(f"   Short Name: {cluster_short}")
    print("=" * 70)

    print(f"   Countries: {len(countries)}")
    print(f"   Priority: {priority}")
    print(f"   Budget Share: {budget_share_pct:.1f}%")

    # Optional description
    if "CLUSTER_DESCRIPTIONS" in globals():

        if cluster_name in CLUSTER_DESCRIPTIONS:

            desc = CLUSTER_DESCRIPTIONS[cluster_name]

            if len(desc) > 200:
                desc = desc[:200] + "..."

            print(f"   Description: {desc}")

    # ------------------------------------------------------------
    # Country list
    # ------------------------------------------------------------

    print("\n   Country List:")
    print("   " + "-" * 60)

    # Five countries per row
    for i in range(0, len(countries), 5):

        chunk = countries[i:i + 5]

        start_number = i + 1
        end_number = min(i + 5, len(countries))

        prefix = f"   {start_number:2d}-{end_number:2d}: "

        print(prefix + ", ".join(chunk))


# ============================================================================
# STEP 3: VALIDATE TOTAL COUNTRY COUNT
# ============================================================================

actual_country_count = len(df_with_clusters_final)

assert total_countries == actual_country_count, (
    f"Country count mismatch: "
    f"{total_countries} countries were assigned to clusters, "
    f"but dataframe contains {actual_country_count} rows."
)

print("\n" + "=" * 70)
print("📊 SUMMARY")
print("=" * 70)

print(f"   Total Clusters: {len(CLUSTER_ORDER)}")
print(f"   Total Countries: {total_countries}")

print("\n   Cluster Sizes:")

for cluster_id in CLUSTER_ORDER:

    cluster_short = CLUSTER_SHORT_NAMES[cluster_id]

    count = len(country_list[cluster_id])

    percentage = (
        count / total_countries * 100
        if total_countries > 0
        else 0
    )

    print(
        f"   • {cluster_short}: "
        f"{count} countries ({percentage:.1f}%)"
    )


# ============================================================================
# STEP 4: CREATE COUNTRY-LEVEL DATAFRAME
# ============================================================================

country_data = []

for cluster_id in CLUSTER_ORDER:

    cluster_name = CLUSTER_NAMES[cluster_id]
    cluster_short = CLUSTER_SHORT_NAMES[cluster_id]

    priority = CLUSTER_PRIORITY[cluster_id]

    # Convert proportion → percentage
    budget_share_pct = CLUSTER_BUDGET[cluster_id] * 100

    countries = country_list[cluster_id]

    for country in countries:

        country_data.append({

            "Cluster_ID": cluster_id,

            "Cluster_Name": cluster_name,

            "Cluster_Short": cluster_short,

            "Country": country,

            "Priority": priority,

            "Budget_Share_%": budget_share_pct
        })


country_df = pd.DataFrame(country_data)

# Sort by cluster ID and country
country_df = (
    country_df
    .sort_values(["Cluster_ID", "Country"])
    .reset_index(drop=True)
)


# ============================================================================
# STEP 5: VALIDATE COUNTRY UNIQUENESS
# ============================================================================

country_occurrences = country_df["Country"].value_counts()

duplicate_countries = country_occurrences[
    country_occurrences > 1
]

assert len(duplicate_countries) == 0, (
    "Some countries appear in more than one cluster:\n"
    f"{duplicate_countries}"
)

assert len(country_df) == total_countries, (
    "Country dataframe size does not match total country count."
)

print("\n✅ Country assignment validation passed.")
print("   • Every country appears exactly once.")
print(f"   • Total countries: {total_countries}")


# ============================================================================
# STEP 6: CREATE CLUSTER SUMMARY DATAFRAME
# ============================================================================

summary_data = []

for cluster_id in CLUSTER_ORDER:

    cluster_name = CLUSTER_NAMES[cluster_id]
    cluster_short = CLUSTER_SHORT_NAMES[cluster_id]

    countries = country_list[cluster_id]

    count = len(countries)

    country_share_pct = (
        count / total_countries * 100
        if total_countries > 0
        else 0
    )

    budget_share_pct = CLUSTER_BUDGET[cluster_id] * 100

    summary_data.append({

        "Cluster_ID": cluster_id,

        "Cluster_Name": cluster_name,

        "Short_Name": cluster_short,

        "Number_of_Countries": count,

        "Country_Share_%": round(country_share_pct, 1),

        "Priority": CLUSTER_PRIORITY[cluster_id],

        "Budget_Share_%": round(budget_share_pct, 1),

        "Countries": ", ".join(countries)
    })


summary_df = pd.DataFrame(summary_data)


# ============================================================================
# STEP 7: CREATE CLUSTER PROFILE DATAFRAME
# ============================================================================

profile_data = []

for cluster_id in CLUSTER_ORDER:

    cluster_name = CLUSTER_NAMES[cluster_id]
    cluster_short = CLUSTER_SHORT_NAMES[cluster_id]

    cluster_data = df_with_clusters_final[
        df_with_clusters_final["cluster"] == cluster_id
    ]

    profile = {

        "Cluster_ID": cluster_id,

        "Cluster_Name": cluster_name,

        "Short_Name": cluster_short,

        "Countries": len(cluster_data),

        "Child_Mortality": cluster_data["child_mort"].mean(),

        "Income": cluster_data["income"].mean(),

        "GDP_per_Capita": cluster_data["gdpp"].mean(),

        "Life_Expectancy": cluster_data["life_expec"].mean(),

        "Inflation": cluster_data["inflation"].mean(),

        "Total_Fertility": cluster_data["total_fer"].mean(),

        "Exports": cluster_data["exports"].mean(),

        "Imports": cluster_data["imports"].mean(),

        "Health_Expenditure": cluster_data["health"].mean()
    }

    profile_data.append(profile)


profile_df = pd.DataFrame(profile_data)


# ============================================================================
# STEP 8: SAVE COUNTRY LIST TO CSV
# ============================================================================

country_csv_path = (
    "../data/processed/country_list_by_cluster.csv"
)

country_df.to_csv(
    country_csv_path,
    index=False
)

print(
    "\n✅ Country list saved to CSV: "
    f"{country_csv_path}"
)


# ============================================================================
# STEP 9: SAVE EXCEL WORKBOOK
# ============================================================================

excel_path = (
    "../data/processed/country_list_by_cluster.xlsx"
)

try:

    with pd.ExcelWriter(
        excel_path,
        engine="openpyxl"
    ) as writer:

        # ------------------------------------------------------------
        # Sheet 1: Countries by Cluster
        # ------------------------------------------------------------

        country_df.to_excel(
            writer,
            sheet_name="Countries by Cluster",
            index=False
        )

        # ------------------------------------------------------------
        # Sheet 2: Cluster Summary
        # ------------------------------------------------------------

        summary_df.to_excel(
            writer,
            sheet_name="Cluster Summary",
            index=False
        )

        # ------------------------------------------------------------
        # Sheet 3: Cluster Profiles
        # ------------------------------------------------------------

        profile_df.to_excel(
            writer,
            sheet_name="Cluster Profiles",
            index=False
        )

        # ------------------------------------------------------------
        # Excel formatting
        # ------------------------------------------------------------

        from openpyxl.styles import Font, Alignment
        from openpyxl.utils import get_column_letter

        for worksheet in writer.book.worksheets:

            # Header
            for cell in worksheet[1]:

                cell.font = Font(
                    bold=True
                )

                cell.alignment = Alignment(
                    horizontal="center",
                    vertical="center"
                )

            # Freeze header
            worksheet.freeze_panes = "A2"

            # Column widths
            for column_cells in worksheet.columns:

                max_length = 0

                column_letter = get_column_letter(
                    column_cells[0].column
                )

                for cell in column_cells:

                    try:

                        cell_length = len(
                            str(cell.value)
                        )

                        max_length = max(
                            max_length,
                            cell_length
                        )

                    except Exception:
                        pass

                adjusted_width = min(
                    max_length + 2,
                    45
                )

                worksheet.column_dimensions[
                    column_letter
                ].width = adjusted_width

    print(
        "✅ Country list saved to Excel: "
        f"{excel_path}"
    )

except Exception as e:

    print(
        f"⚠️ Could not save Excel file: {e}"
    )

    print(
        "   Install openpyxl with: "
        "pip install openpyxl"
    )


# ============================================================================
# STEP 10: COUNTRY COUNT SUMMARY
# ============================================================================

print("\n" + "=" * 90)
print("📊 COUNTRY COUNT SUMMARY")
print("=" * 90)

print(
    f"\n{'Cluster':<8}"
    f"{'Name':<43}"
    f"{'Countries':<12}"
    f"{'Country %':<12}"
    f"{'Priority':<12}"
    f"{'Budget %':<10}"
)

print("-" * 100)

for cluster_id in CLUSTER_ORDER:

    cluster_name = CLUSTER_NAMES[cluster_id]

    count = len(country_list[cluster_id])

    country_percentage = (
        count / total_countries * 100
    )

    priority = CLUSTER_PRIORITY[cluster_id]

    # IMPORTANT: convert proportion → percentage
    budget_percentage = (
        CLUSTER_BUDGET[cluster_id] * 100
    )

    print(
        f"{cluster_id:<8}"
        f"{cluster_name[:41]:<43}"
        f"{count:<12}"
        f"{country_percentage:>6.1f}%"
        f"{'':<6}"
        f"{priority:<12}"
        f"{budget_percentage:>6.1f}%"
    )

print("-" * 100)

print(
    f"{'':<8}"
    f"{'TOTAL':<43}"
    f"{total_countries:<12}"
    f"{'100.0%':<12}"
    f"{'':<12}"
    f"{total_budget_share * 100:>6.1f}%"
)

print("=" * 90)


# ============================================================================
# STEP 11: QUICK COUNTRY REFERENCE
# ============================================================================

print("\n" + "=" * 90)
print("📋 COUNTRY LISTS BY CLUSTER (QUICK REFERENCE)")
print("=" * 90)

for cluster_id in CLUSTER_ORDER:

    cluster_short = CLUSTER_SHORT_NAMES[cluster_id]

    countries = country_list[cluster_id]

    print(
        f"\n{cluster_short} "
        f"({len(countries)} countries):"
    )

    # Ten countries per row
    for i in range(0, len(countries), 10):

        chunk = countries[i:i + 10]

        print(
            "   " + ", ".join(chunk)
        )

    print()


# ============================================================================
# STEP 12: FINAL VALIDATION
# ============================================================================

# ------------------------------------------------------------
# Validate cluster coverage
# ------------------------------------------------------------

assigned_cluster_ids = set(
    df_with_clusters_final["cluster"].dropna().unique()
)

expected_cluster_ids = set(CLUSTER_ORDER)

assert assigned_cluster_ids == expected_cluster_ids, (
    "Mismatch between expected and actual cluster IDs."
)

# ------------------------------------------------------------
# Validate country count
# ------------------------------------------------------------

assert len(country_df) == len(df_with_clusters_final), (
    "Country dataframe does not contain all dataframe rows."
)

# ------------------------------------------------------------
# Validate unique countries
# ------------------------------------------------------------

assert country_df["Country"].nunique() == total_countries, (
    "Duplicate country assignments detected."
)

# ------------------------------------------------------------
# Validate budget
# ------------------------------------------------------------

assert abs(
    sum(CLUSTER_BUDGET.values()) - 1.0
) < 1e-9, (
    "Budget shares do not sum to 100%."
)


print("\n" + "=" * 90)
print("✅ COUNTRY LIST GENERATION COMPLETE")
print("=" * 90)

print("\n📁 Files saved:")

print(
    "   • ../data/processed/"
    "country_list_by_cluster.csv"
)

print(
    "   • ../data/processed/"
    "country_list_by_cluster.xlsx"
)

print("\n✅ Validation checks passed:")
print("   • All clusters represented")
print("   • All 167 countries assigned")
print("   • No duplicate country assignments")
print("   • Country count verified")
print("   • Budget shares verified")
print("=" * 90)

In [ ]:
# ============================================================================
# 15. FINAL SUMMARY DASHBOARD
# ============================================================================
#
# Purpose:
#   Provide a final, presentation-ready summary of the K=5 clustering
#   analysis, cluster profiles, statistical characteristics, and
#   illustrative budget allocation.
#
# IMPORTANT INTERPRETATION:
#   The five clusters are DATA-DRIVEN SOCIOECONOMIC PROFILES.
#   They should NOT be interpreted as a simple development ranking
#   or a chronological progression from "bad" to "good".
#
#   In particular:
#   - Cluster 3 is distinguished by high trade intensity.
#   - Cluster 4 is distinguished primarily by comparatively high inflation.
#   - Cluster 1 represents the strongest overall socioeconomic profile.
#   - Cluster 2 represents the strongest development vulnerability.
#
# ============================================================================

print("📊 FINAL SUMMARY DASHBOARD")
print("=" * 80)




# ============================================================================
# STEP 0: CONFIGURATION AND VALIDATION
# ============================================================================

# Ensure output directory exists
os.makedirs('../data/processed', exist_ok=True)

# ---------------------------------------------------------------------------
# Main analysis parameters
# ---------------------------------------------------------------------------

total_countries = len(df_with_clusters_final)

optimal_k = len(CLUSTER_ORDER)

total_budget = 1_000_000_000


# ---------------------------------------------------------------------------
# Silhouette score
# ---------------------------------------------------------------------------

# Use the previously calculated silhouette score if available.
# Otherwise, use the validated result from the clustering analysis.

try:
    silhouette_score = best_silhouette_score
except NameError:
    silhouette_score = 0.2226


# ---------------------------------------------------------------------------
# Budget configuration
# ---------------------------------------------------------------------------
#
# IMPORTANT:
# CLUSTER_BUDGET stores FRACTIONS, not percentages.
#
# Example:
#
#   0.25 = 25%
#   0.02 = 2%
#   0.45 = 45%
#   0.08 = 8%
#   0.20 = 20%
#
# Therefore:
#
#   budget_share   = CLUSTER_BUDGET[id]
#   budget_percent = budget_share * 100
#   budget_amount  = budget_share * total_budget
#
# Final illustrative allocation:
#
#   Cluster 0 = 25%
#   Cluster 1 = 2%
#   Cluster 2 = 45%
#   Cluster 3 = 8%
#   Cluster 4 = 20%
#
# ============================================================================

budget_total_share = sum(
    CLUSTER_BUDGET[i]
    for i in CLUSTER_ORDER
)


# Validate that budget allocations sum to 100%

if not np.isclose(budget_total_share, 1.0):

    raise ValueError(
        f"CLUSTER_BUDGET allocations must sum to 1.0 (100%), "
        f"but currently sum to {budget_total_share:.4f} "
        f"({budget_total_share * 100:.2f}%)."
    )


# ============================================================================
# STEP 1: CALCULATE SUMMARY STATISTICS
# ============================================================================

cluster_counts = {}
cluster_pcts = {}


for cluster_id in CLUSTER_ORDER:

    count = len(
        df_with_clusters_final[
            df_with_clusters_final['cluster'] == cluster_id
        ]
    )

    cluster_counts[cluster_id] = count

    cluster_pcts[cluster_id] = (
        count / total_countries * 100
        if total_countries > 0
        else 0
    )


# ---------------------------------------------------------------------------
# Validate country counts
# ---------------------------------------------------------------------------

if sum(cluster_counts.values()) != total_countries:

    raise ValueError(
        "Cluster counts do not sum to total number of countries."
    )


# ============================================================================
# STEP 2: PRINT DASHBOARD HEADER
# ============================================================================

print(
    f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║                   COUNTRY DEVELOPMENT CLUSTERING                             ║
║                           FINAL RESULTS                                      ║
╚══════════════════════════════════════════════════════════════════════════════╝

📊 OVERVIEW
────────────────────────────────────────────────────────────────────────────────
Total Countries:       {total_countries}
Selected Clusters:     {optimal_k}
Algorithm:             K-Means
Silhouette Score:      {silhouette_score:.4f}
Total Budget:          ${total_budget:,.0f}

🏷️ CLUSTER PROFILES
────────────────────────────────────────────────────────────────────────────────
"""
)


# ============================================================================
# STEP 3: PRINT CLUSTER PROFILES
# ============================================================================

for cluster_id in CLUSTER_ORDER:

    # ---------------------------------------------------------
    # Cluster naming
    # ---------------------------------------------------------

    cluster_name = CLUSTER_NAMES[cluster_id]

    cluster_short = CLUSTER_SHORT_NAMES[cluster_id]

    cluster_label = CLUSTER_LABELS[cluster_id]


    # ---------------------------------------------------------
    # Cluster size
    # ---------------------------------------------------------

    count = cluster_counts[cluster_id]

    country_pct = cluster_pcts[cluster_id]


    # ---------------------------------------------------------
    # Budget calculation
    # ---------------------------------------------------------

    budget_share = CLUSTER_BUDGET[cluster_id]

    budget_percent = budget_share * 100

    budget_amount = budget_share * total_budget


    # ---------------------------------------------------------
    # Policy priority
    # ---------------------------------------------------------

    priority = CLUSTER_PRIORITY[cluster_id]


    # ---------------------------------------------------------
    # Cluster color
    # ---------------------------------------------------------

    try:

        color = CLUSTER_ID_COLORS[cluster_id]

    except NameError:

        color = CLUSTER_COLORS[cluster_id]


    # ---------------------------------------------------------
    # Budget per country
    # ---------------------------------------------------------

    per_country = (
        budget_amount / count
        if count > 0
        else 0
    )


    # ---------------------------------------------------------
    # Get cluster data
    # ---------------------------------------------------------

    cluster_data = df_with_clusters_final[
        df_with_clusters_final['cluster'] == cluster_id
    ]


    # ---------------------------------------------------------
    # Key characteristics
    # ---------------------------------------------------------

    child_mort = cluster_data['child_mort'].mean()

    life_expec = cluster_data['life_expec'].mean()

    income = cluster_data['income'].mean()


    # ---------------------------------------------------------
    # Print profile
    # ---------------------------------------------------------

    print(f"  Cluster {cluster_id}: {cluster_name}")

    print(
        f"     Short Name : {cluster_short}"
    )

    print(
        f"     Label      : {cluster_label}"
    )

    print(
        f"     Countries  : {count} ({country_pct:.1f}%)"
    )

    print(
        f"     Priority   : {priority}"
    )

    print(
        f"     Budget     : {budget_percent:.1f}% "
        f"(${budget_amount:,.0f})"
    )

    print(
        f"     Per Country: ${per_country:,.0f}"
    )

    print(
        f"     Key Stats  : "
        f"child_mort={child_mort:.1f}, "
        f"life_expec={life_expec:.1f}, "
        f"income=${income:,.0f}"
    )

    print(
        f"     Color      : {color}"
    )

    print()


# ============================================================================
# STEP 4: MOST DISCRIMINATING FEATURES
# ============================================================================

print(
    """
📊 MOST DISCRIMINATING FEATURES (ANOVA F-STATISTIC)
────────────────────────────────────────────────────────────────────────────────
"""
)


# ANOVA results from the analysis

feature_importance = {

    'Child Mortality': 120.61,

    'Total Fertility': 105.42,

    'Life Expectancy': 86.92,

    'GDP per Capita': 36.44,

    'Imports': 32.09,

    'Income': 31.37,

    'Exports': 27.49,

    'Health Expenditure': 22.13,

    'Inflation': 8.84

}


for i, (feature, f_stat) in enumerate(
    feature_importance.items(),
    1
):

    bar = '█' * int(f_stat / 5)

    print(
        f"  {i}. {feature:<20} "
        f"F={f_stat:.2f}  {bar}"
    )


# ============================================================================
# STEP 5: CLUSTER PROFILE SUMMARY
# ============================================================================
#
# IMPORTANT:
# These clusters are NOT presented as a linear progression.
# They represent different socioeconomic profiles.
# ============================================================================

print(
    """
📊 CLUSTER PROFILE SUMMARY
────────────────────────────────────────────────────────────────────────────────
"""
)


profile_summary = [

    (
        "Intermediate Development Profile",
        "Intermediate socioeconomic outcomes",
        "High Priority"
    ),

    (
        "High Development Profile",
        "Strongest overall socioeconomic outcomes",
        "Very Low Priority"
    ),

    (
        "High Development Vulnerability",
        "Most adverse health, demographic, and economic outcomes",
        "Very High Priority"
    ),

    (
        "Trade-Intensive Development Profile",
        "Strong development outcomes with exceptionally high trade intensity",
        "Medium Priority"
    ),

    (
        "Higher-Inflation Development Profile",
        "Relatively strong development outcomes with comparatively high inflation",
        "High Priority"
    )
]


for name, profile, priority in profile_summary:

    print(
        f"  {name:<45}"
        f" → {profile:<70}"
        f" → {priority}"
    )


# ============================================================================
# STEP 6: KEY INSIGHTS
# ============================================================================

print(
    """
🎯 KEY INSIGHTS
────────────────────────────────────────────────────────────────────────────────

  1. Five distinct socioeconomic profiles were identified across
     167 countries.

  2. Child mortality, total fertility, life expectancy, GDP per capita,
     and income are among the strongest discriminating characteristics
     between the resulting clusters.

  3. 44 countries (26.3%) belong to the High Development Vulnerability
     profile and exhibit the most adverse combination of health,
     demographic, and economic indicators.

  4. 31 countries (18.6%) belong to the High Development Profile and
     exhibit the strongest overall socioeconomic outcomes in the dataset.

  5. The illustrative budget allocation follows a needs-based approach,
     assigning the largest share to the High Development Vulnerability
     profile and smaller shares to stronger overall profiles.

  6. Trade intensity and inflation distinguish additional socioeconomic
     profiles beyond a simple development ranking.

  7. All nine features show statistically significant differences
     across the resulting clusters (p < 0.001).

  8. Cluster labels are interpretive descriptions of observed profiles
     and should not be interpreted as official international classifications.

"""
)


# ============================================================================
# STEP 7: OUTPUT FILES
# ============================================================================

print(
    """
📁 OUTPUT FILES
────────────────────────────────────────────────────────────────────────────────

  ✅ Data Files:

     • ../data/processed/countries_with_clusters_k5.csv

     • ../data/processed/cluster_summary_named.csv

     • ../data/processed/cluster_profiles_named.csv

     • ../data/processed/budget_allocation_final.csv

     • ../data/processed/country_list_by_cluster.csv

     • ../data/processed/final_summary_dashboard.csv


  ✅ Figures (Clustering):

     • ../figures/clustering/elbow_curve.png

     • ../figures/clustering/cluster_profiles_heatmap.png

     • ../figures/clustering/cluster_profiles_radar.png

     • ../figures/clustering/cluster_profiles_barchart.png

     • ../figures/clustering/pca_clusters_professional.png

     • ../figures/clustering/anova_significance.png

     • ../figures/clustering/budget_allocation_final.png

     • ../figures/clustering/sankey_budget.png

     • ../figures/clustering/world_map_clusters.png

     • ../figures/clustering/map_all_clusters.png


  ✅ Figures (EDA):

     • ../figures/eda/key_variable_distributions.png

     • ../figures/eda/correlation_matrix.png

     • ../figures/eda/outlier_boxplots.png

     • ../figures/eda/skewness_improvement.png

     • ../figures/eda/transformation_improvement_chart.png


  ✅ Interactive Maps:

     • ../figures/clustering/world_map_clusters.html

     • ../figures/clustering/map_*.html

     • ../figures/clustering/sankey_budget.html

"""
)


# ============================================================================
# STEP 8: BUDGET ALLOCATION SUMMARY
# ============================================================================

print(
    """
💰 BUDGET ALLOCATION SUMMARY
────────────────────────────────────────────────────────────────────────────────
"""
)


print(
    f"{'Cluster':<8} "
    f"{'Name':<40} "
    f"{'Countries':<10} "
    f"{'Country %':<10} "
    f"{'Budget %':<10} "
    f"{'Amount ($M)':<15}"
)


print("-" * 105)


for cluster_id in CLUSTER_ORDER:

    name = CLUSTER_SHORT_NAMES[cluster_id]

    count = cluster_counts[cluster_id]

    country_pct = cluster_pcts[cluster_id]


    # ---------------------------------------------------------
    # Correct handling of CLUSTER_BUDGET
    # ---------------------------------------------------------

    budget_share = CLUSTER_BUDGET[cluster_id]

    budget_percent = budget_share * 100

    amount_m = (
        budget_share
        * total_budget
        / 1_000_000
    )


    print(
        f"{cluster_id:<8} "
        f"{name[:40]:<40} "
        f"{count:<10} "
        f"{country_pct:>6.1f}%   "
        f"{budget_percent:>6.1f}%   "
        f"${amount_m:>11,.1f}M"
    )


print("-" * 105)


print(
    f"{'':<8} "
    f"{'TOTAL':<40} "
    f"{total_countries:<10} "
    f"{100.0:>6.1f}%   "
    f"{budget_total_share * 100:>6.1f}%   "
    f"${total_budget / 1_000_000:>11,.1f}M"
)


print("=" * 105)


# ============================================================================
# STEP 9: BUDGET VALIDATION
# ============================================================================

print(
    """
🔍 VALIDATING FINAL BUDGET ALLOCATION
────────────────────────────────────────────────────────────────────────────────
"""
)


calculated_total_budget = 0


for cluster_id in CLUSTER_ORDER:

    budget_share = CLUSTER_BUDGET[cluster_id]

    budget_amount = (
        budget_share
        * total_budget
    )

    calculated_total_budget += budget_amount


    print(
        f"Cluster {cluster_id}: "
        f"${budget_amount / 1_000_000:.1f}M "
        f"({budget_share * 100:.1f}%)"
    )


print("-" * 80)


if np.isclose(
    calculated_total_budget,
    total_budget
):

    print(
        f"Total allocation: "
        f"${calculated_total_budget / 1_000_000:.1f}M "
        f"= ${total_budget / 1_000_000:.1f}M ✅"
    )


else:

    raise ValueError(
        f"Budget mismatch: "
        f"${calculated_total_budget:,.2f} != "
        f"${total_budget:,.2f}"
    )


# ============================================================================
# STEP 10: FINAL CONCLUSION
# ============================================================================

print(
    """
╔══════════════════════════════════════════════════════════════════════════════╗
║                                                                              ║
║                    ✅ ANALYSIS COMPLETE!                                    ║
║                                                                              ║
║    The clustering analysis identified five distinct country profiles.       ║
║                                                                              ║
║    The selected profiles represent different combinations of health,        ║
║    demographic, economic, trade, and macroeconomic characteristics.         ║
║                                                                              ║
║    A needs-based illustrative budget allocation of $1,000,000,000 has       ║
║    been proposed, prioritizing the most development-vulnerable profile.     ║
║                                                                              ║
║    📌 Next Steps:                                                           ║
║                                                                              ║
║    1. Review cluster profiles with stakeholders                             ║
║    2. Finalize budget allocation assumptions                                ║
║    3. Prepare presentation materials                                        ║
║    4. Generate final report                                                  ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
)


print("=" * 80)

print(
    "✅ ANALYSIS COMPLETE! READY FOR PRESENTATION."
)

print("=" * 80)


# ============================================================================
# STEP 11: SAVE DASHBOARD SUMMARY TO CSV
# ============================================================================

dashboard_summary = []


for cluster_id in CLUSTER_ORDER:

    # ---------------------------------------------------------
    # Cluster naming
    # ---------------------------------------------------------

    name = CLUSTER_NAMES[cluster_id]

    short = CLUSTER_SHORT_NAMES[cluster_id]

    label = CLUSTER_LABELS[cluster_id]


    # ---------------------------------------------------------
    # Cluster statistics
    # ---------------------------------------------------------

    count = cluster_counts[cluster_id]

    country_pct = cluster_pcts[cluster_id]

    priority = CLUSTER_PRIORITY[cluster_id]


    # ---------------------------------------------------------
    # Budget
    # ---------------------------------------------------------

    budget_share = CLUSTER_BUDGET[cluster_id]

    budget_percent = budget_share * 100

    budget_amount = (
        budget_share
        * total_budget
    )


    # ---------------------------------------------------------
    # Per-country budget
    # ---------------------------------------------------------

    per_country = (

        budget_amount / count

        if count > 0

        else 0

    )


    # ---------------------------------------------------------
    # Color
    # ---------------------------------------------------------

    try:

        color = CLUSTER_ID_COLORS[cluster_id]

    except NameError:

        color = CLUSTER_COLORS[cluster_id]


    # ---------------------------------------------------------
    # Description
    # ---------------------------------------------------------

    try:

        description = CLUSTER_DESCRIPTIONS[cluster_id]

    except NameError:

        description = ""


    # ---------------------------------------------------------
    # Save record
    # ---------------------------------------------------------

    dashboard_summary.append(

        {

            'Cluster_ID':
                cluster_id,

            'Cluster_Name':
                name,

            'Short_Name':
                short,

            'Label':
                label,

            'Countries':
                count,

            'Country_Share_%':
                round(country_pct, 2),

            'Priority':
                priority,

            'Budget_Share':
                round(budget_share, 4),

            'Budget_%':
                round(budget_percent, 2),

            'Total_Budget_USD':
                round(budget_amount, 2),

            'Total_Budget_M_USD':
                round(
                    budget_amount / 1_000_000,
                    2
                ),

            'Per_Country_USD':
                round(
                    per_country,
                    2
                ),

            'Color':
                color,

            'Description':
                description

        }

    )


# ============================================================================
# CREATE DATAFRAME
# ============================================================================

summary_df = pd.DataFrame(
    dashboard_summary
)


# ============================================================================
# SAVE CSV
# ============================================================================

summary_df.to_csv(

    '../data/processed/final_summary_dashboard.csv',

    index=False

)


print(
    "\n✅ Dashboard summary saved to: "
    "../data/processed/final_summary_dashboard.csv"
)


# ============================================================================
# STEP 12: DISPLAY FINAL DATAFRAME
# ============================================================================

print(
    "\n📋 FINAL DASHBOARD DATA"
)

print("-" * 80)


display(summary_df)


print(
    "\n" + "=" * 80
)

print(
    "✅ FINAL SUMMARY DASHBOARD COMPLETE"
)

print(
    "=" * 80
)

In [ ]:
# ============================================================================
# 16. FINAL SUMMARY DASHBOARD
# ============================================================================

print("📊 FINAL SUMMARY DASHBOARD")
print("=" * 80)



# ============================================================================
# STEP 0: Configuration and Validation
# ============================================================================

# Ensure output directory exists
os.makedirs('../data/processed', exist_ok=True)

# Main analysis parameters
total_countries = len(df_with_clusters_final)
optimal_k = len(CLUSTER_ORDER)
total_budget = 1_000_000_000

# Use the previously calculated silhouette score if available.
# Otherwise, use the validated result from the clustering analysis.
try:
    silhouette_score = best_silhouette_score
except NameError:
    silhouette_score = 0.2226

# ---------------------------------------------------------------------------
# IMPORTANT:
# CLUSTER_BUDGET stores FRACTIONS, not percentages.
#
# Example:
#   0.25 = 25%
#   0.02 = 2%
#   0.45 = 45%
#   0.08 = 8%
#   0.20 = 20%
#
# Therefore:
#   budget_share   = CLUSTER_BUDGET[id]
#   budget_percent = budget_share * 100
#   budget_amount  = budget_share * total_budget
# ---------------------------------------------------------------------------

budget_total_share = sum(CLUSTER_BUDGET[i] for i in CLUSTER_ORDER)

# Validate that budget allocations sum to 100%
if not np.isclose(budget_total_share, 1.0):
    raise ValueError(
        f"CLUSTER_BUDGET allocations must sum to 1.0 (100%), "
        f"but currently sum to {budget_total_share:.4f} "
        f"({budget_total_share * 100:.2f}%)."
    )

# ============================================================================
# STEP 1: Calculate Summary Statistics
# ============================================================================

cluster_counts = {}
cluster_pcts = {}

for cluster_id in CLUSTER_ORDER:

    count = len(
        df_with_clusters_final[
            df_with_clusters_final['cluster'] == cluster_id
        ]
    )

    cluster_counts[cluster_id] = count
    cluster_pcts[cluster_id] = (
        count / total_countries * 100
        if total_countries > 0
        else 0
    )

# Validate country counts
if sum(cluster_counts.values()) != total_countries:
    raise ValueError(
        "Cluster counts do not sum to total number of countries."
    )

# ============================================================================
# STEP 2: Print Dashboard Header
# ============================================================================

print(
    f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║                   COUNTRY DEVELOPMENT CLUSTERING                             ║
║                           FINAL RESULTS                                      ║
╚══════════════════════════════════════════════════════════════════════════════╝

📊 OVERVIEW
────────────────────────────────────────────────────────────────────────────────
Total Countries:       {total_countries}
Optimal Clusters:      {optimal_k}
Algorithm:             K-Means
Silhouette Score:      {silhouette_score:.4f}
Total Budget:          ${total_budget:,.0f}

🏷️ CLUSTER PROFILES
────────────────────────────────────────────────────────────────────────────────
"""
)

# ============================================================================
# STEP 3: Print Cluster Profiles
# ============================================================================

for cluster_id in CLUSTER_ORDER:

    cluster_name = CLUSTER_NAMES[cluster_id]
    cluster_short = CLUSTER_SHORT_NAMES[cluster_id]

    count = cluster_counts[cluster_id]
    country_pct = cluster_pcts[cluster_id]

    # ---------------------------------------------------------
    # CORRECT BUDGET CALCULATION
    # ---------------------------------------------------------
    budget_share = CLUSTER_BUDGET[cluster_id]
    budget_percent = budget_share * 100
    budget_amount = budget_share * total_budget

    priority = CLUSTER_PRIORITY[cluster_id]
    color = CLUSTER_ID_COLORS[cluster_id]

    # Budget per country
    per_country = (
        budget_amount / count
        if count > 0
        else 0
    )

    # Get cluster data
    cluster_data = df_with_clusters_final[
        df_with_clusters_final['cluster'] == cluster_id
    ]

    # Key characteristics
    child_mort = cluster_data['child_mort'].mean()
    life_expec = cluster_data['life_expec'].mean()
    income = cluster_data['income'].mean()

    print(f"  {cluster_id}. {cluster_name}")
    print(f"     Short Name: {cluster_short}")
    print(f"     Countries: {count} ({country_pct:.1f}%)")
    print(f"     Priority: {priority}")
    print(
        f"     Budget: {budget_percent:.1f}% "
        f"(${budget_amount:,.0f})"
    )
    print(f"     Per Country: ${per_country:,.0f}")
    print(
        f"     Key Stats: "
        f"child_mort={child_mort:.1f}, "
        f"life_expec={life_expec:.1f}, "
        f"income=${income:,.0f}"
    )
    print(f"     Color: {color}")
    print()

# ============================================================================
# STEP 4: Feature Importance
# ============================================================================

print(
    """
📊 MOST DISCRIMINATING FEATURES (ANOVA F-STATISTIC)
────────────────────────────────────────────────────────────────────────────────
"""
)

# ANOVA results from the analysis
feature_importance = {
    'Child Mortality': 120.61,
    'Total Fertility': 105.42,
    'Life Expectancy': 86.92,
    'GDP per Capita': 36.44,
    'Imports': 32.09,
    'Income': 31.37,
    'Exports': 27.49,
    'Health Expenditure': 22.13,
    'Inflation': 8.84
}

for i, (feature, f_stat) in enumerate(
    feature_importance.items(),
    1
):

    bar = '█' * int(f_stat / 5)

    print(
        f"  {i}. {feature:<20} "
        f"F={f_stat:.2f}  {bar}"
    )

# ============================================================================
# STEP 5: Development Progression
# ============================================================================

print(
    """
📈 DEVELOPMENT PROGRESSION
────────────────────────────────────────────────────────────────────────────────
"""
)

progression = [
    (
        "Severe Vulnerability",
        "⬇️ Most Vulnerable",
        "Very High Priority"
    ),
    (
        "Moderate Development",
        "⬇️ Significant Improvement Needed",
        "High Priority"
    ),
    (
        "Macroeconomic Vulnerability",
        "⬇️ Good Development + High Inflation",
        "High Priority"
    ),
    (
        "Trade-Integrated",
        "⬇️ Strong Development + Trade Intensity",
        "Medium Priority"
    ),
    (
        "Advanced Development",
        "⬇️ Best Overall Outcomes",
        "Very Low Priority"
    )
]

for name, status, priority in progression:

    print(
        f"  {name:<30} "
        f"→ {status:<40} "
        f"→ {priority}"
    )

# ============================================================================
# STEP 6: Key Insights
# ============================================================================

print(
    """
🎯 KEY INSIGHTS
────────────────────────────────────────────────────────────────────────────────
  1. Five distinct socioeconomic profiles identified across 167 countries
  2. Child mortality, fertility, and life expectancy are the strongest
     discriminating factors between clusters
  3. 44 countries (26.3%) face severe development vulnerability requiring
     urgent intervention
  4. 31 countries (18.6%) represent advanced human development economies
     with the strongest overall profile
  5. Budget allocation follows a needs-based approach prioritizing the
     most vulnerable clusters
  6. Trade integration and macroeconomic vulnerability represent unique
     dimensions beyond simple income-based classifications
  7. All nine features show statistically significant differences across
     clusters (p < 0.001)
"""
)

# ============================================================================
# STEP 7: Output Files
# ============================================================================

print(
    """
📁 OUTPUT FILES
────────────────────────────────────────────────────────────────────────────────
  ✅ Data Files:
     • ../data/processed/countries_with_clusters_k5.csv
     • ../data/processed/cluster_summary_named.csv
     • ../data/processed/cluster_profiles_named.csv
     • ../data/processed/budget_allocation_final.csv
     • ../data/processed/country_list_by_cluster.csv

  ✅ Figures (Clustering):
     • ../figures/clustering/elbow_curve.png
     • ../figures/clustering/cluster_profiles_heatmap.png
     • ../figures/clustering/cluster_profiles_radar.png
     • ../figures/clustering/cluster_profiles_barchart.png
     • ../figures/clustering/pca_clusters_professional.png
     • ../figures/clustering/anova_significance.png
     • ../figures/clustering/budget_allocation_final.png
     • ../figures/clustering/sankey_budget.png
     • ../figures/clustering/world_map_clusters.png
     • ../figures/clustering/map_all_clusters.png

  ✅ Figures (EDA):
     • ../figures/eda/key_variable_distributions.png
     • ../figures/eda/correlation_matrix.png
     • ../figures/eda/outlier_boxplots.png
     • ../figures/eda/skewness_improvement.png
     • ../figures/eda/transformation_improvement_chart.png

  ✅ Interactive Maps:
     • ../figures/clustering/world_map_clusters.html
     • ../figures/clustering/map_*.html
     • ../figures/clustering/sankey_budget.html
"""
)

# ============================================================================
# STEP 8: Budget Allocation Summary
# ============================================================================

print(
    """
💰 BUDGET ALLOCATION SUMMARY
────────────────────────────────────────────────────────────────────────────────
"""
)

print(
    f"{'Cluster':<8} "
    f"{'Name':<35} "
    f"{'Countries':<10} "
    f"{'Country %':<10} "
    f"{'Budget %':<10} "
    f"{'Amount ($M)':<15}"
)

print("-" * 100)

for cluster_id in CLUSTER_ORDER:

    name = CLUSTER_SHORT_NAMES[cluster_id]

    count = cluster_counts[cluster_id]
    country_pct = cluster_pcts[cluster_id]

    # CORRECT handling of CLUSTER_BUDGET
    budget_share = CLUSTER_BUDGET[cluster_id]
    budget_percent = budget_share * 100
    amount_m = budget_share * total_budget / 1_000_000

    print(
        f"{cluster_id:<8} "
        f"{name[:35]:<35} "
        f"{count:<10} "
        f"{country_pct:>6.1f}%   "
        f"{budget_percent:>6.1f}%   "
        f"${amount_m:>11,.1f}M"
    )

print("-" * 100)

print(
    f"{'':<8} "
    f"{'TOTAL':<35} "
    f"{total_countries:<10} "
    f"{100.0:>6.1f}%   "
    f"{budget_total_share * 100:>6.1f}%   "
    f"${total_budget / 1_000_000:>11,.1f}M"
)

print("=" * 100)

# ============================================================================
# STEP 9: Budget Validation
# ============================================================================

print(
    """
🔍 VALIDATING FINAL BUDGET ALLOCATION
────────────────────────────────────────────────────────────────────────────────
"""
)

calculated_total_budget = 0

for cluster_id in CLUSTER_ORDER:

    budget_share = CLUSTER_BUDGET[cluster_id]
    budget_amount = budget_share * total_budget

    calculated_total_budget += budget_amount

    print(
        f"Cluster {cluster_id}: "
        f"${budget_amount / 1_000_000:.1f}M "
        f"({budget_share * 100:.1f}%)"
    )

print("-" * 80)

if np.isclose(calculated_total_budget, total_budget):

    print(
        f"Total allocation: "
        f"${calculated_total_budget / 1_000_000:.1f}M "
        f"= ${total_budget / 1_000_000:.1f}M ✅"
    )

else:

    raise ValueError(
        f"Budget mismatch: "
        f"${calculated_total_budget:,.2f} != "
        f"${total_budget:,.2f}"
    )

# ============================================================================
# STEP 10: Final Conclusion
# ============================================================================

print(
    """
╔══════════════════════════════════════════════════════════════════════════════╗
║                                                                              ║
║                    ✅ ANALYSIS COMPLETE!                                    ║
║                                                                              ║
║    The clustering analysis identified 5 distinct country profiles.           ║
║                                                                              ║
║    A needs-based budget allocation of $1,000,000,000 has been proposed,      ║
║    prioritizing the most vulnerable clusters.                               ║
║                                                                              ║
║    📌 Next Steps:                                                            ║
║    1. Review cluster profiles with stakeholders                             ║
║    2. Finalize budget allocation percentages                                ║
║    3. Prepare presentation materials                                        ║
║    4. Generate final report                                                  ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
)

print("=" * 80)
print("✅ ANALYSIS COMPLETE! READY FOR PRESENTATION.")
print("=" * 80)

# ============================================================================
# STEP 11: Save Dashboard Summary to CSV
# ============================================================================

dashboard_summary = []

for cluster_id in CLUSTER_ORDER:

    name = CLUSTER_NAMES[cluster_id]
    short = CLUSTER_SHORT_NAMES[cluster_id]

    count = cluster_counts[cluster_id]
    country_pct = cluster_pcts[cluster_id]

    priority = CLUSTER_PRIORITY[cluster_id]

    # CORRECT budget handling
    budget_share = CLUSTER_BUDGET[cluster_id]
    budget_percent = budget_share * 100
    budget_amount = budget_share * total_budget

    per_country = (
        budget_amount / count
        if count > 0
        else 0
    )

    color = CLUSTER_ID_COLORS[cluster_id]

    dashboard_summary.append(
        {
            'Cluster_ID': cluster_id,
            'Cluster_Name': name,
            'Short_Name': short,
            'Countries': count,
            'Country_Share_%': round(country_pct, 2),
            'Priority': priority,
            'Budget_Share': round(budget_share, 4),
            'Budget_%': round(budget_percent, 2),
            'Total_Budget_USD': round(budget_amount, 2),
            'Total_Budget_M_USD': round(
                budget_amount / 1_000_000,
                2
            ),
            'Per_Country_USD': round(
                per_country,
                2
            ),
            'Color': color
        }
    )

summary_df = pd.DataFrame(dashboard_summary)

summary_df.to_csv(
    '../data/processed/final_summary_dashboard.csv',
    index=False
)

print(
    "\n✅ Dashboard summary saved to: "
    "../data/processed/final_summary_dashboard.csv"
)

# ============================================================================
# STEP 12: Display Final DataFrame
# ============================================================================

print("\n📋 FINAL DASHBOARD DATA")
print("-" * 80)

display(summary_df)

print("\n" + "=" * 80)
print("✅ FINAL SUMMARY DASHBOARD COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================================
# 17. QUICK VIEW: ALL FIGURE FILES
# ============================================================================


base = os.path.join(os.path.dirname(os.getcwd()), 'figures')

for folder in ['eda', 'clustering', 'report']:
    path = os.path.join(base, folder)
    if os.path.exists(path):
        print(f"\n📁 {folder.upper()}:")
        for f in sorted(os.listdir(path)):
            if os.path.isfile(os.path.join(path, f)):
                print(f"  {f}")